# Conformal Prediction for Physical Chemists: all six notebooks in one run

This notebook runs the six tutorial notebooks one after the other and packs everything they produce
(figures as PNG and PDF, all numbers as JSON) into **`ConformalPrimer_outputs.zip`**.

| part | topic |
|---|---|
| 1 | Two models, and why an MAE is not an error bar |
| 2 | Split conformal prediction from scratch (with CV+ for small data sets) |
| 3 | Adaptive intervals: normalized scores and conformalized quantile regression |
| 4 | Mondrian conformal prediction: a guarantee for every reaction class |
| 5 | When the guarantee breaks: exchangeability in practice |
| 6 | From intervals to decisions: selection with a guaranteed false discovery rate |

**How to run it**

* *Google Colab*: open the notebook and choose **Runtime → Run all**. No GPU is needed.
* *Kaggle*: switch **Internet on** in the notebook settings, then **Run All**. The zip file appears under *Output*.
* *Locally*: `pip install numpy pandas scipy scikit-learn matplotlib rdkit jupyterlab`, then run all cells.

The run takes about 8 minutes on Kaggle and about 12 minutes on Colab (free CPU runtimes, October 2026). The data (two tables of the RDB7 data set, 3.8 MB each) are
downloaded from Zenodo (DOI 10.5281/zenodo.6618262) on first use. The last cell compares every number with the
reference run and reports how many agree.


In [ ]:
#@title Setup: install RDKit if needed and write the helper module primer.py { display-mode: "form" }
# This cell is identical in all notebooks. It makes the notebook self-contained:
# on Colab or Kaggle nothing else has to be uploaded.
import importlib, importlib.util, os, subprocess, sys, time

if importlib.util.find_spec("rdkit") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
    except Exception as exc:
        raise RuntimeError("RDKit is not installed and could not be installed with pip. On Kaggle, switch "
                           "'Internet' on in the notebook settings (right-hand panel) and run this cell again.") from exc
    importlib.invalidate_caches()

PRIMER_SOURCE = r'''"""
primer.py -- shared plumbing for the ConformalPrimer notebooks
==============================================================

Everything that is *not* the subject of the tutorial lives in sections 1-3 and 5-6,
so that the notebooks can concentrate on conformal prediction:

    1. data        download and check the RDB7 reaction-barrier tables
    2. features    turn atom-mapped reaction SMILES into numbers
    3. models      the two point predictors used throughout the tutorial
    4. toolbox     the conformal functions that the notebooks derive one by one
    5. plotting    one figure style for all notebooks
    6. results     save numbers, compare them with the reference run

Section 4 is self-contained (NumPy and SciPy only) and can be copied into other projects;
`conformal_toolbox.py` in the repository is exactly that section.

The module needs NumPy, pandas, SciPy, scikit-learn, Matplotlib and RDKit.
Nothing in it uses a GPU; the slowest step takes well under a minute on two cores.
"""

from __future__ import annotations

import glob
import hashlib
import inspect
import json
import os
import platform
import shutil
import sys
import time
import urllib.request
from types import SimpleNamespace

import numpy as np
import pandas as pd

__version__ = "0.3.0"

# --------------------------------------------------------------------------------------
# 0. Constants
# --------------------------------------------------------------------------------------

SEED = 0            # one seed for every random choice that defines the problem
ALPHA = 0.10        # default miscoverage level: 1 - ALPHA = 90% intervals
DIFFICULTY_TREES = 100   # size of the model that predicts |residual|
DIFFICULTY_FLOOR = 0.5   # sigma(x) is never smaller than this fraction of the median out-of-fold residual
R_KCAL = 1.987204e-3     # gas constant in kcal/(mol K)
KB_OVER_H = 2.083662e10  # Boltzmann constant / Planck constant in 1/(s K)

# RDB7: Spiekermann, Pattanaik, Green, Sci. Data 2022, 9, 417 (CC BY 4.0).
# The two tables used here are in version 1.0.1 of the Zenodo record, DOI 10.5281/zenodo.6618262.
# (Version 1.1.0, DOI 10.5281/zenodo.13328872, only adds a file with heats of formation.)
ZENODO_RECORD = "6618262"
DATA_FILES = {
    "ccsdtf12_dz.csv": "CCSD(T)-F12a/cc-pVDZ-F12//ωB97X-D3/def2-TZVP barrier heights (reference)",
    "wb97xd3.csv": "ωB97X-D3/def2-TZVP barrier heights (the cheap level used for Δ-learning)",
}
DATA_URLS = [
    "https://zenodo.org/records/{rec}/files/{name}?download=1",
    "https://zenodo.org/api/records/{rec}/files/{name}/content",
]
REQUIRED_COLUMNS = ["idx", "rsmi", "psmi", "rinchi", "dE0", "dHrxn298", "rmg_family"]
# SHA-256 of the files as served by Zenodo (checked on 11 October 2026) ...
OFFICIAL_SHA256 = {
    "ccsdtf12_dz.csv": "d5db1487a1cb9324f5a2b4a0044a3130194be02525059cb6a029b5fa69ec3141",
    "wb97xd3.csv": "ac1d7d06437f79593a224f9b7b1b2af4d5c1f4335f729dbdab2f0a6b86f36f8e",
}
# ... and fingerprints of their *content* (see `table_fingerprint`), which do not depend on
# line endings or on how a spreadsheet program re-writes the numbers.
REFERENCE_FINGERPRINTS = {
    "ccsdtf12_dz.csv": "ffc3f46925de474f8465a3483f225e0e929a5ee0f3314daf097113f99de717a6",
    "wb97xd3.csv": "0e705bba9602f483838e736caeb18e545e872da547a618b36c3b2992bd12ce74",
}
ENERGY_DECIMALS = 5      # energies are rounded to 1e-5 kcal/mol on loading (see `load_rdb7`)

ELEMENTS = ["H", "C", "N", "O"]                       # RDB7 contains only these
HEAVY = ["C", "N", "O"]
PAIRS_ALL = [f"{a}-{b}" for i, a in enumerate(ELEMENTS) for b in ELEMENTS[i:]]
PAIRS_HEAVY = [f"{a}-{b}" for i, a in enumerate(HEAVY) for b in HEAVY[i:]]

CLASS_ORDER = [
    "H-transfer isomerization",
    "skeletal rearrangement",
    "fragmentation",
    "carbene product",
    "zwitterionic product",
]
CLASS_SHORT = {
    "H-transfer isomerization": "H transfer",
    "skeletal rearrangement": "skeletal",
    "fragmentation": "fragmentation",
    "carbene product": "carbene",
    "zwitterionic product": "zwitterion",
}

MODEL_LABELS = {"structure": "Structure-only model", "delta": "Δ-learning model"}


def log(msg: str, verbose: bool = True) -> None:
    if verbose:
        print(msg, flush=True)


def _digest(*parts) -> str:
    """Short hash of the given strings / bytes: used for the names of cache files."""
    h = hashlib.sha256()
    for part in parts:
        h.update(part if isinstance(part, bytes) else str(part).encode())
        h.update(b"|")
    return h.hexdigest()[:16]


# --------------------------------------------------------------------------------------
# 1. Data
# --------------------------------------------------------------------------------------

def table_fingerprint(table: pd.DataFrame) -> str:
    """SHA-256 over idx, SMILES, the two energies (5 decimals) and the family label: identifies the
    *content* of a table, independently of line endings, float formatting or row order."""
    h = hashlib.sha256()
    cols = table[["idx", "rsmi", "psmi", "dE0", "dHrxn298", "rmg_family"]].sort_values("idx")
    for r in cols.itertuples(index=False):
        fam = "" if (isinstance(r.rmg_family, float) and np.isnan(r.rmg_family)) or r.rmg_family is pd.NA else str(r.rmg_family)
        h.update(f"{int(r.idx)}|{r.rsmi}|{r.psmi}|{r.dE0:.5f}|{r.dHrxn298:.5f}|{fam}\n".encode())
    return h.hexdigest()


def _file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _read_table(path: str) -> pd.DataFrame:
    """Read one RDB7 table and make sure that it is one."""
    table = pd.read_csv(path)
    missing = [c for c in REQUIRED_COLUMNS if c not in table.columns]
    if missing:
        raise ValueError(f"{path} is not an RDB7 table (missing columns: {', '.join(missing)})")
    return table


def _is_reference_table(path: str, name: str) -> bool:
    try:
        return table_fingerprint(_read_table(path)) == REFERENCE_FINGERPRINTS[name]
    except Exception:
        return False


def _find_or_download(name: str, data_dir: str, verbose: bool = True) -> str:
    """Return a local path for `name`: look in `data_dir`, then in Kaggle inputs, then download.

    A downloaded file is accepted only if it is the expected table; an incomplete download or an
    error page is discarded, so that it cannot be mistaken for the data on the next run."""
    local = os.path.join(data_dir, name)
    if os.path.exists(local) and os.path.getsize(local) > 0:
        return local
    for hit in sorted(glob.glob(os.path.join("/kaggle/input", "**", name), recursive=True)):
        if _is_reference_table(hit, name):
            log(f"  using {hit}", verbose)
            return hit
    errors = []
    try:
        os.makedirs(data_dir, exist_ok=True)
    except OSError as exc:
        errors.append(f"cannot create the folder '{data_dir}': {exc}")
    for template in DATA_URLS if not errors else []:
        url = template.format(rec=ZENODO_RECORD, name=name)
        part = local + ".part"
        try:
            log(f"  downloading {name} from Zenodo record {ZENODO_RECORD} ...", verbose)
            req = urllib.request.Request(url, headers={"User-Agent": f"ConformalPrimer/{__version__}"})
            with urllib.request.urlopen(req, timeout=60) as resp, open(part, "wb") as out:
                shutil.copyfileobj(resp, out)
            if not _is_reference_table(part, name):
                raise ValueError("the file received is not the expected table (incomplete download or an error page)")
            os.replace(part, local)
            return local
        except Exception as exc:  # try the next address
            errors.append(f"{url}: {exc}")
            if os.path.exists(part):
                os.remove(part)
    raise RuntimeError(
        f"Could not obtain {name}.\n"
        f"Download it by hand from https://doi.org/10.5281/zenodo.{ZENODO_RECORD} "
        f"and put it in the folder '{data_dir}/' next to the notebook (on Kaggle: switch Internet on, "
        f"or attach the file as a dataset).\nTried:\n  " + "\n  ".join(errors)
    )


def load_rdb7(data_dir: str = "data", verbose: bool = True, strict: bool = True) -> pd.DataFrame:
    """Load RDB7 as one table.

    Columns
    -------
    idx, rsmi, psmi, rinchi : reaction index, atom-mapped reactant / product SMILES, reactant InChI
    barrier_cc, dH_cc       : CCSD(T)-F12a barrier height (the quantity we predict) and reaction enthalpy
    barrier_dft, dH_dft     : the same two numbers at the ωB97X-D3 level
    rmg_family              : RMG reaction family, given by the authors for about 12% of the reactions
    All energies are in kcal/mol; barrier heights include zero-point energy.

    The energies are rounded to 1e-5 kcal/mol, so that the numbers of the notebooks do not depend on
    how many digits a particular copy of the files carries. With `strict=True` (default) a table whose
    content differs from the Zenodo file is refused.
    """
    paths = {name: _find_or_download(name, data_dir, verbose) for name in DATA_FILES}
    tables, status = {}, {}
    for name in DATA_FILES:
        table = _read_table(paths[name])
        if table["idx"].duplicated().any():
            raise ValueError(f"{name}: reaction indices are not unique")
        fp = table_fingerprint(table)
        if _file_sha256(paths[name]) == OFFICIAL_SHA256[name]:
            verdict = "byte-identical to the Zenodo file"
        elif fp == REFERENCE_FINGERPRINTS[name]:
            verdict = "same content as the Zenodo file"
        else:
            verdict = "DIFFERENT from the Zenodo file"
            if strict:
                raise ValueError(
                    f"{paths[name]} is not the RDB7 table of Zenodo record {ZENODO_RECORD} (version 1.0.1): its content differs.\n"
                    f"Delete the file to have it downloaded again, or call load_rdb7(strict=False) to use it anyway.")
        for column in ("dE0", "dHrxn298"):                   # the same numbers from every copy of the file
            table[column] = [float(f"{value:.{ENERGY_DECIMALS}f}") for value in table[column]]
        tables[name], status[name] = table, {"fingerprint": fp, "verdict": verdict}
    cc, dft = tables["ccsdtf12_dz.csv"], tables["wb97xd3.csv"]
    df = cc.rename(columns={"dE0": "barrier_cc", "dHrxn298": "dH_cc"})
    low = dft[["idx", "dE0", "dHrxn298"]].rename(columns={"dE0": "barrier_dft", "dHrxn298": "dH_dft"})
    df = df.merge(low, on="idx", how="left", validate="one_to_one").sort_values("idx").reset_index(drop=True)
    if df[["barrier_cc", "barrier_dft", "dH_dft"]].isna().any().any():
        raise ValueError("RDB7 tables are incomplete: some reactions lack a CCSD(T)-F12a or ωB97X-D3 value")
    df.attrs["fingerprints"] = {name: s["fingerprint"] for name, s in status.items()}
    df.attrs["data_status"] = {name: s["verdict"] for name, s in status.items()}
    if verbose:
        print(f"RDB7: {len(df):,} reactions of {df['rinchi'].nunique()} distinct reactants")
        for name, s in status.items():
            print(f"  {name:16s} {s['verdict']}  (content fingerprint {s['fingerprint'][:12]}...)")
    return df


# --------------------------------------------------------------------------------------
# 2. Features
# --------------------------------------------------------------------------------------

def parse_mapped_smiles(smiles: str):
    """RDKit molecule that keeps the explicit, atom-mapped hydrogens of RDB7."""
    from rdkit import Chem

    params = Chem.SmilesParserParams()
    params.removeHs = False
    mol = Chem.MolFromSmiles(smiles, params)
    if mol is None:                       # unusual valence: sanitize as far as possible
        params.sanitize = False
        mol = Chem.MolFromSmiles(smiles, params)
        mol.UpdatePropertyCache(strict=False)
        Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES, catchErrors=True)
    return mol


def readable(smiles: str) -> str:
    """Plain SMILES for display: atom-map numbers and explicit hydrogens removed."""
    from rdkit import Chem

    mol = parse_mapped_smiles(smiles)
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)
    try:
        return Chem.MolToSmiles(Chem.RemoveHs(mol))
    except Exception:
        return Chem.MolToSmiles(mol)


def bond_orders(mol) -> dict:
    """{(map number i, map number j): bond order}, aromatic bonds counted as 1.5."""
    table = {}
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtom().GetAtomMapNum(), bond.GetEndAtom().GetAtomMapNum()
        table[(min(i, j), max(i, j))] = bond.GetBondTypeAsDouble()
    return table


def _pair(symbols: dict, i: int, j: int) -> str:
    a, b = sorted((symbols[i], symbols[j]), key=ELEMENTS.index)
    return f"{a}-{b}"


def count_rings(mol) -> int:
    """Number of independent rings (bonds - atoms + molecules); the same in every RDKit version."""
    from rdkit import Chem

    return mol.GetNumBonds() - mol.GetNumAtoms() + len(Chem.GetMolFrags(mol))


def has_charge_separation(mol) -> bool:
    """True if the Lewis structure carries formal charges, other than those with which carbon monoxide
    and isocyanides are conventionally written ([C-]#[O+], [C-]#[N+]R)."""
    charged = {atom.GetIdx() for atom in mol.GetAtoms() if atom.GetFormalCharge() != 0}
    for bond in mol.GetBonds():
        if bond.GetBondTypeAsDouble() == 3.0:
            a, b = bond.GetBeginAtom(), bond.GetEndAtom()
            for carbon, partner in ((a, b), (b, a)):
                if (carbon.GetSymbol() == "C" and carbon.GetFormalCharge() == -1
                        and partner.GetSymbol() in ("N", "O") and partner.GetFormalCharge() == 1):
                    charged -= {carbon.GetIdx(), partner.GetIdx()}
    return bool(charged)


def describe_reaction(rsmi: str, psmi: str, n_bits: int = 1024, radius: int = 2):
    """Features of one reaction.

    Returns
    -------
    desc    : dict of interpretable descriptors (bonds broken / formed / changed, sizes, rings, ...)
    fp_r    : Morgan count fingerprint of the reactant (hydrogens removed)
    fp_diff : product fingerprint minus reactant fingerprint ("difference fingerprint")
    """
    from rdkit import Chem
    from rdkit.Chem import rdFingerprintGenerator

    r, p = parse_mapped_smiles(rsmi), parse_mapped_smiles(psmi)
    symbols = {a.GetAtomMapNum(): a.GetSymbol() for a in r.GetAtoms()}
    b_r, b_p = bond_orders(r), bond_orders(p)
    broken = [k for k in b_r if k not in b_p]
    formed = [k for k in b_p if k not in b_r]
    changed = [k for k in b_r if k in b_p and b_r[k] != b_p[k]]

    d = {}
    for kind, keys in (("broken", broken), ("formed", formed)):
        for name in PAIRS_ALL:
            d[f"{kind}_{name}"] = 0
        for i, j in keys:
            d[f"{kind}_{_pair(symbols, i, j)}"] += 1
    for name in PAIRS_HEAVY:
        d[f"order_up_{name}"] = 0
        d[f"order_down_{name}"] = 0
    for i, j in changed:
        direction = "order_up_" if b_p[(i, j)] > b_r[(i, j)] else "order_down_"
        d[direction + _pair(symbols, i, j)] += 1

    d["n_broken"], d["n_formed"], d["n_changed"] = len(broken), len(formed), len(changed)
    d["n_reacting_atoms"] = len({m for key in broken + formed + changed for m in key})
    d["n_XH_broken"] = sum("H" in (symbols[i], symbols[j]) for i, j in broken)
    d["n_heavy"] = sum(a.GetAtomicNum() > 1 for a in r.GetAtoms())
    for el in ELEMENTS:
        d[f"n_{el}"] = sum(a.GetSymbol() == el for a in r.GetAtoms())
    d["n_products"] = len(Chem.GetMolFrags(p))
    d["rings_reactant"] = count_rings(r)
    d["rings_product"] = count_rings(p)
    d["delta_rings"] = d["rings_product"] - d["rings_reactant"]
    d["zwitterion_reactant"] = int(has_charge_separation(r))
    d["zwitterion_product"] = int(has_charge_separation(p))
    d["carbene_product"] = int(sum(a.GetNumRadicalElectrons() for a in p.GetAtoms()) > 0)

    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = []
    for mol in (r, p):
        heavy = Chem.RemoveHs(mol, sanitize=False)
        heavy.UpdatePropertyCache(strict=False)
        Chem.FastFindRings(heavy)
        fps.append(generator.GetCountFingerprintAsNumPy(heavy).astype(np.int16))
    return d, fps[0], fps[1] - fps[0]


def reaction_class(d) -> str:
    """Five coarse, automatically assigned reaction classes (used as Mondrian categories).

    The rules are applied in this order to the *product* side of the reaction, as RDB7 writes it:
      a product with a divalent carbon (rarely a nitrene or two radical centers)   -> carbene product
      a product written with formal charges (CO and isocyanides not counted)       -> zwitterionic product
      two or three product molecules                                               -> fragmentation
      one product, at least one X-H bond broken                                    -> H-transfer isomerization
      one product, no X-H bond broken                                              -> skeletal rearrangement
    These are bookkeeping classes read off the Lewis structures, not mechanistic classes.
    """
    if d["carbene_product"]:
        return "carbene product"
    if d["zwitterion_product"]:
        return "zwitterionic product"
    if d["n_products"] >= 2:
        return "fragmentation"
    if d["n_XH_broken"] > 0:
        return "H-transfer isomerization"
    return "skeletal rearrangement"


def featurize(df: pd.DataFrame, cache_dir: str = "cache", verbose: bool = True, n_bits: int = 1024, radius: int = 2):
    """Descriptors and fingerprints for every reaction (cached on disk)."""
    import rdkit

    key = _digest("|".join(df["rsmi"]), "|".join(df["psmi"]), __version__, rdkit.__version__, n_bits, radius)
    path = os.path.join(cache_dir, f"features_{key}.npz")
    if os.path.exists(path):
        z = np.load(path)
        desc = pd.DataFrame(z["desc"], columns=[str(c) for c in z["columns"]])
        return desc, z["fp_r"], z["fp_diff"]
    try:
        from rdkit import RDLogger
        RDLogger.DisableLog("rdApp.*")        # RDB7 contains unusual valences; the warnings are expected
    except Exception:
        pass
    t0 = time.time()
    rows, fp_r, fp_diff = [], [], []
    for rsmi, psmi in zip(df["rsmi"], df["psmi"]):
        d, a, b = describe_reaction(rsmi, psmi, n_bits, radius)
        rows.append(d)
        fp_r.append(a)
        fp_diff.append(b)
    desc = pd.DataFrame(rows).astype(np.float32)
    fp_r, fp_diff = np.asarray(fp_r), np.asarray(fp_diff)
    try:
        os.makedirs(cache_dir, exist_ok=True)
        np.savez_compressed(path, desc=desc.values, columns=np.array(list(desc.columns), dtype=str), fp_r=fp_r, fp_diff=fp_diff)
    except OSError:                           # read-only folder: work without a cache
        pass
    log(f"featurized {len(df):,} reactions in {time.time() - t0:.0f} s", verbose)
    return desc, fp_r, fp_diff


# --------------------------------------------------------------------------------------
# 3. Models
# --------------------------------------------------------------------------------------

def make_model(loss: str = "squared_error", **kwargs):
    """Gradient-boosted trees (scikit-learn): 300 trees, learning rate 0.1, no early stopping, all
    other settings at their defaults. `loss` may also be 'absolute_error' or 'quantile'."""
    from sklearn.ensemble import HistGradientBoostingRegressor

    params = dict(loss=loss, max_iter=300, learning_rate=0.1, early_stopping=False, random_state=SEED)
    params.update(kwargs)
    return HistGradientBoostingRegressor(**params)


def group_split(groups, fraction: float = 0.5, seed: int = SEED):
    """Split row indices so that all rows of one group land on the same side.

    Returns (first, second): sorted row indices; `first` holds about `fraction` of the groups."""
    groups = np.asarray(groups)
    unique = np.unique(groups)
    chosen = np.random.default_rng(seed).permutation(len(unique))[: int(round(fraction * len(unique)))]
    in_first = np.isin(groups, unique[chosen])
    return np.flatnonzero(in_first), np.flatnonzero(~in_first)


def grouped_folds(groups, n_folds: int = 5, seed: int = SEED) -> np.ndarray:
    """Fold index (0 ... n_folds-1) for every row; all rows of one group are in the same fold."""
    unique, inverse = np.unique(np.asarray(groups), return_inverse=True)
    fold_of_group = np.random.default_rng(seed).permutation(len(unique)) % n_folds
    return fold_of_group[inverse]


def frequent_columns(fp: np.ndarray, rows: np.ndarray, min_fraction: float = 0.05) -> np.ndarray:
    """Fingerprint bits that are nonzero in at least `min_fraction` of the given (training) rows."""
    return (fp[rows] != 0).mean(axis=0) >= min_fraction


def fit_model(rdb7: SimpleNamespace, name: str, train_rows) -> SimpleNamespace:
    """Fit one of the two point predictors on `train_rows` and predict every reaction.

    name = "structure": structure-only model. Input: descriptors and fingerprints. Output: the barrier.
    name = "delta"    : Δ-learning model. Input: descriptors, ωB97X-D3 barrier, and reaction enthalpy.
                        Output: the correction from the ωB97X-D3 barrier to the CCSD(T)-F12a barrier.
    Everything that is learned (including which fingerprint bits are used) comes from `train_rows` only.
    """
    train_rows = np.asarray(train_rows)
    if name == "structure":
        keep_r = frequent_columns(rdb7.fp_r, train_rows)       # bits present in at least 5% of the training reactions
        keep_d = frequent_columns(rdb7.fp_diff, train_rows)
        X = np.hstack([rdb7.desc.values, rdb7.fp_r[:, keep_r], rdb7.fp_diff[:, keep_d]]).astype(np.float32)
        baseline = np.zeros(rdb7.n)
    elif name == "delta":
        X = np.hstack([rdb7.desc.values, rdb7.df[["barrier_dft", "dH_dft"]].to_numpy(float)]).astype(np.float32)
        baseline = rdb7.df["barrier_dft"].to_numpy(float)
    else:
        raise ValueError(f"unknown model {name!r}")
    target = rdb7.y - baseline                               # what the trees actually learn
    estimator = make_model().fit(X[train_rows], target[train_rows])
    return SimpleNamespace(name=name, label=MODEL_LABELS[name], X=X, baseline=baseline, target=target,
                           estimator=estimator, train=train_rows, mu=estimator.predict(X) + baseline)


_PROBLEMS: dict = {}


def load_problem(data_dir: str = "data", cache_dir: str = "cache", verbose: bool = True) -> SimpleNamespace:
    """Download the data, build the features and fit the two point predictors.

    Returns a namespace `rdb7` with

        rdb7.df, rdb7.desc            the reaction table and its descriptors
        rdb7.y                        CCSD(T)-F12a barrier heights (kcal/mol) -- the target
        rdb7.rxn_class, rdb7.reactant reaction class and reactant identity of every reaction
        rdb7.train, rdb7.hold         row indices of the fixed split (seed 0): half of the *reactants*
                                      on each side, so that no reactant molecule is both trained on and tested
        rdb7.models["structure"|"delta"]   one namespace per model:
            .label     display name
            .X         feature matrix
            .baseline  what is added to the model output (0, or the DFT barrier)
            .train     training rows (= rdb7.train)
            .hold      held-out rows (= rdb7.hold): the pool for calibration and test sets
            .mu        point prediction for every reaction

    The held-out half is never used for fitting; every notebook splits it again into a
    calibration set and a test set.
    """
    key = (os.path.abspath(data_dir), os.path.abspath(cache_dir))
    if key in _PROBLEMS:                                   # already built in this Python session
        log("RDB7 and both models are already in memory", verbose)
        return _PROBLEMS[key]
    t0 = time.time()
    df = load_rdb7(data_dir, verbose)
    desc, fp_r, fp_diff = featurize(df, cache_dir, verbose)
    y = df["barrier_cc"].to_numpy(float)
    reactant = df["rinchi"].to_numpy()
    train, hold = group_split(reactant, 0.5, SEED)
    classes = np.array([reaction_class(r) for r in desc.to_dict("records")])
    rdb7 = SimpleNamespace(df=df, desc=desc, y=y, n=len(df), train=train, hold=hold, models={},
                           rxn_class=classes, reactant=reactant, n_heavy=desc["n_heavy"].to_numpy(int),
                           fp_r=fp_r, fp_diff=fp_diff, cache_dir=cache_dir)
    for name in MODEL_LABELS:
        mod = fit_model(rdb7, name, train)
        mod.hold = hold
        rdb7.models[name] = mod
        if verbose:
            err = np.abs(y - mod.mu)[hold]
            print(f"{mod.label:22s} {mod.X.shape[1]:4d} features | train {len(train)} | held out {len(hold)}"
                  f" | held-out MAE {err.mean():.2f} kcal/mol")
    log(f"ready in {time.time() - t0:.0f} s", verbose)
    _PROBLEMS[key] = rdb7
    return rdb7


def fit_difficulty(rdb7: SimpleNamespace, name: str, n_folds: int = 5, verbose: bool = True) -> np.ndarray:
    """sigma(x) > 0: how large an error to expect for reaction x (Notebook 3 builds this step by step).

    1. Out-of-fold predictions on the training set give honest residuals for the training reactions
       (the folds keep all reactions of one reactant together).
    2. A second, smaller model learns |out-of-fold residual| from the same features.
    3. Its prediction, bounded from below, is sigma(x).
    The held-out reactions are not used anywhere.
    """
    import sklearn

    mod = rdb7.models[name]
    key = _digest(mod.mu.tobytes(), name, n_folds, DIFFICULTY_TREES, DIFFICULTY_FLOOR, SEED, __version__, sklearn.__version__)
    path = os.path.join(rdb7.cache_dir, f"sigma_{name}_{key}.npy")
    if os.path.exists(path):
        return np.load(path)
    fold = grouped_folds(rdb7.reactant[mod.train], n_folds, SEED)
    oof = np.empty(len(mod.train))
    for k in range(n_folds):
        fit_rows, out_rows = mod.train[fold != k], mod.train[fold == k]
        oof[fold == k] = make_model().fit(mod.X[fit_rows], mod.target[fit_rows]).predict(mod.X[out_rows])
    abs_res = np.abs(mod.target[mod.train] - oof)
    sigma_model = make_model(max_iter=DIFFICULTY_TREES).fit(mod.X[mod.train], abs_res)
    floor = DIFFICULTY_FLOOR * np.median(abs_res)
    sigma = np.maximum(sigma_model.predict(mod.X), floor)
    try:
        os.makedirs(rdb7.cache_dir, exist_ok=True)
        np.save(path, sigma)
    except OSError:
        pass
    log(f"{mod.label}: difficulty model fitted (out-of-fold MAE {abs_res.mean():.2f} kcal/mol)", verbose)
    return sigma


# --------------------------------------------------------------------------------------
# 4. Conformal toolbox
# --------------------------------------------------------------------------------------

# >>> toolbox
# The conformal functions that the notebooks derive step by step, collected here for reuse.
# They need only NumPy (and SciPy for `coverage_law`).

def _rank(n: int, alpha: float) -> int:
    """ceil((n+1)(1-alpha)), protected against floating-point round-up (e.g., 75 * 0.68 = 51.00000000000001)."""
    if not 0 < alpha < 1:
        raise ValueError("alpha must lie strictly between 0 and 1")
    return int(np.ceil((n + 1) * (1 - alpha) - 1e-9))


def conformal_quantile(scores, alpha: float = 0.10) -> float:
    """The ceil((n+1)(1-alpha))-th smallest calibration score; infinity if n is too small.   [Notebook 2]"""
    scores = np.asarray(scores, dtype=float).ravel()
    if np.isnan(scores).any():
        raise ValueError("the calibration scores contain NaN")
    n = scores.size
    k = _rank(n, alpha)                                 # rank of the order statistic we need
    if k > n:                                           # fewer than (1-alpha)/alpha calibration points
        return np.inf
    return float(np.partition(scores, k - 1)[k - 1])


def split_conformal(y_cal, mu_cal, mu_test, alpha: float = 0.10):
    """Absolute-residual split conformal interval mu +/- q. Returns (lower, upper, q).   [Notebook 2]"""
    mu_test = np.asarray(mu_test, dtype=float)
    q = conformal_quantile(np.abs(np.asarray(y_cal, dtype=float) - np.asarray(mu_cal, dtype=float)), alpha)
    return mu_test - q, mu_test + q, q


def coverage_law(n_cal: int, alpha: float = 0.10):
    """Beta distribution of the coverage obtained from ONE calibration set of size n_cal.   [Notebook 2]
    Returns None if n_cal is too small for a finite interval at this level."""
    from scipy import stats

    k = _rank(n_cal, alpha)
    if k > n_cal:
        return None
    return stats.beta(k, n_cal + 1 - k)


def cv_plus(y, oof_pred, fold, test_pred, alpha: float = 0.10):
    """CV+ interval (Barber et al., 2021): no separate calibration set is needed.   [Notebook 2]

    y, oof_pred : reference values of the n labeled points and their out-of-fold predictions
    fold        : fold index (0 ... K-1) of each labeled point
    test_pred   : array (K, m); row k holds the predictions for the m test points made by the
                  model that was fitted WITHOUT fold k
    Returns (lower, upper). Worst-case coverage: 1 - 2 alpha for the jackknife+ (K = n) and slightly less
    for K < n (the bound is given in Notebook 2); in practice the coverage is close to 1 - alpha.
    """
    y, oof_pred, fold = np.asarray(y, float), np.asarray(oof_pred, float), np.asarray(fold, int)
    test_pred = np.asarray(test_pred, float)
    n = y.size
    residual = np.abs(y - oof_pred)
    k_hi = _rank(n, alpha)                              # rank for the upper end
    k_lo = int(np.floor((n + 1) * alpha + 1e-9))        # rank for the lower end
    m = test_pred.shape[1]
    if k_hi > n or k_lo < 1:
        return np.full(m, -np.inf), np.full(m, np.inf)
    low = test_pred[fold] - residual[:, None]           # (n, m): what point i "says" about test point j
    high = test_pred[fold] + residual[:, None]
    lower = np.partition(low, k_lo - 1, axis=0)[k_lo - 1]
    upper = np.partition(high, k_hi - 1, axis=0)[k_hi - 1]
    return lower, upper


def normalized_conformal(y_cal, mu_cal, sigma_cal, mu_test, sigma_test, alpha: float = 0.10):
    """Interval mu +/- q * sigma(x) from the score |y - mu| / sigma. Returns (lower, upper, q).   [Notebook 3]"""
    sigma_cal, sigma_test = np.asarray(sigma_cal, dtype=float), np.asarray(sigma_test, dtype=float)
    mu_test = np.asarray(mu_test, dtype=float)
    if (sigma_cal <= 0).any() or (sigma_test <= 0).any():
        raise ValueError("sigma must be positive")
    q = conformal_quantile(np.abs(np.asarray(y_cal, dtype=float) - np.asarray(mu_cal, dtype=float)) / sigma_cal, alpha)
    return mu_test - q * sigma_test, mu_test + q * sigma_test, q


def cqr(y_cal, lo_cal, hi_cal, lo_test, hi_test, alpha: float = 0.10):
    """Conformalized quantile regression (Romano et al., 2019). Returns (lower, upper, q).   [Notebook 3]

    lo_*, hi_* : predictions of a lower and an upper quantile model (e.g., the 5% and 95% quantiles)
    q may be negative: the raw quantile band is then *shrunk* (and where the band is narrower than
    2|q| the interval is empty: lower > upper).
    """
    y_cal = np.asarray(y_cal, dtype=float)
    lo_test, hi_test = np.asarray(lo_test, dtype=float), np.asarray(hi_test, dtype=float)
    q = conformal_quantile(np.maximum(np.asarray(lo_cal, dtype=float) - y_cal, y_cal - np.asarray(hi_cal, dtype=float)), alpha)
    return lo_test - q, hi_test + q, q


def interval_score(y, lower, upper, alpha: float = 0.10) -> float:
    """Mean interval (Winkler) score: width plus (2/alpha) x the distance by which y is missed.   [Notebook 3]
    Smaller is better; it rewards narrow intervals and penalizes misses in proportion to their size."""
    y, lower, upper = np.asarray(y, dtype=float), np.asarray(lower, dtype=float), np.asarray(upper, dtype=float)
    miss = np.maximum(lower - y, 0) + np.maximum(y - upper, 0)
    return float(np.mean((upper - lower) + (2 / alpha) * miss))


def mondrian_quantiles(scores, groups, alpha: float = 0.10) -> dict:
    """One conformal quantile per group: {group label: q}.   [Notebook 4]"""
    scores, groups = np.asarray(scores, float), np.asarray(groups)
    return {g: conformal_quantile(scores[groups == g], alpha) for g in np.unique(groups)}


def weighted_conformal_quantile(scores, w_cal, w_test, alpha: float = 0.10) -> np.ndarray:
    """Weighted conformal quantile (Tibshirani et al., 2019), one value per test point.   [Notebook 5]

    scores, w_cal : calibration scores and their weights w(x_i) = dP_test/dP_cal (any common scale)
    w_test        : weight(s) of the test point(s)
    The result is infinite where the calibration weight at or below the largest score does not reach
    (1 - alpha) of the total weight, the test point included.
    """
    if not 0 < alpha < 1:
        raise ValueError("alpha must lie strictly between 0 and 1")
    scores, w_cal = np.asarray(scores, float).ravel(), np.asarray(w_cal, float).ravel()
    w_test = np.atleast_1d(np.asarray(w_test, float))
    if scores.size != w_cal.size:
        raise ValueError("one weight is needed for every calibration score")
    if np.isnan(scores).any():
        raise ValueError("the calibration scores contain NaN")
    for w in (w_cal, w_test):
        if not np.isfinite(w).all() or (w < 0).any():
            raise ValueError("weights must be finite and non-negative")
    order = np.argsort(scores)
    cum = np.cumsum(w_cal[order])                       # weight at or below each sorted score
    total = (cum[-1] if cum.size else 0.0) + w_test     # total weight, test point included
    need = (1 - alpha) * total                          # mass that must lie at or below the quantile
    idx = np.searchsorted(cum, need * (1 - 1e-9), side="left")      # relative tolerance, like the 1e-9 in the ranks
    padded = np.append(scores[order], np.inf)           # idx == n: the test point's own mass is needed
    result = padded[idx]
    result[total <= 0] = np.inf                         # no weight at all: nothing can be said
    return result


def effective_sample_size(weights) -> float:
    """(sum w)^2 / sum w^2: how many equally weighted calibration points the weighted set is worth.   [Notebook 5]"""
    w = np.asarray(weights, float)
    return float(w.sum() ** 2 / np.sum(w ** 2)) if np.any(w > 0) else 0.0


def conformal_pvalues(score_cal, null_cal, score_test) -> np.ndarray:
    """Conformal p-values for the hypothesis 'this candidate is NOT a hit' (Jin and Candès, 2023).   [Notebook 6]

    score     : any number computed from x alone; smaller = looks more promising
    score_cal, null_cal : scores of the WHOLE calibration set (hits and non-hits, in the proportions
                in which they occur among the candidates) and True for those that are known non-hits
    p_j = (1 + number of calibration non-hits that look at least as promising as j) / (n + 1)
    """
    score_cal, null_cal = np.asarray(score_cal, float), np.asarray(null_cal, bool)
    score_test = np.asarray(score_test, float)
    if np.isnan(score_cal).any() or np.isnan(score_test).any():
        raise ValueError("the scores contain NaN")
    null_scores = np.sort(score_cal[null_cal])
    return (1 + np.searchsorted(null_scores, score_test, side="right")) / (score_cal.size + 1)


def benjamini_hochberg(pvalues, q: float) -> np.ndarray:
    """Boolean mask of the hypotheses rejected by the Benjamini–Hochberg procedure at level q.   [Notebook 6]"""
    if not 0 < q < 1:
        raise ValueError("q must lie strictly between 0 and 1")
    p = np.asarray(pvalues, float)
    m = p.size
    order = np.argsort(p, kind="stable")
    line = q * np.arange(1, m + 1) / m * (1 + 1e-12)    # the tiny factor makes "p equal to the line" count as below it
    passed = np.flatnonzero(p[order] <= line)
    selected = np.zeros(m, dtype=bool)
    if passed.size:
        selected[order[: passed[-1] + 1]] = True
    return selected
# <<< toolbox


def resplits(pool, n_cal: int, n_rep: int, seed: int = SEED):
    """Yield `n_rep` random (calibration, test) partitions of the index array `pool`."""
    pool = np.asarray(pool)
    if not 0 < n_cal < len(pool):
        raise ValueError("n_cal must be smaller than the pool, so that a test set remains")
    rng = np.random.default_rng(seed)
    for _ in range(n_rep):
        perm = rng.permutation(pool)
        yield perm[:n_cal], perm[n_cal:]


def show(function) -> None:
    """Print the source code of a function of this module (so that no notebook hides its ingredients)."""
    print(inspect.getsource(function))


def rate_factor(delta_e, temperature: float = 298.15):
    """Factor by which a rate constant changes when the barrier changes by delta_e (kcal/mol)."""
    return np.exp(np.asarray(delta_e, float) / (R_KCAL * temperature))


# --------------------------------------------------------------------------------------
# 5. Plotting
# --------------------------------------------------------------------------------------

# Color-blind-safe palette of Wong (Nat. Methods 2011, 8, 441); black is reserved for text.
BLUE, VERMILLION, GREEN = "#0072B2", "#D55E00", "#009E73"
PURPLE, ORANGE, SKY = "#CC79A7", "#E69F00", "#56B4E9"
GRAY, LIGHT_GRAY = "#6E6E6E", "#D0D0D0"
MODEL_COLORS = {"structure": VERMILLION, "delta": BLUE}
MODEL_MARKERS = {"structure": "s", "delta": "o"}


def set_style() -> None:
    import matplotlib as mpl

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
        "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
        "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.major.size": 3, "ytick.major.size": 3,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": GRAY, "xtick.color": GRAY, "ytick.color": GRAY,
        "xtick.labelcolor": "black", "ytick.labelcolor": "black", "axes.labelcolor": "black",
        "lines.linewidth": 1.4, "lines.markersize": 4,
        "legend.frameon": False, "legend.handlelength": 1.6,
        "figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.03,
        "pdf.fonttype": 42, "ps.fonttype": 42, "figure.facecolor": "white",
    })


def panels(n: int, height: float = 2.45, width: float = 7.0, **kwargs):
    """A row of n panels, 7 inches wide (two journal columns), laid out so that no label is cut off."""
    import matplotlib.pyplot as plt

    return plt.subplots(1, n, figsize=(width, height), layout="constrained", **kwargs)


def panel_label(ax, letter: str, title: str | None = None) -> None:
    """Bold panel letter just outside the upper left corner of the axes, and optionally the panel title next to it."""
    ax.text(-0.02, 1.04, f"({letter})", transform=ax.transAxes, fontsize=9, fontweight="bold",
            va="bottom", ha="right", color="black")
    if title is not None:
        ax.set_title(title, loc="left")


def savefig(fig, name: str, out_dir: str = "outputs") -> None:
    """Save a figure as 300-dpi PNG and as vector PDF."""
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, f"{name}.png"))
    fig.savefig(os.path.join(out_dir, f"{name}.pdf"), metadata={"CreationDate": None})


# --------------------------------------------------------------------------------------
# 6. Results
# --------------------------------------------------------------------------------------

NOTEBOOKS = ("nb1", "nb2", "nb3", "nb4", "nb5", "nb6")
LIBRARIES = ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "rdkit", "primer")
MODEL_DEPENDENT_COUNTS = ("selected", "false discoveries", "hits found", "smallest non-empty")


def _plain(obj):
    """Make NumPy scalars / arrays JSON-serializable."""
    if isinstance(obj, dict):
        return {str(k): _plain(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_plain(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [_plain(v) for v in obj.tolist()]
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        return None if not np.isfinite(obj) else round(float(obj), 6)
    return obj


def environment() -> dict:
    import matplotlib
    import scipy
    import sklearn

    try:
        import rdkit
        rd = rdkit.__version__
    except Exception:  # pragma: no cover
        rd = "not installed"
    return {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
            "matplotlib": matplotlib.__version__, "rdkit": rd, "primer": __version__}


def flatten(d: dict, prefix: str = "") -> dict:
    flat = {}
    for k, v in d.items():
        key = f"{prefix}/{k}" if prefix else str(k)
        if isinstance(v, dict):
            flat.update(flatten(v, key))
        elif isinstance(v, list):
            flat.update(flatten({str(i): item for i, item in enumerate(v)}, key))
        else:
            flat[key] = v
    return flat


def save_results(name: str, results: dict, rdb7: SimpleNamespace | None = None, out_dir: str = "outputs") -> str:
    """Write the numbers of one notebook to outputs/<name>_results.json."""
    os.makedirs(out_dir, exist_ok=True)
    payload = {"notebook": name, "environment": environment(), "results": _plain(results)}
    if rdb7 is not None:
        payload["data_fingerprints"] = rdb7.df.attrs.get("fingerprints", {})
        payload["data_status"] = rdb7.df.attrs.get("data_status", {})
        payload["n_reactions"] = int(rdb7.n)
    path = os.path.join(out_dir, f"{name}_results.json")
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=1, ensure_ascii=False)
    print(f"saved {path}")
    return path


# Reference values from the reference run are appended below by build_notebooks.py.
REFERENCE_RESULTS: dict = {}

# >>> reference results (written by build_notebooks.py -- do not edit by hand)
REFERENCE_RESULTS.update({'_environment': {'matplotlib': '3.10.0', 'numpy': '2.1.3', 'pandas': '2.2.3', 'primer': '0.3.0', 'python': '3.12.3', 'rdkit': '2026.09.1', 'scikit-learn': '1.6.1', 'scipy': '1.16.3'}, 'nb1': {'barrier_mean': 80.060894, 'class_counts/H-transfer isomerization': 2643, 'class_counts/carbene product': 1985, 'class_counts/fragmentation': 2534, 'class_counts/skeletal rearrangement': 1736, 'class_counts/zwitterionic product': 3028, 'delta/class_mae/H-transfer isomerization': 1.927528, 'delta/class_mae/carbene product': 0.890999, 'delta/class_mae/fragmentation': 1.550066, 'delta/class_mae/skeletal rearrangement': 2.468339, 'delta/class_mae/zwitterionic product': 1.459087, 'delta/class_q90/H-transfer isomerization': 4.795443, 'delta/class_q90/carbene product': 2.151114, 'delta/class_q90/fragmentation': 3.427818, 'delta/class_q90/skeletal rearrangement': 6.133364, 'delta/class_q90/zwitterionic product': 3.252838, 'delta/errors/90': 3.786015, 'delta/errors/95': 5.613923, 'delta/errors/99': 10.763432, 'delta/errors/MAE': 1.620686, 'delta/errors/RMSE': 2.768401, 'delta/errors/largest': 32.966523, 'delta/errors/median': 0.895411, 'delta/excess_kurtosis': 18.034909, 'delta/gaussian_coverage/0.50': 0.729065, 'delta/gaussian_coverage/0.68': 0.832687, 'delta/gaussian_coverage/0.80': 0.88829, 'delta/gaussian_coverage/0.90': 0.927211, 'delta/gaussian_coverage/0.95': 0.94642, 'delta/gaussian_coverage/0.99': 0.970851, 'delta/n_features': 51, 'family_share': 0.124182, 'n_hold': 5935, 'n_reactants': 1135, 'n_reactions': 11926, 'n_train': 5991, 'shared_reactants': 0, 'structure/class_mae/H-transfer isomerization': 8.343394, 'structure/class_mae/carbene product': 7.209082, 'structure/class_mae/fragmentation': 7.555171, 'structure/class_mae/skeletal rearrangement': 10.636322, 'structure/class_mae/zwitterionic product': 9.486534, 'structure/class_q90/H-transfer isomerization': 18.806988, 'structure/class_q90/carbene product': 17.167928, 'structure/class_q90/fragmentation': 17.310724, 'structure/class_q90/skeletal rearrangement': 22.827262, 'structure/class_q90/zwitterionic product': 22.389925, 'structure/errors/90': 20.100311, 'structure/errors/95': 25.765606, 'structure/errors/99': 41.287609, 'structure/errors/MAE': 8.585918, 'structure/errors/RMSE': 12.213441, 'structure/errors/largest': 100.291134, 'structure/errors/median': 5.927732, 'structure/excess_kurtosis': 3.749813, 'structure/gaussian_coverage/0.50': 0.628981, 'structure/gaussian_coverage/0.68': 0.774052, 'structure/gaussian_coverage/0.80': 0.841112, 'structure/gaussian_coverage/0.90': 0.899916, 'structure/gaussian_coverage/0.95': 0.935131, 'structure/gaussian_coverage/0.99': 0.975063, 'structure/n_features': 173}, 'nb2': {'coverage_vs_n/100/hi': 0.950647, 'coverage_vs_n/100/lo': 0.833068, 'coverage_vs_n/100/mean': 0.900555, 'coverage_vs_n/100/theory_hi': 0.9515, 'coverage_vs_n/100/theory_lo': 0.835476, 'coverage_vs_n/100/theory_mean': 0.90099, 'coverage_vs_n/1000/hi': 0.920365, 'coverage_vs_n/1000/lo': 0.879635, 'coverage_vs_n/1000/mean': 0.900135, 'coverage_vs_n/1000/theory_hi': 0.919554, 'coverage_vs_n/1000/theory_lo': 0.878825, 'coverage_vs_n/1000/theory_mean': 0.9001, 'coverage_vs_n/20/hi': 0.988005, 'coverage_vs_n/20/lo': 0.740651, 'coverage_vs_n/20/mean': 0.903421, 'coverage_vs_n/20/theory_hi': 0.987828, 'coverage_vs_n/20/theory_lo': 0.750972, 'coverage_vs_n/20/theory_mean': 0.904762, 'coverage_vs_n/200/hi': 0.937929, 'coverage_vs_n/200/lo': 0.854926, 'coverage_vs_n/200/mean': 0.900169, 'coverage_vs_n/200/theory_hi': 0.938448, 'coverage_vs_n/200/theory_lo': 0.854926, 'coverage_vs_n/200/theory_mean': 0.900498, 'coverage_vs_n/2000/hi': 0.915883, 'coverage_vs_n/2000/lo': 0.883602, 'coverage_vs_n/2000/mean': 0.899968, 'coverage_vs_n/2000/theory_hi': 0.915629, 'coverage_vs_n/2000/theory_lo': 0.883355, 'coverage_vs_n/2000/theory_mean': 0.90005, 'coverage_vs_n/50/hi': 0.964325, 'coverage_vs_n/50/lo': 0.803726, 'coverage_vs_n/50/mean': 0.900327, 'coverage_vs_n/50/theory_hi': 0.967035, 'coverage_vs_n/50/theory_lo': 0.807307, 'coverage_vs_n/50/theory_mean': 0.901961, 'coverage_vs_n/500/hi': 0.926224, 'coverage_vs_n/500/lo': 0.871385, 'coverage_vs_n/500/mean': 0.899947, 'coverage_vs_n/500/theory_hi': 0.926035, 'coverage_vs_n/500/theory_lo': 0.871205, 'coverage_vs_n/500/theory_mean': 0.9002, 'dft_offset/mean': 3.477724, 'dft_offset/share_positive': 0.88466, 'levels/0.50/conformal': 0.500063, 'levels/0.50/conformal_se': 0.000558, 'levels/0.50/gaussian': 0.729065, 'levels/0.68/conformal': 0.679239, 'levels/0.68/conformal_se': 0.000511, 'levels/0.68/gaussian': 0.832687, 'levels/0.80/conformal': 0.799297, 'levels/0.80/conformal_se': 0.000428, 'levels/0.80/gaussian': 0.88829, 'levels/0.90/conformal': 0.899662, 'levels/0.90/conformal_se': 0.000326, 'levels/0.90/gaussian': 0.927211, 'levels/0.95/conformal': 0.950033, 'levels/0.95/conformal_se': 0.000244, 'levels/0.95/gaussian': 0.94642, 'levels/0.99/conformal': 0.990067, 'levels/0.99/conformal_se': 0.000107, 'levels/0.99/gaussian': 0.970851, 'n_needed/pm1': 3450, 'n_needed/pm2': 854, 'n_needed/pm3': 374, 'n_needed/pm5': 131, 'pitfall_quantile/1000/conformal': 0.900075, 'pitfall_quantile/1000/plain': 0.899166, 'pitfall_quantile/20/conformal': 0.90461, 'pitfall_quantile/20/plain': 0.865542, 'pitfall_quantile/200/conformal': 0.900781, 'pitfall_quantile/200/plain': 0.896344, 'pitfall_quantile/50/conformal': 0.901299, 'pitfall_quantile/50/plain': 0.884803, 'pitfall_training/coverage': 0.760284, 'pitfall_training/q': 2.099674, 'sharpness/constant/coverage': 0.89999, 'sharpness/constant/mae': 17.328224, 'sharpness/constant/q': 35.956381, 'sharpness/delta/coverage': 0.899822, 'sharpness/delta/mae': 1.620686, 'sharpness/delta/q': 3.798717, 'sharpness/dft/coverage': 0.900752, 'sharpness/dft/mae': 3.614416, 'sharpness/dft/q': 8.242418, 'sharpness/dft_shifted/coverage': 0.90031, 'sharpness/dft_shifted/mae': 2.822347, 'sharpness/dft_shifted/q': 5.385178, 'sharpness/structure/coverage': 0.900015, 'sharpness/structure/mae': 8.585918, 'sharpness/structure/q': 20.118245, 'single_split/coverage': 0.898683, 'single_split/k': 901, 'single_split/n_cal': 1000, 'single_split/n_test': 4935, 'single_split/q_hat': 3.771071, 'small_data/N=100 | CV+ (K = 10)/coverage': 0.911155, 'small_data/N=100 | CV+ (K = 10)/coverage_sd': 0.030147, 'small_data/N=100 | CV+ (K = 10)/half_width': 5.759923, 'small_data/N=100 | CV+ (K = 10)/half_width_sd': 0.977303, 'small_data/N=100 | split 50/50/coverage': 0.907845, 'small_data/N=100 | split 50/50/coverage_sd': 0.040093, 'small_data/N=100 | split 50/50/half_width': 6.255902, 'small_data/N=100 | split 50/50/half_width_sd': 1.436231, 'small_data/N=100 | training residuals (wrong)/coverage': 0.822685, 'small_data/N=100 | training residuals (wrong)/coverage_sd': 0.045207, 'small_data/N=100 | training residuals (wrong)/half_width': 4.090479, 'small_data/N=100 | training residuals (wrong)/half_width_sd': 0.675308, 'small_data/N=1000 | CV+ (K = 10)/coverage': 0.902787, 'small_data/N=1000 | CV+ (K = 10)/coverage_sd': 0.012303, 'small_data/N=1000 | CV+ (K = 10)/half_width': 4.774937, 'small_data/N=1000 | CV+ (K = 10)/half_width_sd': 0.263609, 'small_data/N=1000 | split 50/50/coverage': 0.90085, 'small_data/N=1000 | split 50/50/coverage_sd': 0.015576, 'small_data/N=1000 | split 50/50/half_width': 4.900471, 'small_data/N=1000 | split 50/50/half_width_sd': 0.344123, 'small_data/N=1000 | training residuals (wrong)/coverage': 0.891713, 'small_data/N=1000 | training residuals (wrong)/coverage_sd': 0.012906, 'small_data/N=1000 | training residuals (wrong)/half_width': 4.530891, 'small_data/N=1000 | training residuals (wrong)/half_width_sd': 0.25961, 'small_data/N=300 | CV+ (K = 10)/coverage': 0.90666, 'small_data/N=300 | CV+ (K = 10)/coverage_sd': 0.018284, 'small_data/N=300 | CV+ (K = 10)/half_width': 5.1184, 'small_data/N=300 | CV+ (K = 10)/half_width_sd': 0.476453, 'small_data/N=300 | split 50/50/coverage': 0.905575, 'small_data/N=300 | split 50/50/coverage_sd': 0.023052, 'small_data/N=300 | split 50/50/half_width': 5.459493, 'small_data/N=300 | split 50/50/half_width_sd': 0.673392, 'small_data/N=300 | training residuals (wrong)/coverage': 0.87148, 'small_data/N=300 | training residuals (wrong)/coverage_sd': 0.022526, 'small_data/N=300 | training residuals (wrong)/half_width': 4.402272, 'small_data/N=300 | training residuals (wrong)/half_width_sd': 0.426196, 'small_data/N=50 | CV+ (K = 10)/coverage': 0.910685, 'small_data/N=50 | CV+ (K = 10)/coverage_sd': 0.039803, 'small_data/N=50 | CV+ (K = 10)/half_width': 6.093728, 'small_data/N=50 | CV+ (K = 10)/half_width_sd': 1.40391, 'small_data/N=50 | jackknife+ (K = N)/coverage': 0.905525, 'small_data/N=50 | jackknife+ (K = N)/coverage_sd': 0.047633, 'small_data/N=50 | jackknife+ (K = N)/half_width': 6.059034, 'small_data/N=50 | jackknife+ (K = N)/half_width_sd': 1.403652, 'small_data/N=50 | split 50/50/coverage': 0.916725, 'small_data/N=50 | split 50/50/coverage_sd': 0.054944, 'small_data/N=50 | split 50/50/half_width': 7.260283, 'small_data/N=50 | split 50/50/half_width_sd': 2.516128, 'small_data/N=50 | training residuals (wrong)/coverage': 0.77505, 'small_data/N=50 | training residuals (wrong)/coverage_sd': 0.076258, 'small_data/N=50 | training residuals (wrong)/half_width': 3.789945, 'small_data/N=50 | training residuals (wrong)/half_width_sd': 1.001647}, 'nb3': {'class_coverage_single_split/H transfer': 0.85224, 'class_coverage_single_split/carbene': 0.968458, 'class_coverage_single_split/fragmentation': 0.924386, 'class_coverage_single_split/skeletal': 0.797059, 'class_coverage_single_split/zwitterion': 0.922601, 'fraction_narrower_than_split': 0.678419, 'pitfall_in_sample/in-sample residuals/coverage': 0.900254, 'pitfall_in_sample/in-sample residuals/easiest fifth': 0.951079, 'pitfall_in_sample/in-sample residuals/hardest fifth': 0.850405, 'pitfall_in_sample/in-sample residuals/skeletal rearrangements': 0.872302, 'pitfall_in_sample/out-of-fold residuals (Section 2)/coverage': 0.900616, 'pitfall_in_sample/out-of-fold residuals (Section 2)/easiest fifth': 0.88976, 'pitfall_in_sample/out-of-fold residuals (Section 2)/hardest fifth': 0.922148, 'pitfall_in_sample/out-of-fold residuals (Section 2)/skeletal rearrangements': 0.905008, 'q/cqr': 0.72819, 'q/normalized': 2.136523, 'q/split': 3.771071, 'raw_quantile_band_coverage': 0.775737, 'sigma/floor': 0.445916, 'sigma/in_sample_mae': 0.912204, 'sigma/oof_mae': 1.645348, 'sigma/share_at_floor': 0.067734, 'sigma/spearman': 0.558497, 'sigma/tenth_error/0': 0.465166, 'sigma/tenth_error/1': 0.552436, 'sigma/tenth_error/2': 0.789108, 'sigma/tenth_error/3': 0.929193, 'sigma/tenth_error/4': 1.138651, 'sigma/tenth_error/5': 1.376384, 'sigma/tenth_error/6': 1.744693, 'sigma/tenth_error/7': 2.140548, 'sigma/tenth_error/8': 2.877148, 'sigma/tenth_error/9': 4.195373, 'sigma/tenth_sigma/0': 0.453751, 'sigma/tenth_sigma/1': 0.560751, 'sigma/tenth_sigma/2': 0.730795, 'sigma/tenth_sigma/3': 0.900941, 'sigma/tenth_sigma/4': 1.112342, 'sigma/tenth_sigma/5': 1.371394, 'sigma/tenth_sigma/6': 1.667395, 'sigma/tenth_sigma/7': 2.098883, 'sigma/tenth_sigma/8': 2.81514, 'sigma/tenth_sigma/9': 4.614093, 'summary/CQR/class: H transfer': 0.899027, 'summary/CQR/class: carbene': 0.93667, 'summary/CQR/class: fragmentation': 0.90406, 'summary/CQR/class: skeletal': 0.864992, 'summary/CQR/class: zwitterion': 0.895997, 'summary/CQR/coverage': 0.901174, 'summary/CQR/interval score': 10.598758, 'summary/CQR/mean half-width': 3.339694, 'summary/CQR/median half-width': 2.886454, 'summary/CQR/own-width fifth 1': 0.953921, 'summary/CQR/own-width fifth 2': 0.936049, 'summary/CQR/own-width fifth 3': 0.897447, 'summary/CQR/own-width fifth 4': 0.873759, 'summary/CQR/own-width fifth 5': 0.844696, 'summary/CQR/sigma fifth 1': 0.96565, 'summary/CQR/sigma fifth 2': 0.924598, 'summary/CQR/sigma fifth 3': 0.909234, 'summary/CQR/sigma fifth 4': 0.875929, 'summary/CQR/sigma fifth 5': 0.830466, 'summary/normalized/class: H transfer': 0.908543, 'summary/normalized/class: carbene': 0.903226, 'summary/normalized/class: fragmentation': 0.895144, 'summary/normalized/class: skeletal': 0.905008, 'summary/normalized/class: zwitterion': 0.894402, 'summary/normalized/coverage': 0.900616, 'summary/normalized/interval score': 10.549804, 'summary/normalized/mean half-width': 3.45226, 'summary/normalized/median half-width': 2.618145, 'summary/normalized/own-width fifth 1': 0.889763, 'summary/normalized/own-width fifth 2': 0.891468, 'summary/normalized/own-width fifth 3': 0.894699, 'summary/normalized/own-width fifth 4': 0.905002, 'summary/normalized/own-width fifth 5': 0.922148, 'summary/normalized/sigma fifth 1': 0.88976, 'summary/normalized/sigma fifth 2': 0.891471, 'summary/normalized/sigma fifth 3': 0.894699, 'summary/normalized/sigma fifth 4': 0.905002, 'summary/normalized/sigma fifth 5': 0.922148, 'summary/split/class: H transfer': 0.854161, 'summary/split/class: carbene': 0.969595, 'summary/split/class: fragmentation': 0.924629, 'summary/split/class: skeletal': 0.804917, 'summary/split/class: zwitterion': 0.925254, 'summary/split/coverage': 0.900827, 'summary/split/interval score': 13.567084, 'summary/split/mean half-width': 3.814366, 'summary/split/median half-width': 3.814366, 'summary/split/sigma fifth 1': 0.995825, 'summary/split/sigma fifth 2': 0.986795, 'summary/split/sigma fifth 3': 0.960906, 'summary/split/sigma fifth 4': 0.895366, 'summary/split/sigma fifth 5': 0.665243, 'widest_tenth_error/others': 1.338438, 'widest_tenth_error/widest': 4.164283}, 'nb4': {'average/Mondrian/H transfer': 0.901451, 'average/Mondrian/all': 0.902753, 'average/Mondrian/carbene': 0.902123, 'average/Mondrian/fragmentation': 0.902897, 'average/Mondrian/mean half-width': 3.91905, 'average/Mondrian/skeletal': 0.905282, 'average/Mondrian/zwitterion': 0.902812, 'average/one q for all/H transfer': 0.854635, 'average/one q for all/all': 0.901098, 'average/one q for all/carbene': 0.969333, 'average/one q for all/fragmentation': 0.924795, 'average/one q for all/mean half-width': 3.822305, 'average/one q for all/skeletal': 0.806187, 'average/one q for all/zwitterion': 0.92524, 'budget/100/class coverage 2.5%': 0.770952, 'budget/100/class coverage 97.5%': 0.996251, 'budget/100/mean finite half-width': 5.487601, 'budget/100/smallest class (mean n_g)': 13.917439, 'budget/100/splits with an infinite interval': 0.036667, 'budget/1000/class coverage 2.5%': 0.849413, 'budget/1000/class coverage 97.5%': 0.94626, 'budget/1000/mean finite half-width': 4.048102, 'budget/1000/smallest class (mean n_g)': 139.174389, 'budget/1000/splits with an infinite interval': 0.0, 'budget/200/class coverage 2.5%': 0.804124, 'budget/200/class coverage 97.5%': 0.980181, 'budget/200/mean finite half-width': 4.49897, 'budget/200/smallest class (mean n_g)': 27.834878, 'budget/200/splits with an infinite interval': 0.0, 'budget/2000/class coverage 2.5%': 0.862724, 'budget/2000/class coverage 97.5%': 0.937224, 'budget/2000/mean finite half-width': 4.0072, 'budget/2000/smallest class (mean n_g)': 278.348778, 'budget/2000/splits with an infinite interval': 0.0, 'budget/50/class coverage 2.5%': 0.737984, 'budget/50/class coverage 97.5%': 0.998583, 'budget/50/mean finite half-width': 5.955044, 'budget/50/smallest class (mean n_g)': 6.958719, 'budget/50/splits with an infinite interval': 0.98, 'budget/500/class coverage 2.5%': 0.83714, 'budget/500/class coverage 97.5%': 0.958185, 'budget/500/mean finite half-width': 4.126196, 'budget/500/smallest class (mean n_g)': 69.587195, 'budget/500/splits with an infinite interval': 0.0, 'class_spread/H transfer/2.5%': 0.857271, 'class_spread/H transfer/97.5%': 0.942937, 'class_spread/H transfer/Beta law 2.5%': 0.859354, 'class_spread/H transfer/Beta law 97.5%': 0.938224, 'class_spread/H transfer/mean coverage': 0.901451, 'class_spread/H transfer/mean n_g': 214, 'class_spread/carbene/2.5%': 0.855476, 'class_spread/carbene/97.5%': 0.942664, 'class_spread/carbene/Beta law 2.5%': 0.854968, 'class_spread/carbene/Beta law 97.5%': 0.942049, 'class_spread/carbene/mean coverage': 0.902123, 'class_spread/carbene/mean n_g': 174, 'class_spread/fragmentation/2.5%': 0.862557, 'class_spread/fragmentation/97.5%': 0.940765, 'class_spread/fragmentation/Beta law 2.5%': 0.861248, 'class_spread/fragmentation/Beta law 97.5%': 0.939092, 'class_spread/fragmentation/mean coverage': 0.902897, 'class_spread/fragmentation/mean n_g': 217, 'class_spread/skeletal/2.5%': 0.846813, 'class_spread/skeletal/97.5%': 0.946089, 'class_spread/skeletal/Beta law 2.5%': 0.845379, 'class_spread/skeletal/Beta law 97.5%': 0.943833, 'class_spread/skeletal/mean coverage': 0.905282, 'class_spread/skeletal/mean n_g': 139, 'class_spread/zwitterion/2.5%': 0.858203, 'class_spread/zwitterion/97.5%': 0.936124, 'class_spread/zwitterion/Beta law 2.5%': 0.863722, 'class_spread/zwitterion/Beta law 97.5%': 0.9358, 'class_spread/zwitterion/mean coverage': 0.902812, 'class_spread/zwitterion/mean n_g': 256, 'combined/Mondrian + normalized/best-covered class': 0.904068, 'combined/Mondrian + normalized/coverage': 0.902064, 'combined/Mondrian + normalized/easiest fifth': 0.89145, 'combined/Mondrian + normalized/hardest fifth': 0.920436, 'combined/Mondrian + normalized/mean half-width': 3.50719, 'combined/Mondrian + normalized/median half-width': 2.664412, 'combined/Mondrian + normalized/worst-covered class': 0.899882, 'combined/Mondrian/best-covered class': 0.905282, 'combined/Mondrian/coverage': 0.902753, 'combined/Mondrian/easiest fifth': 0.990754, 'combined/Mondrian/hardest fifth': 0.720182, 'combined/Mondrian/mean half-width': 3.91905, 'combined/Mondrian/median half-width': 3.592667, 'combined/Mondrian/worst-covered class': 0.901451, 'combined/normalized/best-covered class': 0.907313, 'combined/normalized/coverage': 0.899794, 'combined/normalized/easiest fifth': 0.889104, 'combined/normalized/hardest fifth': 0.92112, 'combined/normalized/mean half-width': 3.44391, 'combined/normalized/median half-width': 2.613038, 'combined/normalized/worst-covered class': 0.894137, 'combined/split (one q)/best-covered class': 0.969333, 'combined/split (one q)/coverage': 0.901098, 'combined/split (one q)/easiest fifth': 0.9958, 'combined/split (one q)/hardest fifth': 0.666955, 'combined/split (one q)/mean half-width': 3.822305, 'combined/split (one q)/median half-width': 3.822305, 'combined/split (one q)/worst-covered class': 0.806187, 'families/n_families_cal': 12, 'families/n_finite': 5, 'families/unseen_in_cal': 3, 'merged/(no family)/calibration reactions (mean)': 879.2, 'merged/(no family)/coverage (mean)': 0.901133, 'merged/(no family)/coverage 2.5%': 0.878782, 'merged/(no family)/coverage 97.5%': 0.921631, 'merged/(no family)/half-width (mean)': 4.040361, 'merged/(no family)/training reactions': 5233, 'merged/(other families)/calibration reactions (mean)': 68.915, 'merged/(other families)/coverage (mean)': 0.907125, 'merged/(other families)/coverage 2.5%': 0.811109, 'merged/(other families)/coverage 97.5%': 0.969068, 'merged/(other families)/half-width (mean)': 1.782723, 'merged/(other families)/training reactions': 380, 'merged/ketoenol/calibration reactions (mean)': 51.885, 'merged/ketoenol/coverage (mean)': 0.909665, 'merged/ketoenol/coverage 2.5%': 0.79846, 'merged/ketoenol/coverage 97.5%': 0.976574, 'merged/ketoenol/half-width (mean)': 2.780448, 'merged/ketoenol/training reactions': 378, 'single_split/coverage_global/H transfer': 0.85224, 'single_split/coverage_global/carbene': 0.968458, 'single_split/coverage_global/fragmentation': 0.924386, 'single_split/coverage_global/skeletal': 0.797059, 'single_split/coverage_global/zwitterion': 0.922601, 'single_split/coverage_mondrian/H transfer': 0.896092, 'single_split/coverage_mondrian/carbene': 0.915888, 'single_split/coverage_mondrian/fragmentation': 0.913043, 'single_split/coverage_mondrian/skeletal': 0.916176, 'single_split/coverage_mondrian/zwitterion': 0.907121, 'single_split/q_class/H transfer': 4.7676, 'single_split/q_class/carbene': 2.382235, 'single_split/q_class/fragmentation': 3.596611, 'single_split/q_class/skeletal': 6.587717, 'single_split/q_class/zwitterion': 3.370481, 'single_split/q_global': 3.771071}, 'nb5': {'alarm/different mix of classes/10': 0.0555, 'alarm/different mix of classes/100': 0.4595, 'alarm/different mix of classes/20': 0.1795, 'alarm/different mix of classes/200': 0.616, 'alarm/different mix of classes/50': 0.222, 'alarm/different mix of classes/500': 0.886, 'alarm/larger molecules/10': 0.0345, 'alarm/larger molecules/100': 0.2465, 'alarm/larger molecules/20': 0.112, 'alarm/larger molecules/200': 0.307, 'alarm/larger molecules/50': 0.114, 'alarm/larger molecules/500': 0.5375, 'alarm/same population (no shift)/10': 0.013, 'alarm/same population (no shift)/100': 0.0475, 'alarm/same population (no shift)/20': 0.0485, 'alarm/same population (no shift)/200': 0.041, 'alarm/same population (no shift)/50': 0.03, 'alarm/same population (no shift)/500': 0.0445, 'alarm/unseen class (skeletal)/10': 0.4785, 'alarm/unseen class (skeletal)/100': 1.0, 'alarm/unseen class (skeletal)/20': 0.8815, 'alarm/unseen class (skeletal)/200': 1.0, 'alarm/unseen class (skeletal)/50': 0.9915, 'alarm/unseen class (skeletal)/500': 1.0, 'alarm_exact/10': 0.013219, 'alarm_exact/100': 0.047323, 'alarm_exact/20': 0.044676, 'alarm_exact/200': 0.039453, 'alarm_exact/50': 0.027494, 'alarm_exact/500': 0.043372, 'alarm_repeated_looks': 0.1385, 'alarm_thresholds/10': 4, 'alarm_thresholds/100': 16, 'alarm_thresholds/20': 5, 'alarm_thresholds/200': 29, 'alarm_thresholds/50': 10, 'alarm_thresholds/500': 65, 'auc/different mix of classes/AUC of the domain classifier': 0.727994, 'auc/different mix of classes/coverage of the Δ-learning intervals': 0.849017, 'auc/larger molecules/AUC of the domain classifier': 1.0, 'auc/larger molecules/coverage of the Δ-learning intervals': 0.869224, 'auc/same population (no shift)/AUC of the domain classifier': 0.479679, 'auc/same population (no shift)/coverage of the Δ-learning intervals': 0.901303, 'auc/unseen class (skeletal)/AUC of the domain classifier': 1.0, 'auc/unseen class (skeletal)/coverage of the Δ-learning intervals': 0.652542, 'clusters/Structure-only model/mean coverage, single reactions': 0.900414, 'clusters/Structure-only model/mean coverage, whole reactants': 0.899878, 'clusters/Structure-only model/standard deviation, single reactions': 0.009585, 'clusters/Structure-only model/standard deviation, whole reactants': 0.01172, 'clusters/Δ-learning model/mean coverage, single reactions': 0.900559, 'clusters/Δ-learning model/mean coverage, whole reactants': 0.900106, 'clusters/Δ-learning model/standard deviation, single reactions': 0.009514, 'clusters/Δ-learning model/standard deviation, whole reactants': 0.01193, 'clusters_extra_scatter': 0.238439, 'dependence/Structure-only model/coverage': 0.895457, 'dependence/Structure-only model/expected': 0.900383, 'dependence/Structure-only model/loss': 0.492619, 'dependence/Structure-only model/loss_se': 0.147338, 'dependence/Structure-only model/width_ratio': 0.983442, 'dependence/Δ-learning model/coverage': 0.894696, 'dependence/Δ-learning model/expected': 0.900383, 'dependence/Δ-learning model/loss': 0.568644, 'dependence/Δ-learning model/loss_se': 0.150703, 'dependence/Δ-learning model/width_ratio': 0.974772, 'mix/Structure-only model | Mondrian by class/coverage': 0.902342, 'mix/Structure-only model | Mondrian by class/mean half-width': 21.553901, 'mix/Structure-only model | Mondrian by class/s.d. of coverage': 0.018016, 'mix/Structure-only model | split conformal (no repair)/coverage': 0.88075, 'mix/Structure-only model | split conformal (no repair)/mean half-width': 20.114122, 'mix/Structure-only model | split conformal (no repair)/s.d. of coverage': 0.016321, 'mix/Structure-only model | weighted, class-frequency ratios/coverage': 0.902108, 'mix/Structure-only model | weighted, class-frequency ratios/mean half-width': 21.795685, 'mix/Structure-only model | weighted, class-frequency ratios/s.d. of coverage': 0.019104, 'mix/Δ-learning model | Mondrian by class/coverage': 0.902183, 'mix/Δ-learning model | Mondrian by class/mean half-width': 5.141913, 'mix/Δ-learning model | Mondrian by class/s.d. of coverage': 0.017977, 'mix/Δ-learning model | split conformal (no repair)/coverage': 0.849017, 'mix/Δ-learning model | split conformal (no repair)/mean half-width': 3.805376, 'mix/Δ-learning model | split conformal (no repair)/s.d. of coverage': 0.016601, 'mix/Δ-learning model | weighted, class-frequency ratios/coverage': 0.9013, 'mix/Δ-learning model | weighted, class-frequency ratios/mean half-width': 5.205457, 'mix/Δ-learning model | weighted, class-frequency ratios/s.d. of coverage': 0.01879, 'mix/Δ-learning model | weighted, learned weights/coverage': 0.89725, 'mix/Δ-learning model | weighted, learned weights/mean half-width': 5.048138, 'mix/Δ-learning model | weighted, learned weights/s.d. of coverage': 0.020643, 'mix_diagnostics/AUC': 0.727994, 'mix_diagnostics/effective sample size': 487.211995, 'recalibration/10/5%': 0.734915, 'recalibration/10/95%': 0.995567, 'recalibration/10/mean coverage': 0.906243, 'recalibration/10/median half-width': 5.256745, 'recalibration/100/5%': 0.853206, 'recalibration/100/95%': 0.946966, 'recalibration/100/mean coverage': 0.903878, 'recalibration/100/median half-width': 4.363069, 'recalibration/20/5%': 0.778675, 'recalibration/20/95%': 0.983554, 'recalibration/20/mean coverage': 0.904117, 'recalibration/20/median half-width': 4.695256, 'recalibration/200/5%': 0.858094, 'recalibration/200/95%': 0.934736, 'recalibration/200/mean coverage': 0.90034, 'recalibration/200/median half-width': 4.25858, 'recalibration/50/5%': 0.827336, 'recalibration/50/95%': 0.956625, 'recalibration/50/mean coverage': 0.899304, 'recalibration/50/median half-width': 4.301581, 'recalibration/500/5%': 0.874006, 'recalibration/500/95%': 0.922047, 'recalibration/500/mean coverage': 0.899964, 'recalibration/500/median half-width': 4.221877, 'same_population/delta': 0.901303, 'same_population/structure': 0.899523, 'single_example/coverage_learned': 0.883333, 'single_example/coverage_plain': 0.843333, 'single_example/coverage_weighted': 0.895, 'single_example/q_plain': 3.829824, 'single_example/share_hard_cal': 0.147333, 'size/Structure-only model/MAE, calibration set': 8.14796, 'size/Structure-only model/MAE, larger molecules': 9.227034, 'size/Structure-only model/coverage on larger molecules': 0.882783, 'size/Structure-only model/half-width needed for 90%': 21.254, 'size/Structure-only model/half-width used': 19.594703, 'size/Δ-learning model/MAE, calibration set': 1.557411, 'size/Δ-learning model/MAE, larger molecules': 1.810033, 'size/Δ-learning model/coverage on larger molecules': 0.869224, 'size/Δ-learning model/half-width needed for 90%': 4.205089, 'size/Δ-learning model/half-width used': 3.624266, 'size_auc': 1.0, 'size_weighted_infinite': 1.0, 'unseen_class/Structure-only model | H transfer/MAE, known classes': 8.525765, 'unseen_class/Structure-only model | H transfer/MAE, unseen class': 12.17448, 'unseen_class/Structure-only model | H transfer/coverage after 50 reference calculations': 0.901505, 'unseen_class/Structure-only model | H transfer/coverage on the unseen class': 0.817255, 'unseen_class/Structure-only model | H transfer/half-width after': 25.827954, 'unseen_class/Structure-only model | H transfer/half-width used': 20.258024, 'unseen_class/Structure-only model | carbene/MAE, known classes': 8.81635, 'unseen_class/Structure-only model | carbene/MAE, unseen class': 14.367148, 'unseen_class/Structure-only model | carbene/coverage after 50 reference calculations': 0.904236, 'unseen_class/Structure-only model | carbene/coverage on the unseen class': 0.739382, 'unseen_class/Structure-only model | carbene/half-width after': 29.251705, 'unseen_class/Structure-only model | carbene/half-width used': 20.268273, 'unseen_class/Structure-only model | fragmentation/MAE, known classes': 8.958574, 'unseen_class/Structure-only model | fragmentation/MAE, unseen class': 14.262648, 'unseen_class/Structure-only model | fragmentation/coverage after 50 reference calculations': 0.904588, 'unseen_class/Structure-only model | fragmentation/coverage on the unseen class': 0.752923, 'unseen_class/Structure-only model | fragmentation/half-width after': 28.974119, 'unseen_class/Structure-only model | fragmentation/half-width used': 20.642053, 'unseen_class/Structure-only model | skeletal/MAE, known classes': 8.437048, 'unseen_class/Structure-only model | skeletal/MAE, unseen class': 20.356517, 'unseen_class/Structure-only model | skeletal/coverage after 50 reference calculations': 0.898935, 'unseen_class/Structure-only model | skeletal/coverage on the unseen class': 0.5, 'unseen_class/Structure-only model | skeletal/half-width after': 39.631001, 'unseen_class/Structure-only model | skeletal/half-width used': 19.210691, 'unseen_class/Structure-only model | zwitterion/MAE, known classes': 8.031977, 'unseen_class/Structure-only model | zwitterion/MAE, unseen class': 18.518988, 'unseen_class/Structure-only model | zwitterion/coverage after 50 reference calculations': 0.901222, 'unseen_class/Structure-only model | zwitterion/coverage on the unseen class': 0.531353, 'unseen_class/Structure-only model | zwitterion/half-width after': 34.976957, 'unseen_class/Structure-only model | zwitterion/half-width used': 18.356577, 'unseen_class/Δ-learning model | H transfer/MAE, known classes': 1.539215, 'unseen_class/Δ-learning model | H transfer/MAE, unseen class': 2.590063, 'unseen_class/Δ-learning model | H transfer/coverage after 50 reference calculations': 0.905271, 'unseen_class/Δ-learning model | H transfer/coverage on the unseen class': 0.772549, 'unseen_class/Δ-learning model | H transfer/half-width after': 6.131933, 'unseen_class/Δ-learning model | H transfer/half-width used': 3.604055, 'unseen_class/Δ-learning model | carbene/MAE, known classes': 1.770994, 'unseen_class/Δ-learning model | carbene/MAE, unseen class': 2.102658, 'unseen_class/Δ-learning model | carbene/coverage after 50 reference calculations': 0.904209, 'unseen_class/Δ-learning model | carbene/coverage on the unseen class': 0.853282, 'unseen_class/Δ-learning model | carbene/half-width after': 5.370352, 'unseen_class/Δ-learning model | carbene/half-width used': 4.061002, 'unseen_class/Δ-learning model | fragmentation/MAE, known classes': 1.631336, 'unseen_class/Δ-learning model | fragmentation/MAE, unseen class': 2.389908, 'unseen_class/Δ-learning model | fragmentation/coverage after 50 reference calculations': 0.906926, 'unseen_class/Δ-learning model | fragmentation/coverage on the unseen class': 0.791894, 'unseen_class/Δ-learning model | fragmentation/half-width after': 5.459833, 'unseen_class/Δ-learning model | fragmentation/half-width used': 3.895433, 'unseen_class/Δ-learning model | skeletal/MAE, known classes': 1.474938, 'unseen_class/Δ-learning model | skeletal/MAE, unseen class': 3.453914, 'unseen_class/Δ-learning model | skeletal/coverage after 50 reference calculations': 0.904171, 'unseen_class/Δ-learning model | skeletal/coverage on the unseen class': 0.652542, 'unseen_class/Δ-learning model | skeletal/half-width after': 8.397182, 'unseen_class/Δ-learning model | skeletal/half-width used': 3.491246, 'unseen_class/Δ-learning model | zwitterion/MAE, known classes': 1.679546, 'unseen_class/Δ-learning model | zwitterion/MAE, unseen class': 2.134405, 'unseen_class/Δ-learning model | zwitterion/coverage after 50 reference calculations': 0.899461, 'unseen_class/Δ-learning model | zwitterion/coverage on the unseen class': 0.861386, 'unseen_class/Δ-learning model | zwitterion/half-width after': 4.795599, 'unseen_class/Δ-learning model | zwitterion/half-width used': 3.942115}, 'nb6': {'experiment/delta/A: predicted barrier below c/FDP, 90th percentile': 0.051586, 'experiment/delta/A: predicted barrier below c/FDR (mean FDP)': 0.045783, 'experiment/delta/A: predicted barrier below c/mean FDP of the non-empty lists': 0.045783, 'experiment/delta/A: predicted barrier below c/power (mean)': 0.93399, 'experiment/delta/A: predicted barrier below c/selected (mean)': 345.146667, 'experiment/delta/A: predicted barrier below c/splits with an empty list': 0.0, 'experiment/delta/B: 90% interval below c/FDP, 90th percentile': 0.0, 'experiment/delta/B: 90% interval below c/FDR (mean FDP)': 0.0, 'experiment/delta/B: 90% interval below c/mean FDP of the non-empty lists': 0.0, 'experiment/delta/B: 90% interval below c/power (mean)': 0.695495, 'experiment/delta/B: 90% interval below c/selected (mean)': 245.243333, 'experiment/delta/B: 90% interval below c/splits with an empty list': 0.0, 'experiment/delta/C: p-value below q, no correction | q=0.02/FDP, 90th percentile': 0.276173, 'experiment/delta/C: p-value below q, no correction | q=0.02/FDR (mean FDP)': 0.220016, 'experiment/delta/C: p-value below q, no correction | q=0.02/mean FDP of the non-empty lists': 0.220016, 'experiment/delta/C: p-value below q, no correction | q=0.02/power (mean)': 0.983485, 'experiment/delta/C: p-value below q, no correction | q=0.02/selected (mean)': 445.956667, 'experiment/delta/C: p-value below q, no correction | q=0.02/splits with an empty list': 0.0, 'experiment/delta/C: p-value below q, no correction | q=0.05/FDP, 90th percentile': 0.462179, 'experiment/delta/C: p-value below q, no correction | q=0.05/FDR (mean FDP)': 0.414612, 'experiment/delta/C: p-value below q, no correction | q=0.05/mean FDP of the non-empty lists': 0.414612, 'experiment/delta/C: p-value below q, no correction | q=0.05/power (mean)': 0.986388, 'experiment/delta/C: p-value below q, no correction | q=0.05/selected (mean)': 596.466667, 'experiment/delta/C: p-value below q, no correction | q=0.05/splits with an empty list': 0.0, 'experiment/delta/C: p-value below q, no correction | q=0.10/FDP, 90th percentile': 0.616568, 'experiment/delta/C: p-value below q, no correction | q=0.10/FDR (mean FDP)': 0.584545, 'experiment/delta/C: p-value below q, no correction | q=0.10/mean FDP of the non-empty lists': 0.584545, 'experiment/delta/C: p-value below q, no correction | q=0.10/power (mean)': 0.995199, 'experiment/delta/C: p-value below q, no correction | q=0.10/selected (mean)': 847.54, 'experiment/delta/C: p-value below q, no correction | q=0.10/splits with an empty list': 0.0, 'experiment/delta/C: p-value below q, no correction | q=0.20/FDP, 90th percentile': 0.75504, 'experiment/delta/C: p-value below q, no correction | q=0.20/FDR (mean FDP)': 0.736369, 'experiment/delta/C: p-value below q, no correction | q=0.20/mean FDP of the non-empty lists': 0.736369, 'experiment/delta/C: p-value below q, no correction | q=0.20/power (mean)': 0.997646, 'experiment/delta/C: p-value below q, no correction | q=0.20/selected (mean)': 1338.24, 'experiment/delta/C: p-value below q, no correction | q=0.20/splits with an empty list': 0.0, 'experiment/delta/C: p-value below q, no correction | q=0.30/FDP, 90th percentile': 0.818615, 'experiment/delta/C: p-value below q, no correction | q=0.30/FDR (mean FDP)': 0.807298, 'experiment/delta/C: p-value below q, no correction | q=0.30/mean FDP of the non-empty lists': 0.807298, 'experiment/delta/C: p-value below q, no correction | q=0.30/power (mean)': 1.0, 'experiment/delta/C: p-value below q, no correction | q=0.30/selected (mean)': 1833.36, 'experiment/delta/C: p-value below q, no correction | q=0.30/splits with an empty list': 0.0, 'experiment/delta/conformal selection | q=0.02/FDP, 90th percentile': 0.037167, 'experiment/delta/conformal selection | q=0.02/FDR (mean FDP)': 0.015907, 'experiment/delta/conformal selection | q=0.02/mean FDP of the non-empty lists': 0.015907, 'experiment/delta/conformal selection | q=0.02/power (mean)': 0.838402, 'experiment/delta/conformal selection | q=0.02/selected (mean)': 300.733333, 'experiment/delta/conformal selection | q=0.02/splits with an empty list': 0.0, 'experiment/delta/conformal selection | q=0.05/FDP, 90th percentile': 0.0758, 'experiment/delta/conformal selection | q=0.05/FDR (mean FDP)': 0.044451, 'experiment/delta/conformal selection | q=0.05/mean FDP of the non-empty lists': 0.044451, 'experiment/delta/conformal selection | q=0.05/power (mean)': 0.922244, 'experiment/delta/conformal selection | q=0.05/selected (mean)': 340.86, 'experiment/delta/conformal selection | q=0.05/splits with an empty list': 0.0, 'experiment/delta/conformal selection | q=0.10/FDP, 90th percentile': 0.151256, 'experiment/delta/conformal selection | q=0.10/FDR (mean FDP)': 0.097613, 'experiment/delta/conformal selection | q=0.10/mean FDP of the non-empty lists': 0.097613, 'experiment/delta/conformal selection | q=0.10/power (mean)': 0.968289, 'experiment/delta/conformal selection | q=0.10/selected (mean)': 379.313333, 'experiment/delta/conformal selection | q=0.10/splits with an empty list': 0.0, 'experiment/delta/conformal selection | q=0.20/FDP, 90th percentile': 0.268969, 'experiment/delta/conformal selection | q=0.20/FDR (mean FDP)': 0.196612, 'experiment/delta/conformal selection | q=0.20/mean FDP of the non-empty lists': 0.196612, 'experiment/delta/conformal selection | q=0.20/power (mean)': 0.982048, 'experiment/delta/conformal selection | q=0.20/selected (mean)': 433.023333, 'experiment/delta/conformal selection | q=0.20/splits with an empty list': 0.0, 'experiment/delta/conformal selection | q=0.30/FDP, 90th percentile': 0.378764, 'experiment/delta/conformal selection | q=0.30/FDR (mean FDP)': 0.296837, 'experiment/delta/conformal selection | q=0.30/mean FDP of the non-empty lists': 0.296837, 'experiment/delta/conformal selection | q=0.30/power (mean)': 0.985516, 'experiment/delta/conformal selection | q=0.30/selected (mean)': 497.91, 'experiment/delta/conformal selection | q=0.30/splits with an empty list': 0.0, 'experiment/structure/A: predicted barrier below c/FDP, 90th percentile': 0.242798, 'experiment/structure/A: predicted barrier below c/FDR (mean FDP)': 0.229084, 'experiment/structure/A: predicted barrier below c/mean FDP of the non-empty lists': 0.229084, 'experiment/structure/A: predicted barrier below c/power (mean)': 0.525224, 'experiment/structure/A: predicted barrier below c/selected (mean)': 240.243333, 'experiment/structure/A: predicted barrier below c/splits with an empty list': 0.0, 'experiment/structure/B: 90% interval below c/FDP, 90th percentile': 0.0, 'experiment/structure/B: 90% interval below c/FDR (mean FDP)': 0.0, 'experiment/structure/B: 90% interval below c/mean FDP of the non-empty lists': 0.0, 'experiment/structure/B: 90% interval below c/power (mean)': 0.001361, 'experiment/structure/B: 90% interval below c/selected (mean)': 0.48, 'experiment/structure/B: 90% interval below c/splits with an empty list': 0.52, 'experiment/structure/C: p-value below q, no correction | q=0.02/FDP, 90th percentile': 0.366717, 'experiment/structure/C: p-value below q, no correction | q=0.02/FDR (mean FDP)': 0.313124, 'experiment/structure/C: p-value below q, no correction | q=0.02/mean FDP of the non-empty lists': 0.313124, 'experiment/structure/C: p-value below q, no correction | q=0.02/power (mean)': 0.60226, 'experiment/structure/C: p-value below q, no correction | q=0.02/selected (mean)': 311.436667, 'experiment/structure/C: p-value below q, no correction | q=0.02/splits with an empty list': 0.0, 'experiment/structure/C: p-value below q, no correction | q=0.05/FDP, 90th percentile': 0.530085, 'experiment/structure/C: p-value below q, no correction | q=0.05/FDR (mean FDP)': 0.48837, 'experiment/structure/C: p-value below q, no correction | q=0.05/mean FDP of the non-empty lists': 0.48837, 'experiment/structure/C: p-value below q, no correction | q=0.05/power (mean)': 0.727258, 'experiment/structure/C: p-value below q, no correction | q=0.05/selected (mean)': 503.75, 'experiment/structure/C: p-value below q, no correction | q=0.05/splits with an empty list': 0.0, 'experiment/structure/C: p-value below q, no correction | q=0.10/FDP, 90th percentile': 0.648387, 'experiment/structure/C: p-value below q, no correction | q=0.10/FDR (mean FDP)': 0.620886, 'experiment/structure/C: p-value below q, no correction | q=0.10/mean FDP of the non-empty lists': 0.620886, 'experiment/structure/C: p-value below q, no correction | q=0.10/power (mean)': 0.847397, 'experiment/structure/C: p-value below q, no correction | q=0.10/selected (mean)': 790.963333, 'experiment/structure/C: p-value below q, no correction | q=0.10/splits with an empty list': 0.0, 'experiment/structure/C: p-value below q, no correction | q=0.20/FDP, 90th percentile': 0.770034, 'experiment/structure/C: p-value below q, no correction | q=0.20/FDR (mean FDP)': 0.75225, 'experiment/structure/C: p-value below q, no correction | q=0.20/mean FDP of the non-empty lists': 0.75225, 'experiment/structure/C: p-value below q, no correction | q=0.20/power (mean)': 0.919259, 'experiment/structure/C: p-value below q, no correction | q=0.20/selected (mean)': 1311.986667, 'experiment/structure/C: p-value below q, no correction | q=0.20/splits with an empty list': 0.0, 'experiment/structure/C: p-value below q, no correction | q=0.30/FDP, 90th percentile': 0.825654, 'experiment/structure/C: p-value below q, no correction | q=0.30/FDR (mean FDP)': 0.814478, 'experiment/structure/C: p-value below q, no correction | q=0.30/mean FDP of the non-empty lists': 0.814478, 'experiment/structure/C: p-value below q, no correction | q=0.30/power (mean)': 0.954038, 'experiment/structure/C: p-value below q, no correction | q=0.30/selected (mean)': 1816.86, 'experiment/structure/C: p-value below q, no correction | q=0.30/splits with an empty list': 0.0, 'experiment/structure/conformal selection | q=0.02/FDP, 90th percentile': 0.0, 'experiment/structure/conformal selection | q=0.02/FDR (mean FDP)': 0.0, 'experiment/structure/conformal selection | q=0.02/power (mean)': 0.0, 'experiment/structure/conformal selection | q=0.02/selected (mean)': 0.0, 'experiment/structure/conformal selection | q=0.02/splits with an empty list': 1.0, 'experiment/structure/conformal selection | q=0.05/FDP, 90th percentile': 0.114396, 'experiment/structure/conformal selection | q=0.05/FDR (mean FDP)': 0.019036, 'experiment/structure/conformal selection | q=0.05/mean FDP of the non-empty lists': 0.126904, 'experiment/structure/conformal selection | q=0.05/power (mean)': 0.044021, 'experiment/structure/conformal selection | q=0.05/selected (mean)': 17.866667, 'experiment/structure/conformal selection | q=0.05/splits with an empty list': 0.85, 'experiment/structure/conformal selection | q=0.10/FDP, 90th percentile': 0.17322, 'experiment/structure/conformal selection | q=0.10/FDR (mean FDP)': 0.077135, 'experiment/structure/conformal selection | q=0.10/mean FDP of the non-empty lists': 0.120524, 'experiment/structure/conformal selection | q=0.10/power (mean)': 0.192577, 'experiment/structure/conformal selection | q=0.10/selected (mean)': 79.126667, 'experiment/structure/conformal selection | q=0.10/splits with an empty list': 0.36, 'experiment/structure/conformal selection | q=0.20/FDP, 90th percentile': 0.291403, 'experiment/structure/conformal selection | q=0.20/FDR (mean FDP)': 0.197565, 'experiment/structure/conformal selection | q=0.20/mean FDP of the non-empty lists': 0.198226, 'experiment/structure/conformal selection | q=0.20/power (mean)': 0.458535, 'experiment/structure/conformal selection | q=0.20/selected (mean)': 208.003333, 'experiment/structure/conformal selection | q=0.20/splits with an empty list': 0.003333, 'experiment/structure/conformal selection | q=0.30/FDP, 90th percentile': 0.394324, 'experiment/structure/conformal selection | q=0.30/FDR (mean FDP)': 0.295267, 'experiment/structure/conformal selection | q=0.30/mean FDP of the non-empty lists': 0.295267, 'experiment/structure/conformal selection | q=0.30/power (mean)': 0.5821, 'experiment/structure/conformal selection | q=0.30/selected (mean)': 298.74, 'experiment/structure/conformal selection | q=0.30/splits with an empty list': 0.0, 'eyring_half_life_ms': 2.813669, 'hit_rate_hold': 0.071272, 'n_candidates': 4935, 'resolution/Structure-only model | n=100/FDR': 0.002672, 'resolution/Structure-only model | n=100/power': 0.003759, 'resolution/Structure-only model | n=100/smallest non-empty list observed': 670, 'resolution/Structure-only model | n=100/smallest possible list m/(q(n+1))': 578, 'resolution/Structure-only model | n=100/splits with an empty list': 0.995, 'resolution/Structure-only model | n=1000/FDR': 0.080344, 'resolution/Structure-only model | n=1000/power': 0.198958, 'resolution/Structure-only model | n=1000/smallest non-empty list observed': 58, 'resolution/Structure-only model | n=1000/smallest possible list m/(q(n+1))': 50, 'resolution/Structure-only model | n=1000/splits with an empty list': 0.31, 'resolution/Structure-only model | n=200/FDR': 0.020917, 'resolution/Structure-only model | n=200/power': 0.042639, 'resolution/Structure-only model | n=200/smallest non-empty list observed': 286, 'resolution/Structure-only model | n=200/smallest possible list m/(q(n+1))': 286, 'resolution/Structure-only model | n=200/splits with an empty list': 0.925, 'resolution/Structure-only model | n=2000/FDR': 0.090403, 'resolution/Structure-only model | n=2000/power': 0.235114, 'resolution/Structure-only model | n=2000/smallest non-empty list observed': 20, 'resolution/Structure-only model | n=2000/smallest possible list m/(q(n+1))': 20, 'resolution/Structure-only model | n=2000/splits with an empty list': 0.03, 'resolution/Structure-only model | n=500/FDR': 0.059026, 'resolution/Structure-only model | n=500/power': 0.141929, 'resolution/Structure-only model | n=500/smallest non-empty list observed': 109, 'resolution/Structure-only model | n=500/smallest possible list m/(q(n+1))': 109, 'resolution/Structure-only model | n=500/splits with an empty list': 0.61, 'resolution/Δ-learning model | n=100/FDR': 0.016041, 'resolution/Δ-learning model | n=100/power': 0.044351, 'resolution/Δ-learning model | n=100/smallest non-empty list observed': 592, 'resolution/Δ-learning model | n=100/smallest possible list m/(q(n+1))': 578, 'resolution/Δ-learning model | n=100/splits with an empty list': 0.955, 'resolution/Δ-learning model | n=1000/FDR': 0.094574, 'resolution/Δ-learning model | n=1000/power': 0.96755, 'resolution/Δ-learning model | n=1000/smallest non-empty list observed': 312, 'resolution/Δ-learning model | n=1000/smallest possible list m/(q(n+1))': 50, 'resolution/Δ-learning model | n=1000/splits with an empty list': 0.0, 'resolution/Δ-learning model | n=200/FDR': 0.065424, 'resolution/Δ-learning model | n=200/power': 0.924569, 'resolution/Δ-learning model | n=200/smallest non-empty list observed': 307, 'resolution/Δ-learning model | n=200/smallest possible list m/(q(n+1))': 286, 'resolution/Δ-learning model | n=200/splits with an empty list': 0.0, 'resolution/Δ-learning model | n=2000/FDR': 0.097194, 'resolution/Δ-learning model | n=2000/power': 0.972028, 'resolution/Δ-learning model | n=2000/smallest non-empty list observed': 262, 'resolution/Δ-learning model | n=2000/smallest possible list m/(q(n+1))': 20, 'resolution/Δ-learning model | n=2000/splits with an empty list': 0.0, 'resolution/Δ-learning model | n=500/FDR': 0.089325, 'resolution/Δ-learning model | n=500/power': 0.95628, 'resolution/Δ-learning model | n=500/smallest non-empty list observed': 331, 'resolution/Δ-learning model | n=500/smallest possible list m/(q(n+1))': 109, 'resolution/Δ-learning model | n=500/splits with an empty list': 0.0, 'scores/Structure-only model | q=0.05 | FDR | (mu - c) / sigma': 0.023545, 'scores/Structure-only model | q=0.05 | FDR | mu - c': 0.019036, 'scores/Structure-only model | q=0.05 | power | (mu - c) / sigma': 0.065497, 'scores/Structure-only model | q=0.05 | power | mu - c': 0.044021, 'scores/Structure-only model | q=0.10 | FDR | (mu - c) / sigma': 0.083131, 'scores/Structure-only model | q=0.10 | FDR | mu - c': 0.077135, 'scores/Structure-only model | q=0.10 | power | (mu - c) / sigma': 0.291772, 'scores/Structure-only model | q=0.10 | power | mu - c': 0.192577, 'scores/Structure-only model | q=0.20 | FDR | (mu - c) / sigma': 0.194856, 'scores/Structure-only model | q=0.20 | FDR | mu - c': 0.197565, 'scores/Structure-only model | q=0.20 | power | (mu - c) / sigma': 0.466433, 'scores/Structure-only model | q=0.20 | power | mu - c': 0.458535, 'scores/Δ-learning model | q=0.05 | FDR | (mu - c) / sigma': 0.043934, 'scores/Δ-learning model | q=0.05 | FDR | mu - c': 0.044451, 'scores/Δ-learning model | q=0.05 | power | (mu - c) / sigma': 0.92334, 'scores/Δ-learning model | q=0.05 | power | mu - c': 0.922244, 'scores/Δ-learning model | q=0.10 | FDR | (mu - c) / sigma': 0.095122, 'scores/Δ-learning model | q=0.10 | FDR | mu - c': 0.097613, 'scores/Δ-learning model | q=0.10 | power | (mu - c) / sigma': 0.97213, 'scores/Δ-learning model | q=0.10 | power | mu - c': 0.968289, 'scores/Δ-learning model | q=0.20 | FDR | (mu - c) / sigma': 0.19645, 'scores/Δ-learning model | q=0.20 | FDR | mu - c': 0.196612, 'scores/Δ-learning model | q=0.20 | power | (mu - c) / sigma': 0.987454, 'scores/Δ-learning model | q=0.20 | power | mu - c': 0.982048, 'single_split/Structure-only model | A: predicted barrier below c/FDP': 0.228448, 'single_split/Structure-only model | A: predicted barrier below c/false discoveries': 53, 'single_split/Structure-only model | A: predicted barrier below c/hits found': 179, 'single_split/Structure-only model | A: predicted barrier below c/power': 0.523392, 'single_split/Structure-only model | A: predicted barrier below c/selected': 232, 'single_split/Structure-only model | B: 90% interval entirely below c/FDP': 0.0, 'single_split/Structure-only model | B: 90% interval entirely below c/false discoveries': 0, 'single_split/Structure-only model | B: 90% interval entirely below c/hits found': 0, 'single_split/Structure-only model | B: 90% interval entirely below c/power': 0.0, 'single_split/Structure-only model | B: 90% interval entirely below c/selected': 0, 'single_split/Structure-only model | C: p-value below 0.10 for each reaction/FDP': 0.652071, 'single_split/Structure-only model | C: p-value below 0.10 for each reaction/false discoveries': 551, 'single_split/Structure-only model | C: p-value below 0.10 for each reaction/hits found': 294, 'single_split/Structure-only model | C: p-value below 0.10 for each reaction/power': 0.859649, 'single_split/Structure-only model | C: p-value below 0.10 for each reaction/selected': 845, 'single_split/Δ-learning model | A: predicted barrier below c/FDP': 0.047904, 'single_split/Δ-learning model | A: predicted barrier below c/false discoveries': 16, 'single_split/Δ-learning model | A: predicted barrier below c/hits found': 318, 'single_split/Δ-learning model | A: predicted barrier below c/power': 0.929825, 'single_split/Δ-learning model | A: predicted barrier below c/selected': 334, 'single_split/Δ-learning model | B: 90% interval entirely below c/FDP': 0.0, 'single_split/Δ-learning model | B: 90% interval entirely below c/false discoveries': 0, 'single_split/Δ-learning model | B: 90% interval entirely below c/hits found': 241, 'single_split/Δ-learning model | B: 90% interval entirely below c/power': 0.704678, 'single_split/Δ-learning model | B: 90% interval entirely below c/selected': 241, 'single_split/Δ-learning model | C: p-value below 0.10 for each reaction/FDP': 0.601874, 'single_split/Δ-learning model | C: p-value below 0.10 for each reaction/false discoveries': 514, 'single_split/Δ-learning model | C: p-value below 0.10 for each reaction/hits found': 340, 'single_split/Δ-learning model | C: p-value below 0.10 for each reaction/power': 0.994152, 'single_split/Δ-learning model | C: p-value below 0.10 for each reaction/selected': 854, 'single_split_bh/delta/FDP': 0.119048, 'single_split_bh/delta/false discoveries': 45, 'single_split_bh/delta/hits found': 333, 'single_split_bh/delta/power': 0.973684, 'single_split_bh/delta/selected': 378, 'single_split_bh/structure/FDP': 0.137097, 'single_split_bh/structure/false discoveries': 17, 'single_split_bh/structure/hits found': 107, 'single_split_bh/structure/power': 0.312865, 'single_split_bh/structure/selected': 124, 'threshold': 50.0, 'versus_rule_a/Structure-only model/FDR of conformal selection at q = FDR of rule A': 0.229291, 'versus_rule_a/Structure-only model/FDR of rule A': 0.229084, 'versus_rule_a/Structure-only model/its power': 0.505286, 'versus_rule_a/Structure-only model/power of rule A': 0.525224, 'versus_rule_a/Δ-learning model/FDR of conformal selection at q = FDR of rule A': 0.042549, 'versus_rule_a/Δ-learning model/FDR of rule A': 0.045783, 'versus_rule_a/Δ-learning model/its power': 0.915492, 'versus_rule_a/Δ-learning model/power of rule A': 0.93399}})
# <<< reference results


def compare(name: str, results: dict) -> dict:
    """Compare the numbers of one notebook with the reference run.

    Returns the lists of keys that are 'identical', 'close', 'different' or 'missing'. Close means:
    within 0.02 for numbers between -1 and 1 (coverages, rates, fractions) and within 5% for larger
    ones; counts that do not depend on the fitted models (sizes of data sets, ranks, thresholds)
    must be identical. A number that is absent or not finite counts as missing.
    """
    ref = {k: v for k, v in REFERENCE_RESULTS.get(name, {}).items() if not isinstance(v, (str, bool))}
    new = flatten(_plain(results))
    out = {"identical": [], "close": [], "different": [], "missing": []}
    for key, expected in ref.items():
        value = new.get(key)
        if key not in new or isinstance(value, (str, bool)):
            out["missing"].append(key)
        elif expected is None:                                # infinite in the reference run
            out["identical" if value is None else "different"].append(key)
        elif value is None:                                   # finite in the reference run, not finite here
            out["missing"].append(key)
        elif value == expected:
            out["identical"].append(key)
        elif isinstance(expected, int) and not any(tag in key for tag in MODEL_DEPENDENT_COUNTS):
            out["different"].append(key)
        elif abs(value - expected) <= (0.02 if abs(expected) <= 1 else 0.05 * abs(expected)):
            out["close"].append(key)
        else:
            out["different"].append(key)
    out["reference"], out["new"] = ref, new
    return out


def _what_differs() -> str:
    """One sentence on how this environment differs from that of the reference run."""
    ref_env, env = REFERENCE_RESULTS.get("_environment", {}), environment()
    changed = [f"{lib} {env.get(lib)} (reference run: {ref_env.get(lib)})" for lib in LIBRARIES
               if ref_env.get(lib) is not None and env.get(lib) != ref_env.get(lib)]
    if not ref_env:
        return ""
    if changed:
        return ("   This environment differs from that of the reference run in: " + "; ".join(changed) + ".\n"
                "   Small differences are then expected: different library versions do not fit gradient-boosted trees identically.")
    return ("   The library versions are those of the reference run, so the numbers should be identical. If you have changed\n"
            "   the notebook (for example for an exercise), differences are expected. Otherwise check that primer.py is the\n"
            "   one that belongs to this notebook and that the data were reported as the Zenodo files.")


def check(name: str, results: dict) -> bool:
    """Print how the numbers just computed compare with the reference run; True if none differs or is missing."""
    if not REFERENCE_RESULTS.get(name):
        print("No reference values are stored for this notebook yet.")
        return True
    c = compare(name, results)
    total = sum(len(c[k]) for k in ("identical", "close", "different", "missing"))
    bad = len(c["different"]) + len(c["missing"])
    line = f"reference check: {total} numbers compared with the reference run: {len(c['identical'])} identical"
    if c["close"]:
        line += f", {len(c['close'])} close (within 0.02 for numbers between −1 and 1, within 5% otherwise)"
    if bad:
        line += f", {len(c['different'])} different, {len(c['missing'])} missing or not finite"
    print(line)

    def size(key):                                            # relative size of a difference, for sorting
        expected, value = c["reference"].get(key), c["new"].get(key)
        if expected is None or value is None:
            return np.inf
        return abs(value - expected) / max(abs(expected), 0.4)

    shown = sorted(c["different"], key=size, reverse=True)[:8]
    for key in shown:
        expected, value = c["reference"].get(key), c["new"].get(key)
        print(f"   different: {key}: reference {'infinite' if expected is None else format(expected, '.6g')}, "
              f"here {'not finite' if value is None else format(value, '.6g')}")
    for key in c["missing"][:4]:
        print(f"   missing: {key}")
    rest = bad - len(shown) - min(len(c["missing"]), 4)
    if rest > 0:
        print(f"   ... and {rest} more")
    note = _what_differs()
    if (bad or c["close"]) and note:
        print(note)
    return not bad


def check_all(out_dir: str = "outputs") -> dict:
    """Compare the results of all six notebooks with the reference run; one line per notebook.

    Returns the totals {'identical', 'close', 'different', 'missing', 'not_run', 'data_status'}."""
    totals = {"identical": 0, "close": 0, "different": 0, "missing": 0, "not_run": [], "data_status": {}}
    for name in NOTEBOOKS:
        path = os.path.join(out_dir, f"{name}_results.json")
        if not os.path.exists(path):
            print(f"{name}: no results found (the notebook was not run)")
            totals["not_run"].append(name)
            totals["missing"] += len(REFERENCE_RESULTS.get(name, {}))
            continue
        with open(path, encoding="utf-8") as fh:
            payload = json.load(fh)
        c = compare(name, payload["results"])
        for k in ("identical", "close", "different", "missing"):
            totals[k] += len(c[k])
        totals["data_status"].update(payload.get("data_status", {}))
        print(f"{name}: {len(c['identical'])} identical, {len(c['close'])} close, "
              f"{len(c['different'])} different, {len(c['missing'])} missing")
    n = totals["identical"] + totals["close"] + totals["different"] + totals["missing"]
    print(f"all notebooks: {totals['identical']} of {n} numbers identical to the reference run, {totals['close']} close, "
          f"{totals['different']} different, {totals['missing']} missing")
    note = _what_differs()
    if totals["identical"] < n and note:
        print(note)
    return totals
'''

if not os.path.exists("primer.py"):
    with open("primer.py", "w", encoding="utf-8") as fh:
        fh.write(PRIMER_SOURCE)
elif open("primer.py", encoding="utf-8").read() != PRIMER_SOURCE:
    print("NOTE: a file primer.py that differs from the copy embedded in this notebook lies next to the notebook,\n"
          "      and that FILE is used. If you did not edit it yourself, delete primer.py and run this cell again\n"
          "      to get the version that belongs to this notebook.")

if "primer" in sys.modules:                 # the cell is run again: make sure that the module is the file on disk
    P = importlib.reload(sys.modules["primer"])
else:
    import primer as P
try:                                        # show figures inside the notebook (some older Matplotlib versions need this)
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
T_START = time.time()
print("primer", P.__version__, "ready")

# Notebook 1 · Two models, and why an MAE is not an error bar

**Conformal Prediction for Physical Chemists: hands-on notebooks (1 of 6)**

This notebook sets the stage. It introduces the data set and the two machine-learning models that are used
in all later notebooks, and it shows what is missing when a model is described only by its mean absolute error (MAE).

**You will**

1. look at the RDB7 data set of reaction barrier heights and at how a reaction is turned into numbers;
2. meet the two point predictors of the tutorial, a *structure-only* model and a *Δ-learning* model (Ramakrishnan
   et al., 2015);
3. see that a mean absolute error says little about how wrong a single prediction can be;
4. see that the usual Gaussian "±σ" error bar does not deliver the *coverage* it promises (coverage: the fraction of
   reactions whose reference value lies inside the error bar).

Runs on a CPU in under a minute. No conformal prediction yet: that starts in Notebook 2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

P.set_style()
rdb7 = P.load_problem()      # downloads RDB7 once, builds features, fits both models
y = rdb7.y                   # CCSD(T)-F12a barrier heights in kcal/mol: the quantity we want to predict

## 1. The data: RDB7

RDB7 (Spiekermann, Pattanaik, and Green, *Sci. Data* **2022**, *9*, 417) contains 11,926 gas-phase reactions of
molecules built from H, C, N, and O with at most seven heavy atoms. Every reaction has one reactant and one to three
products. The reactions were found by automated reaction discovery (Grambow, Pattanaik, and Green, 2020), not collected
from textbooks, which is why unusual, high-energy products are common. For each reaction the data set gives
*atom-mapped* SMILES strings (every atom carries a number that identifies it on both sides of the reaction) and the
barrier height at several levels of theory, of which we use two:

| column here | level of theory | role in this tutorial |
|---|---|---|
| `barrier_cc` | CCSD(T)-F12a/cc-pVDZ-F12//ωB97X-D3/def2-TZVP | the reference value we want to predict |
| `barrier_dft` | ωB97X-D3/def2-TZVP | a cheaper estimate, used as an input by the Δ-learning model |

A barrier height is the energy of the transition state minus that of the reactant, including zero-point energy, in
kcal/mol. The lines printed above confirm that the two tables are the ones deposited on Zenodo
(DOI 10.5281/zenodo.6618262).

**Reference values are not the truth.** Throughout these notebooks the "reference barrier" $y$ is the CCSD(T)-F12a
number. It is a high-level estimate, but it has errors of its own: a double-zeta F12 basis set, a single-reference
treatment of transition states that can have multireference character, and geometries and harmonic zero-point energies
taken from density functional theory (DFT). Every statement about errors, coverage, and intervals that follows is a
statement about this reference level.

In [ ]:
table = rdb7.df[["idx", "barrier_cc", "barrier_dft", "dH_cc", "rmg_family"]].head(5).copy()
table.insert(1, "reaction", [f"{P.readable(r)} >> {P.readable(p)}" for r, p in zip(rdb7.df.rsmi[:5], rdb7.df.psmi[:5])])
table["rmg_family"] = table["rmg_family"].fillna("–")
display(table.round(2))
print(f"barrier heights: mean {y.mean():.1f}, standard deviation {y.std():.1f}, range {y.min():.1f} to {y.max():.1f} kcal/mol")
print(f"reactions per reactant molecule: {rdb7.n / len(set(rdb7.reactant)):.1f} on average")

(`dH_cc` is the reaction enthalpy at 298 K at the reference level, shown for orientation; `rmg_family` is explained
next. In the SMILES strings, `>>` separates the reactant from the products, a dot separates two product molecules,
lower-case letters are aromatic atoms, square brackets hold atoms with a formal charge, an unusual valence, or a stereo
label (`@`, `@@`), and `/` and `\` give the geometry at a double bond; `[C]`, `[CH]`, and `[CH2]` are carbene carbons
(in a few cases radical centers).)

### Reaction classes

Later notebooks need to know what *kind* of reaction each entry is. The authors of RDB7 assigned a reaction family of
the Reaction Mechanism Generator (RMG; Gao et al., 2016) to about 12% of the reactions; the rest match no template. We
therefore use a coarser classification that can be assigned to every reaction automatically, from the way the products
are written and from the bonds that are broken. The rules are applied in this order, and the first one that applies
decides (see `primer.reaction_class`):

| class | rule (structures as RDB7 writes them) | what it contains |
|---|---|---|
| carbene product | a product has a divalent carbon atom | carbenes left behind by 1,2-shifts, ring openings, and extrusions (in about 2% of the cases a nitrene or two radical centers instead) |
| zwitterionic product | a product is written with formal charges | heteroatom-stabilized carbenes drawn as ylides (three cases in ten); ylides, diazo compounds, and other structures with the charges on neighboring atoms (one third); zwitterions with the charges farther apart (one third) |
| fragmentation | two or three product molecules | loss of H₂, H₂O, CH₂O, C₂H₄, CH₄, or CO in seven cases out of ten; retro-ene reactions and retro-cycloadditions |
| H-transfer isomerization | one product; at least one X–H bond is broken | tautomerizations and hydrogen shifts, in six cases out of ten combined with a change of the heavy-atom skeleton |
| skeletal rearrangement | one product; no X–H bond is broken | ring contractions and expansions, shifts of heavy-atom groups, ring openings, and cyclizations |

These are **bookkeeping classes**, read off the Lewis structures in the data set; they are not mechanistic classes.
Three details matter. Carbon monoxide and isocyanides are conventionally written with formal charges
(`[C-]#[O+]`, `[C-]#[N+]R`); these are not counted as charge separation, so the loss of CO alone is a fragmentation.
Hydrogen isocyanide, which RDB7 writes as the carbene `[C]=N`, counts as a carbene product. And the first two classes
overlap chemically: a heteroatom-stabilized carbene lands in one or the other depending on which resonance structure
the data set happens to use. (The tables below list the classes in a fixed order, which is not the order of the rules.)

In [ ]:
classes = pd.Series(rdb7.rxn_class)
overview = pd.DataFrame({
    "reactions": classes.value_counts().reindex(P.CLASS_ORDER),
    "mean barrier (kcal/mol)": pd.Series(y).groupby(classes.values).mean().reindex(P.CLASS_ORDER).round(1),
})
EXAMPLES = {"H-transfer isomerization": 75, "skeletal rearrangement": 8, "fragmentation": 21,      # row numbers of one
            "carbene product": 7, "zwitterionic product": 0}                                        # clear example each
examples = []
for c in P.CLASS_ORDER:
    row = EXAMPLES[c]
    assert rdb7.rxn_class[row] == c
    examples.append(f"{P.readable(rdb7.df.rsmi[row])} >> {P.readable(rdb7.df.psmi[row])}")
overview["example"] = examples
display(overview)
print(f"reactions with an RMG family label: {rdb7.df.rmg_family.notna().mean():.1%}")

## 2. From a reaction to numbers

The atom maps tell us which bonds are broken, formed, or change their order. Counting these events by bond type gives
a small set of interpretable descriptors. *Morgan fingerprints* (counts of the circular atom environments that occur in
a molecule, up to two bonds around each atom; Rogers and Hahn, 2010) of the reactant, and the *difference* between the
product and reactant fingerprints, add information about the molecular environment. All of this is computed with RDKit.
Here is the keto–enol tautomerization of butanal and its nonzero descriptors:

In [ ]:
i = EXAMPLES["H-transfer isomerization"]
descriptors, fp_reactant, fp_difference = P.describe_reaction(rdb7.df.rsmi[i], rdb7.df.psmi[i])
print(P.readable(rdb7.df.rsmi[i]), ">>", P.readable(rdb7.df.psmi[i]), f"   (barrier {y[i]:.1f} kcal/mol)\n")
print({name: value for name, value in descriptors.items() if value != 0})
print(f"\nreactant fingerprint: {np.count_nonzero(fp_reactant)} nonzero entries; "
      f"difference fingerprint: {np.count_nonzero(fp_difference)} nonzero entries")

Read the names as follows: `broken_H-C` is the number of C–H bonds broken, `formed_H-O` the number of O–H bonds formed,
`order_down_C-O` the number of C–O bonds whose order decreases (here C=O becomes C–O), `order_up_C-C` the number of C–C
bonds whose order increases, `n_reacting_atoms` the number of atoms that take part in any of these changes.

## 3. Two point predictors

Both models are gradient-boosted regression trees (Friedman, 2001) from scikit-learn (Pedregosa et al., 2011): 300
trees, learning rate 0.1, no early stopping, all other settings at their defaults. Of the fingerprint entries, the
structure-only model keeps those that are nonzero in at least 5% of the training reactions. The models differ in what
they are given.

| | **Structure-only model** | **Δ-learning model** |
|---|---|---|
| input | descriptors and fingerprints of the reaction | descriptors **plus** the ωB97X-D3 barrier and reaction enthalpy |
| output | the CCSD(T)-F12a barrier | the *correction* from the ωB97X-D3 to the CCSD(T)-F12a barrier |
| what a prediction requires | only the reaction itself (reactant, products, and atom map) | a DFT transition-state search, with optimizations and frequency calculations for reactant, transition state, and products |
| typical use | screening very many candidate reactions | refining barriers that were already computed with DFT |
| name in the code | `rdb7.models["structure"]` | `rdb7.models["delta"]` |

Reference and ωB97X-D3 values share geometries and harmonic frequencies, so the Δ-learning model learns a difference of
electronic energies at fixed geometry; what it saves are the coupled-cluster single-point calculations.

The models are deliberately simple and fast. They are not the state of the art: a graph neural network that was
pretrained on lower-level DFT barriers and then trained on these data (forward and reverse reactions, about 24,000 in
all) reaches a test MAE of 2.6 kcal/mol from the atom-mapped structures of reactant and product alone (Spiekermann,
Pattanaik, and Green, *J. Phys. Chem. A* 2022; their test set is split by the scaffold of the reactant, so the number
is not directly comparable with ours). What we need here is one weak and one strong predictor, because the subject of
the notebooks is what can be said about the errors of a *given* model.

The data are split once into a training half and a held-out half. The split is made **by reactant**: all reactions of a
given reactant molecule go to the same side, so the models are judged on reactant molecules they have never seen
(Notebook 5 comes back to why this matters). The split is by reactant only: in about 70 held-out reactions (about 1% of
the held-out half) the product is a reactant of the training half, and about 20 of these are the exact reverse of a
training reaction (stereoisomers count as different molecules). The models are fitted to the training half only.
**Everything that follows is evaluated on the held-out half.** The function that builds either model is short:

In [ ]:
P.show(P.fit_model)

In [ ]:
def error_summary(residuals):
    a = np.abs(residuals)
    return {"MAE": a.mean(), "RMSE": np.sqrt((a ** 2).mean()), "median": np.median(a),
            "90th pct.": np.quantile(a, 0.90), "95th pct.": np.quantile(a, 0.95),
            "99th pct.": np.quantile(a, 0.99), "largest": a.max()}

residuals = {name: (y - mod.mu)[mod.hold] for name, mod in rdb7.models.items()}
errors = pd.DataFrame({rdb7.models[name].label: error_summary(res) for name, res in residuals.items()}).T
print(f"training half: {len(rdb7.train)} reactions; held-out half: {len(rdb7.hold)} reactions; reactants in common: "
      f"{len(set(rdb7.reactant[rdb7.train]) & set(rdb7.reactant[rdb7.hold]))}")
print("Absolute errors on the held-out reactions (kcal/mol)")
display(errors.round(2))

Read the table from left to right. The MAE of the Δ-learning model is about 1.6 kcal/mol, not far from the 1 kcal/mol
that is conventionally called chemical accuracy (here relative to the reference level). But one prediction in ten is
off by more than about 3.8 kcal/mol, one in a hundred by more than about 11 kcal/mol, and the worst prediction by more
than 30 kcal/mol. **The MAE describes the typical reaction; a user needs a statement about *this* reaction.**

What such an error means for kinetics follows from $k \propto \exp(-E/RT)$, with $E$ the barrier height: an error
$\Delta E$ in the barrier changes the rate constant by the factor $\exp(\Delta E/RT)$.

In [ ]:
rows = []
for name, res in residuals.items():
    for label, value in (("MAE", np.abs(res).mean()), ("90th percentile", np.quantile(np.abs(res), 0.90))):
        rows.append({"model": rdb7.models[name].label, "error measure": label, "kcal/mol": round(value, 2),
                     "factor in k at 298 K": f"{P.rate_factor(value, 298.15):.1e}",
                     "factor in k at 1000 K": f"{P.rate_factor(value, 1000.0):.2g}"})
display(pd.DataFrame(rows))

## 4. The Gaussian error bar, and why it fails

The textbook way to attach an uncertainty is to assume that errors are normally distributed with a standard deviation
$\sigma$ equal to the root-mean-square error (RMSE) and to report $\hat\mu(x) \pm z\sigma$, where $\hat\mu(x)$ is the
prediction for reaction $x$: $z = 1$ for 68%, $z = 1.645$ for 90%, $z = 2.576$ for 99%. We can test this promise
directly, because we know the reference barriers of the held-out reactions. (We even use the held-out RMSE, which is
the most favorable choice for the Gaussian recipe.)

In [ ]:
LEVELS = np.array([0.50, 0.68, 0.80, 0.90, 0.95, 0.99])

def gaussian_coverage(res, levels=LEVELS):
    """Fraction of residuals inside +/- z * RMSE, for each nominal level."""
    rmse = np.sqrt(np.mean(res ** 2))
    return np.array([np.mean(np.abs(res) <= stats.norm.ppf(0.5 + level / 2) * rmse) for level in levels])

gauss = pd.DataFrame({rdb7.models[name].label: gaussian_coverage(res) for name, res in residuals.items()},
                     index=[f"{level:.0%}" for level in LEVELS])
gauss.index.name = "promised"
print("Observed coverage of Gaussian intervals")
display(gauss.round(3))
print("excess kurtosis of the residuals (0 for a normal distribution):",
      {rdb7.models[name].label: round(float(stats.kurtosis(res)), 1) for name, res in residuals.items()})

The promise is broken in both directions. Low and medium levels are over-covered (the "68%" interval of the Δ-learning
model contains more than 80% of the reactions, so it is wider than it needs to be), while the 99% interval misses two
to three times as often as promised. Panel c of the figure below shows this as the ratio of the observed to the promised
miss rate, on a logarithmic axis. An honest error bar would sit at 1 for every level; values above 1 mean more misses
than promised, values below 1 mean that the interval is wider than it needs to be. This is the signature of a
**heavy-tailed error distribution**: most reactions are predicted well, a minority very badly. No choice of a single
$\sigma$ can repair that. (Pernot, 2022, reviews how prediction uncertainties in computational chemistry can be
validated.)

The figure summarizes the notebook.

In [ ]:
fig, axes = P.panels(3)

for ax, letter, name in zip(axes[:2], "ab", ("structure", "delta")):
    mod = rdb7.models[name]
    ax.scatter(y[mod.hold], mod.mu[mod.hold], s=3, alpha=0.25, color=P.MODEL_COLORS[name], linewidths=0, rasterized=True)
    ax.plot([0, 200], [0, 200], color=P.GRAY, lw=0.8)
    ax.set(xlim=(0, 200), ylim=(0, 200), xticks=[0, 50, 100, 150, 200], yticks=[0, 50, 100, 150, 200],
           xlabel="reference barrier (kcal/mol)", ylabel="predicted barrier (kcal/mol)")
    ax.text(0.97, 0.05, f"MAE {errors.loc[mod.label, 'MAE']:.1f}\nRMSE {errors.loc[mod.label, 'RMSE']:.1f}\nkcal/mol",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=7)
    P.panel_label(ax, letter, mod.label)

# panel c: how often does the Gaussian interval miss, relative to what it promises?
ax = axes[2]
positions = np.arange(len(LEVELS))
for name in ("structure", "delta"):
    mod = rdb7.models[name]
    ax.plot(positions, (1 - gauss[mod.label].values) / (1 - LEVELS), marker=P.MODEL_MARKERS[name],
            color=P.MODEL_COLORS[name], label=mod.label)
ax.axhline(1, color=P.GRAY, lw=0.8, ls="--")
ax.set_yscale("log")
ax.set_yticks([0.5, 1, 2, 3], ["0.5", "1", "2", "3"])
ax.minorticks_off()
ax.set_ylim(0.4, 4.2)
ax.set_xticks(positions, [f"{100 * level:.0f}" for level in LEVELS])
ax.set(xlabel="promised coverage (%)", ylabel="observed / promised miss rate")
ax.legend(loc="upper left", borderaxespad=0.2)
ax.text(-0.1, 1.05, "1 = as promised", fontsize=6.5, va="bottom")
P.panel_label(ax, "c", "Gaussian error bars")
for ax in axes:
    ax.set_box_aspect(1)
P.savefig(fig, "nb1_models_and_gaussian_error_bars")
plt.show()

Panels **a** and **b**: predicted against reference barriers for the held-out reactions. Panel **c**: miss rate of the
Gaussian interval $\hat\mu \pm z\sigma$ relative to the promised one (logarithmic vertical axis; the six levels are
equally spaced on the horizontal axis).

## 5. Errors depend on the chemistry

A last observation prepares Notebooks 3 and 4. The size of the errors is not the same for all reaction classes:

In [ ]:
by_class = {}
for name, mod in rdb7.models.items():
    a = pd.Series(np.abs(y - mod.mu)[mod.hold])
    g = a.groupby(rdb7.rxn_class[mod.hold])
    by_class[(mod.label, "MAE")] = g.mean()
    by_class[(mod.label, "90th pct.")] = g.quantile(0.90)
by_class = pd.DataFrame(by_class).reindex(P.CLASS_ORDER)
by_class.insert(0, "held-out reactions", pd.Series(rdb7.rxn_class[rdb7.hold]).value_counts().reindex(P.CLASS_ORDER))
display(by_class.round(2))

For the Δ-learning model the 90th percentile of the absolute error differs by a factor of about three between the
class with the smallest errors (carbene products) and the one with the largest (skeletal rearrangements). A single
error bar for all reactions will be too wide for the former and too narrow for the latter.

## Notation used in all six notebooks

| symbol | meaning |
|---|---|
| $x$, $y$ | a reaction (its descriptors) and its reference barrier height in kcal/mol |
| $\hat\mu(x)$ | the prediction of a model for reaction $x$ (`mu` in the code) |
| $C(x)$ | the prediction interval for reaction $x$ |
| coverage | the fraction of cases whose reference value lies inside the interval |
| $\alpha$ | the miscoverage level: the fraction of misses we accept; $1-\alpha$ is the nominal coverage (default 90%) |
| $n$ | number of calibration reactions (Notebook 2) |
| $N$, $K$ | number of labeled reactions when no calibration set is set aside, and number of folds (Notebook 2, CV+) |
| $s_i$ | score of calibration reaction $i$: a number that says how wrong the model was (in Notebook 6: how promising a candidate looks) |
| $k$, $\hat q$ | $k = \lceil (n+1)(1-\alpha)\rceil$, and the conformal quantile $\hat q$: the $k$-th smallest calibration score (in Notebooks 1, 2, and 6, $k$ also stands for a rate constant) |
| half-width | half the length of an interval, the number after the "±" |
| $\hat\sigma(x)$ | a difficulty estimate for reaction $x$ (Notebook 3); not to be confused with the Gaussian $\sigma$ above |
| $w(x)$ | weight of a calibration reaction under covariate shift (Notebook 5) |
| $m$ | number of new cases: test reactions, candidates, or new reference calculations |
| $c$, $p_j$ | the threshold that defines a hit, and the conformal p-value of candidate $j$ (Notebook 6) |
| $q$, $r$ | the target false discovery rate of Notebook 6 (the usual symbol in that field; unrelated to $\hat q$), and the rank of a candidate in the Benjamini–Hochberg step |

## Take-aways

* A point prediction and an MAE do not tell the user how far to trust one particular number.
* The Gaussian "±σ" recipe assumes a shape for the error distribution. Here the shape is wrong (heavy tails): the
  intervals are too wide at low levels and too narrow in the tails, and right only where the two errors happen to
  cancel.
* What we want instead is an interval with a *guaranteed* coverage that does not depend on the shape of the error
  distribution or on the model. Notebook 2 constructs exactly that, in about fifteen lines of NumPy.

## Try it yourself

1. Replace the held-out RMSE in `gaussian_coverage` by the RMSE of the *training* residuals, as one would do without a
   held-out set. What happens to the observed coverages?
2. Restrict the comparison to one reaction class. Is the Gaussian recipe better within a class?

## References

* Spiekermann, K. A.; Pattanaik, L.; Green, W. H. High Accuracy Barrier Heights, Enthalpies, and Rate Coefficients for
  Chemical Reactions. *Sci. Data* **2022**, *9*, 417. https://doi.org/10.1038/s41597-022-01529-6
  (data: https://doi.org/10.5281/zenodo.6618262, CC BY 4.0)
* Grambow, C. A.; Pattanaik, L.; Green, W. H. Reactants, Products, and Transition States of Elementary Chemical
  Reactions Based on Quantum Chemistry. *Sci. Data* **2020**, *7*, 137. https://doi.org/10.1038/s41597-020-0460-4
* Spiekermann, K. A.; Pattanaik, L.; Green, W. H. Fast Predictions of Reaction Barrier Heights: Toward Coupled-Cluster
  Accuracy. *J. Phys. Chem. A* **2022**, *126*, 3976–3986. https://doi.org/10.1021/acs.jpca.2c02614
* Gao, C. W.; Allen, J. W.; Green, W. H.; West, R. H. Reaction Mechanism Generator: Automatic Construction of Chemical
  Kinetic Mechanisms. *Comput. Phys. Commun.* **2016**, *203*, 212–225. https://doi.org/10.1016/j.cpc.2016.02.013
* Ramakrishnan, R.; Dral, P. O.; Rupp, M.; von Lilienfeld, O. A. Big Data Meets Quantum Chemistry Approximations: The
  Δ-Machine Learning Approach. *J. Chem. Theory Comput.* **2015**, *11*, 2087–2096. https://doi.org/10.1021/acs.jctc.5b00099
* Rogers, D.; Hahn, M. Extended-Connectivity Fingerprints. *J. Chem. Inf. Model.* **2010**, *50*, 742–754.
  https://doi.org/10.1021/ci100050t
* Friedman, J. H. Greedy Function Approximation: A Gradient Boosting Machine. *Ann. Stat.* **2001**, *29*, 1189–1232.
  https://doi.org/10.1214/aos/1013203451
* Pedregosa, F.; et al. Scikit-learn: Machine Learning in Python. *J. Mach. Learn. Res.* **2011**, *12* (85), 2825–2830.
  https://jmlr.org/papers/v12/pedregosa11a.html
* RDKit: Open-Source Cheminformatics. https://www.rdkit.org (accessed 2026-10-11). The version used in a run is stored
  with its results.
* Pernot, P. The Long Road to Calibrated Prediction Uncertainty in Computational Chemistry. *J. Chem. Phys.* **2022**,
  *156*, 114109. https://doi.org/10.1063/5.0084302

In [ ]:
results = {"n_reactions": rdb7.n, "n_train": len(rdb7.train), "n_hold": len(rdb7.hold),
           "n_reactants": len(set(rdb7.reactant)), "shared_reactants": len(set(rdb7.reactant[rdb7.train]) & set(rdb7.reactant[rdb7.hold]))}
for name, mod in rdb7.models.items():
    results[name] = {
        "n_features": mod.X.shape[1],
        "errors": {k.replace(" pct.", "").replace("th", ""): v for k, v in errors.loc[mod.label].items()},
        "gaussian_coverage": {f"{level:.2f}": c for level, c in zip(LEVELS, gauss[mod.label].values)},
        "excess_kurtosis": float(stats.kurtosis(residuals[name])),
        "class_q90": {c: by_class.loc[c, (mod.label, "90th pct.")] for c in P.CLASS_ORDER},
        "class_mae": {c: by_class.loc[c, (mod.label, "MAE")] for c in P.CLASS_ORDER},
    }
results["class_counts"] = {c: int((rdb7.rxn_class == c).sum()) for c in P.CLASS_ORDER}
results["barrier_mean"], results["family_share"] = float(y.mean()), float(rdb7.df.rmg_family.notna().mean())
P.save_results("nb1", results, rdb7)
P.check("nb1", results);

# Notebook 2 · Split conformal prediction from scratch

**Conformal Prediction for Physical Chemists: hands-on notebooks (2 of 6)**

Notebook 1 ended with a wish: an interval around every prediction whose coverage is *guaranteed*, whatever the model
and whatever the shape of its error distribution. Split conformal prediction (also called inductive conformal
prediction; Papadopoulos et al., 2002; Lei et al., 2018) delivers this with a held-out calibration set and one sorted
array. The general theory is laid out in the monograph of Vovk, Gammerman, and Shafer (2022), and Angelopoulos and Bates
(2023) give a broader introduction for machine-learning practitioners. In chemistry the method has mostly been used in
cheminformatics (Norinder et al., 2014; Svensson et al., 2018) and, more recently, for machine-learned interatomic
potentials (Hu et al., 2022).

**You will**

1. build 90% prediction intervals for reaction barriers in about fifteen lines of NumPy;
2. understand *why* the recipe works (a statement about ranks, not about distributions);
3. learn what the guarantee does **not** promise;
4. see that the coverage obtained from one calibration set is itself a random number, and how its spread depends
   on the size of the calibration set;
5. reproduce the two most common ways of getting it wrong;
6. see what to do when there are too few reference values to set a calibration set aside (CV+).

Runs on a CPU in under a minute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

P.set_style()
rdb7 = P.load_problem()
y = rdb7.y

model = rdb7.models["delta"]        # the Δ-learning model is the running example of this notebook
mu = model.mu                       # its point predictions, for all reactions

## 1. The recipe

We have a fitted model $\hat\mu$ and a set of reactions that the model has **not** seen during fitting, with known
reference barriers. Split this held-out set into a **calibration set** of size $n$ and a **test set** that plays the
role of future predictions. Choose the *miscoverage level* $\alpha$, the fraction of misses you are willing to accept:
$\alpha = 0.10$ for 90% intervals.

1. For each calibration reaction compute a *score* that says how wrong the model was (the literature calls it a
   conformity or nonconformity score). The simplest choice is the absolute residual, $s_i = |y_i - \hat\mu(x_i)|$.
2. Sort the scores and take the $\lceil (n+1)(1-\alpha)\rceil$-th smallest one. Call it $\hat q$.
3. For any new reaction $x$ report the interval $C(x) = [\hat\mu(x) - \hat q,\ \hat\mu(x) + \hat q]$.

That is all. The claim is

$$\Pr\{\,Y \in C(X)\,\} \;\ge\; 1 - \alpha ,$$

for **any** model and **any** error distribution, provided the calibration reactions and the new reaction are
*exchangeable* (roughly: drawn from the same population, in no particular order). The model was fitted on separate
data, so it can be good or bad; the guarantee holds either way.

In [ ]:
# a calibration set of 1000 reactions; the rest of the held-out reactions act as "future" test cases
rng = np.random.default_rng(P.SEED)
shuffled = rng.permutation(model.hold)
cal, test = shuffled[:1000], shuffled[1000:]

alpha = 0.10                                         # we want 90% intervals

# 1. scores on the calibration set
scores = np.abs(y[cal] - mu[cal])

# 2. the conformal quantile: the ceil((n + 1)(1 - alpha))-th smallest score
n = len(scores)
k = int(np.ceil((n + 1) * (1 - alpha)))
q_hat = np.sort(scores)[k - 1]

# 3. intervals for the test reactions
lower, upper = mu[test] - q_hat, mu[test] + q_hat

# 4. check them against the reference values (possible here because we kept the answers)
covered = (y[test] >= lower) & (y[test] <= upper)
print(f"n = {n}, k = {k}:  q_hat = {q_hat:.2f} kcal/mol")
print(f"every prediction becomes  mu ± {q_hat:.2f} kcal/mol;  {covered.mean():.1%} of {len(test)} test barriers lie inside")

The same steps, written as two functions. They are also stored in `primer.py`, so that the later notebooks can simply
import them.

In [ ]:
def conformal_quantile(scores, alpha=0.10):
    """The ceil((n+1)(1-alpha))-th smallest calibration score; infinity if n is too small."""
    scores = np.asarray(scores, dtype=float)
    n = scores.size
    k = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))     # the tiny offset guards against floating-point round-up
    if k > n:
        return np.inf
    return float(np.partition(scores, k - 1)[k - 1])


def split_conformal(y_cal, mu_cal, mu_test, alpha=0.10):
    """Absolute-residual split conformal interval: mu ± q."""
    q = conformal_quantile(np.abs(y_cal - mu_cal), alpha)
    return mu_test - q, mu_test + q, q


assert conformal_quantile(scores, alpha) == q_hat == P.conformal_quantile(scores, alpha)

## 2. Why it works

Nothing in the recipe refers to a normal distribution, to the quality of the model, or to chemistry. The argument is
about **ranks**. Take the $n$ calibration scores and the score of one new reaction. Because $\hat q$ is the $k$-th
smallest calibration score, the new score is at most $\hat q$ exactly when its rank among all $n+1$ scores is at most
$k$. If the $n+1$ scores are exchangeable and no two are equal, the new score is equally likely to be the smallest,
the second smallest, ..., or the largest of them: each rank has probability $1/(n+1)$. The new score is therefore at
most $\hat q$ with probability $k/(n+1)$, and "the new score is at most $\hat q$" is the same event as "the reference
barrier lies in the interval". Choosing $k = \lceil (n+1)(1-\alpha) \rceil$ makes that probability at least $1-\alpha$.
(If some scores are equal, the probability can only be larger.)

The cell below checks the rank statement with numbers that have nothing to do with chemistry: squared Cauchy variates,
a distribution so heavy-tailed that it has no mean.

In [ ]:
rng = np.random.default_rng(1)
n_demo = 19
draws = rng.standard_cauchy(size=(200_000, n_demo + 1)) ** 2            # 19 "calibration" values + 1 "new" value per row
rank_of_new = 1 + (draws[:, :-1] < draws[:, -1:]).sum(axis=1)           # rank of the new value among all 20
frequencies = np.bincount(rank_of_new, minlength=n_demo + 2)[1:] / len(draws)
print("frequency of each rank (should all be 1/20 = 0.05):")
print(np.round(frequencies, 3))
print(f"new value <= largest of the 19 others: {np.mean(rank_of_new <= n_demo):.4f}   (theory 19/20 = 0.95)")

With $n = 19$ and $\alpha = 0.05$ the recipe gives $k = 19$: the interval half-width is simply the largest of 19
calibration errors, and it covers 95% of new cases. This also shows the smallest calibration set that can support a
given level: $k \le n$ requires $n \ge (1-\alpha)/\alpha$, i.e., 9 reactions for 90%, 19 for 95%, and 99 for 99%
intervals. With fewer, the only interval with a guarantee is the infinite one.

## 3. The result

In [ ]:
LEVELS = np.array([0.50, 0.68, 0.80, 0.90, 0.95, 0.99])
abs_res = np.abs(y - mu)

# conformal coverage at several levels, averaged over 1000 random calibration/test splits
n_splits = 1000
cov_splits = np.zeros((n_splits, len(LEVELS)))
for i, (cal_b, test_b) in enumerate(P.resplits(model.hold, 1000, n_splits, seed=1)):
    for j, level in enumerate(LEVELS):
        cov_splits[i, j] = np.mean(abs_res[test_b] <= conformal_quantile(abs_res[cal_b], 1 - level))
conformal_cov = cov_splits.mean(axis=0)
conformal_se = cov_splits.std(axis=0, ddof=1) / np.sqrt(n_splits)       # Monte Carlo error of these averages

# Gaussian intervals from Notebook 1 for comparison
res_hold = (y - mu)[model.hold]
rmse = np.sqrt(np.mean(res_hold ** 2))
gauss_cov = np.array([np.mean(np.abs(res_hold) <= stats.norm.ppf(0.5 + level / 2) * rmse) for level in LEVELS])

comparison = pd.DataFrame({"conformal": conformal_cov, "Gaussian ± zσ": gauss_cov}, index=[f"{l:.0%}" for l in LEVELS])
comparison.index.name = "promised"
display(comparison.round(3))
print(f"The conformal column averages {n_splits} random splits; the standard error of these averages is "
      f"{conformal_se.min():.4f} to {conformal_se.max():.4f}.")

In [ ]:
fig, axes = P.panels(3)

# a: calibration scores and the conformal quantile
ax = axes[0]
x_max = 12.0 if q_hat < 8 else 4.0 * np.ceil(1.5 * q_hat / 4)        # 12 kcal/mol unless the quantile lies farther out
ax.hist(np.clip(scores, 0, x_max - 0.01), bins=np.linspace(0, x_max, 25), color=P.BLUE, rwidth=0.82)
ax.axvline(q_hat, color=P.VERMILLION, lw=1.2, ls="--")
ax.text(q_hat + 0.025 * x_max, 0.92, f"$\\hat{{q}}$ = {q_hat:.2f}", transform=ax.get_xaxis_transform(), fontsize=7, va="top")
ax.text(q_hat + 0.025 * x_max, 0.80, f"{1 - alpha:.0%} of the scores\nlie to the left", transform=ax.get_xaxis_transform(),
        fontsize=6.5, va="top")
ax.set(xlabel=r"calibration score $|y - \hat{\mu}|$ (kcal/mol)", ylabel="calibration reactions", xlim=(0, x_max))
ticks = np.linspace(0, x_max, 5)
ax.set_xticks(ticks, [f"{t:g}" for t in ticks[:-1]] + [f"≥{x_max:g}"])
P.panel_label(ax, "a", f"{len(scores)} calibration scores")

# b: the interval as a band of constant half-width around the prediction
ax = axes[1]
res_test = y[test] - mu[test]
inside = np.abs(res_test) <= q_hat
ax.axhspan(-q_hat, q_hat, color=P.SKY, alpha=0.30, lw=0)
ax.scatter(mu[test][inside], res_test[inside], s=3, alpha=0.25, color=P.BLUE, linewidths=0, rasterized=True)
ax.scatter(mu[test][~inside], np.clip(res_test[~inside], -19.5, 19.5), s=7, alpha=0.8, color=P.VERMILLION,
           marker="^", linewidths=0, rasterized=True)
ax.text(0.97, 0.97, f"inside (dots): {inside.mean():.1%}\noutside (triangles): {(~inside).mean():.1%}",
        transform=ax.transAxes, ha="right", va="top", fontsize=6.5)
ax.set(xlabel=r"predicted barrier $\hat{\mu}$ (kcal/mol)", ylabel="reference − predicted (kcal/mol)", ylim=(-20, 20), xlim=(0, 200))
ax.set_xticks([0, 50, 100, 150, 200])
P.panel_label(ax, "b", f"$\\hat{{\\mu}}$ ± {q_hat:.1f} kcal/mol")

# c: observed / promised miss rate at every level
ax = axes[2]
positions = np.arange(len(LEVELS))
ax.plot(positions, (1 - gauss_cov) / (1 - LEVELS), marker="D", color=P.ORANGE, label="Gaussian ± zσ")
ax.plot(positions, (1 - conformal_cov) / (1 - LEVELS), marker="o", color=P.BLUE, label="split conformal")
ax.axhline(1, color=P.GRAY, lw=0.8, ls="--")
ax.set_yscale("log")
ax.set_yticks([0.5, 1, 2, 3], ["0.5", "1", "2", "3"])
ax.minorticks_off()
ax.set_ylim(0.4, 4.2)
ax.set_xticks(positions, [f"{100 * level:.0f}" for level in LEVELS])
ax.set(xlabel="promised coverage (%)", ylabel="observed / promised miss rate")
ax.legend(loc="upper left", borderaxespad=0.2)
P.panel_label(ax, "c", "Promise kept")
for ax in axes:
    ax.set_box_aspect(1)
P.savefig(fig, "nb2_split_conformal")
plt.show()

Panel **a** shows the 1000 calibration scores and $\hat q$. Panel **b** shows the resulting band of constant half-width
around the predictions, with the test reactions that fall outside (errors beyond ±20 kcal/mol are drawn at the edge).
Panel **c** repeats the test of Notebook 1 for all levels, on the same logarithmic axis. The conformal intervals miss
as often as promised, from 50% to 99%, without any assumption about the error distribution. (In the table, the
conformal coverages differ from the promised levels by less than 0.001. Two small effects add up: the rounding in $k$
puts the expected coverage up to $1/(n+1) \approx 0.001$ above $1-\alpha$, and the average over 1000 random splits has
a standard error of a few ten-thousandths, printed below the table.)

## 4. What the guarantee does *not* say

The statement $\Pr\{Y \in C(X)\} \ge 1-\alpha$ is easy to overinterpret. Four limits matter in practice.

* **It is marginal, not conditional.** The probability is an average over all reactions of the population. It does
  not say that 90% of the *skeletal rearrangements* are covered, or 90% of the reactions with a predicted barrier
  below 40 kcal/mol. Notebook 1 showed that the errors of this model depend on the reaction class, so a single
  $\hat q$ will over-cover some classes and under-cover others (Notebooks 3 and 4).
* **It is an average over calibration sets.** A particular calibration set gives a particular $\hat q$, and the
  coverage *of that* $\hat q$ can be above or below $1-\alpha$. Section 5 quantifies this.
* **It needs exchangeability.** If tomorrow's reactions differ systematically from the calibration reactions, the
  guarantee is void (Notebook 5).
* **It refers to the reference level.** The interval covers the CCSD(T)-F12a value with probability $1-\alpha$. The
  error of the reference method itself is outside the guarantee.

## 5. Coverage is a random variable

Suppose a colleague repeats our work with a different random calibration set. Will they find the same coverage? No:
they will get a slightly different $\hat q$ and a different test coverage. For split conformal prediction the
distribution of this coverage is known exactly (Vovk, 2013). For independent draws from one population (and scores
without ties), the coverage that a calibration set of size $n$ achieves on an infinitely large test set follows a Beta
distribution,

$$\text{coverage} \sim \mathrm{Beta}(k,\; n + 1 - k), \qquad k = \lceil (n+1)(1-\alpha) \rceil ,$$

which again does not depend on the model or the data. Its mean is $k/(n+1)$, just above $1-\alpha$. On a finite test
set of $m$ reactions the number of covered reactions follows the corresponding Beta-binomial distribution (a binomial
distribution whose success probability is itself Beta-distributed). The next cell repeats the calibration 2000 times
for several $n$ and compares with this law.

In [ ]:
def coverage_experiment(score_pool, n_cal, alpha=0.10, n_rep=2000, seed=0):
    """Test coverage of split conformal intervals for `n_rep` random calibration sets of size `n_cal`."""
    rng = np.random.default_rng(seed)
    out = np.empty(n_rep)
    for b in range(n_rep):
        perm = rng.permutation(score_pool)
        out[b] = np.mean(perm[n_cal:] <= conformal_quantile(perm[:n_cal], alpha))
    return out


def finite_test_law(n_cal, n_test, alpha=0.10):
    """Exact distribution of the number of covered test points (Beta-binomial)."""
    k = int(np.ceil((n_cal + 1) * (1 - alpha) - 1e-9))
    return stats.betabinom(n_test, k, n_cal + 1 - k)


pool = abs_res[model.hold]
sizes = [20, 50, 100, 200, 500, 1000, 2000]
experiments = {n_cal: coverage_experiment(pool, n_cal) for n_cal in sizes}

rows = []
for n_cal, cov in experiments.items():
    n_test = len(pool) - n_cal
    law = finite_test_law(n_cal, n_test)
    rows.append({"n": n_cal, "mean": cov.mean(), "2.5%": np.quantile(cov, 0.025), "97.5%": np.quantile(cov, 0.975),
                 "theory mean": law.mean() / n_test, "theory 2.5%": law.ppf(0.025) / n_test, "theory 97.5%": law.ppf(0.975) / n_test})
coverage_table = pd.DataFrame(rows).set_index("n")
print("Test coverage of nominal 90% intervals over 2000 random calibration sets")
display(coverage_table.round(3))

The agreement is as close as 2000 repetitions allow, and the message is practical: **with 50 calibration reactions, a
"90%" interval has an actual coverage between about 81% and 97% for 95% of all calibration sets; with 1000 it is pinned
down to about ±2 percentage points.** A coverage of 87% obtained with a small calibration set is therefore not evidence
that conformal prediction has failed, and 93% is not evidence that it works unusually well. (A small *test* set adds
ordinary binomial noise on top.)

The same law answers a planning question: how many reference calculations should be set aside for calibration? (The
law assumes calibration reactions that are unrelated to one another. Notebook 5 shows that reactions which come in
clusters, such as several reactions of one reactant, are worth fewer.)

In [ ]:
def calibration_size_needed(tolerance, alpha=0.10, confidence=0.95, n_max=20000):
    """Smallest n such that, for this and every larger calibration set (up to n_max), the coverage lies
    within ±tolerance of 1 - alpha with the given probability."""
    sizes = np.arange(10, n_max + 1)
    k = np.ceil((sizes + 1) * (1 - alpha) - 1e-9)                    # the rank k for every n at once
    finite = k <= sizes                                              # otherwise the interval is infinite
    inside = np.zeros(len(sizes))                                    # probability that the coverage is within the tolerance
    inside[finite] = (stats.beta.cdf(1 - alpha + tolerance, k[finite], (sizes + 1 - k)[finite])
                      - stats.beta.cdf(1 - alpha - tolerance, k[finite], (sizes + 1 - k)[finite]))
    enough = inside >= confidence
    too_small = np.flatnonzero(~enough)      # the rounding in k makes the probability wiggle, so take the last failure
    if too_small.size and too_small[-1] == len(sizes) - 1:
        raise ValueError(f"the tolerance is not reached with calibration sets of up to n_max = {n_max} reactions")
    return int(sizes[too_small[-1] + 1]) if too_small.size else int(sizes[0])

needed = pd.DataFrame({"calibration reactions needed": {f"±{100 * t:.0f}": calibration_size_needed(t) for t in (0.05, 0.03, 0.02, 0.01)}})
needed.index.name = "coverage within (percentage points, 95% of the time)"
display(needed)
print("smallest calibration set with a finite interval:",
      {f"{level:.0%}": int(np.ceil(level / (1 - level) - 1e-9)) for level in (0.90, 0.95, 0.99)})

In [ ]:
fig, axes = P.panels(2, height=2.6, gridspec_kw={"width_ratios": [1, 1.15]})

# a: two calibration-set sizes, experiment versus theory
ax = axes[0]
for n_cal, color, label_x in ((50, P.ORANGE, 0.80), (1000, P.BLUE, 0.928)):
    cov = experiments[n_cal]
    n_test = len(pool) - n_cal
    ax.hist(cov, bins=np.linspace(0.74, 1.0, 53), density=True, color=color, alpha=0.35, lw=0)
    law = finite_test_law(n_cal, n_test)
    grid = np.arange(int(0.74 * n_test), n_test + 1)
    ax.plot(grid / n_test, law.pmf(grid) * n_test, color=color, lw=1.4)
    ax.text(label_x, 0.62 if n_cal == 1000 else 0.2, f"$n$ = {n_cal}", transform=ax.get_xaxis_transform(), fontsize=7, ha="left")
ax.axvline(0.90, color=P.GRAY, lw=0.8, ls="--")
ax.set(xlabel="coverage on the test set", ylabel="probability density", xlim=(0.74, 1.0))
P.panel_label(ax, "a", "2000 calibration sets (bars) and theory (lines)")

# b: the 95% range of the coverage as a function of n
ax = axes[1]
grid_n = np.unique(np.round(np.logspace(1, np.log10(5000), 200)).astype(int))
laws = [P.coverage_law(n_cal) for n_cal in grid_n]
ax.fill_between(grid_n, [l.ppf(0.025) for l in laws], [l.ppf(0.975) for l in laws], color=P.SKY, alpha=0.35, lw=0,
                label="95% range (Beta law)")
ax.plot(grid_n, [l.mean() for l in laws], color=P.BLUE, lw=1.2, label="mean (Beta law)")
ax.plot(sizes, coverage_table["2.5%"], ls="none", marker="v", color=P.VERMILLION, label="2.5% and 97.5% (experiment)")
ax.plot(sizes, coverage_table["97.5%"], ls="none", marker="^", color=P.VERMILLION)
ax.axhline(0.90, color=P.GRAY, lw=0.8, ls="--")
ax.set_xscale("log")
ax.set(xlabel="calibration reactions $n$", ylabel="coverage of a nominal 90% interval", xlim=(10, 5000), ylim=(0.68, 1.0))
ax.set_xticks([10, 100, 1000], ["10", "100", "1000"])
ax.legend(loc="lower right")
P.panel_label(ax, "b", "How much can one calibration set be off?")
P.savefig(fig, "nb2_coverage_distribution")
plt.show()

Panel **a**: the coverage found in 2000 repetitions (bars) and the exact law (lines) for two calibration-set sizes.
Panel **b**: the range that contains the coverage of 95% of all calibration sets, as a function of $n$. The band is the
Beta law, which describes an infinitely large test set; its small teeth come from the rounding in $k$. The triangles
are the experiment, whose finite test sets add a little scatter of their own (visible at $n = 2000$). Note that the
vertical axis starts at 0.68.

## 6. Two ways of getting it wrong

**Pitfall 1: calibrating on the training set.** It is tempting to save reference calculations by computing the scores
on the reactions the model was fitted to. But a flexible model has partly memorized its training set, so those
residuals are too small, and so is $\hat q$.

In [ ]:
q_from_training = conformal_quantile(abs_res[model.train], alpha)
q_from_holdout = conformal_quantile(abs_res[cal], alpha)
pitfall_training = np.mean(abs_res[test] <= q_from_training)
print(f"q from {len(model.train)} TRAINING residuals : {q_from_training:.2f} kcal/mol -> test coverage {pitfall_training:.1%}")
print(f"q from {len(cal)} held-out residuals : {q_from_holdout:.2f} kcal/mol -> test coverage {np.mean(abs_res[test] <= q_from_holdout):.1%}")

The "90%" interval now covers only about three reactions in four. The same happens, less visibly, whenever the
calibration set is not independent of the fitted model: when it was used for early stopping or for choosing
hyperparameters, or when it contains near-duplicates of training points (conformers of a training molecule, the reverse
of a training reaction).

**Pitfall 2: forgetting the finite-sample correction.** The conformal quantile is the $\lceil (n+1)(1-\alpha)\rceil$-th
smallest score, not the empirical 90th percentile. For large $n$ the difference is negligible; for small $n$ it is
the difference between a guarantee and a shortfall.

In [ ]:
def coverage_with(quantile_rule, n_cal, n_rep=4000, seed=3):
    rng = np.random.default_rng(seed)
    out = np.empty(n_rep)
    for b in range(n_rep):
        perm = rng.permutation(pool)
        out[b] = np.mean(perm[n_cal:] <= quantile_rule(perm[:n_cal]))
    return out.mean()

rows = []
for n_cal in (20, 50, 200, 1000):
    rows.append({"n": n_cal,
                 "np.quantile(scores, 0.90)": coverage_with(lambda s: np.quantile(s, 1 - alpha), n_cal),
                 "conformal quantile": coverage_with(lambda s: conformal_quantile(s, alpha), n_cal)})
pitfall_quantile = pd.DataFrame(rows).set_index("n")
print("Mean test coverage of nominal 90% intervals")
display(pitfall_quantile.round(3))

## 7. Validity does not depend on the model; sharpness does

An interval method is called *valid* if it keeps its coverage promise, and *sharp* (or efficient) if its intervals are
narrow. The recipe did not ask how good the predictor is, or even what it is. To see what that means, apply it to five
predictors of very different quality: a constant (the mean barrier of the training set, i.e., no model at all), the
structure-only model, the ωB97X-D3 barrier taken as it is (a quantum-chemical method, no machine learning), the same
barrier after subtracting its mean error on the training set, and the Δ-learning model.

In [ ]:
barrier_dft = rdb7.df["barrier_dft"].to_numpy()
dft_offset = (barrier_dft - y)[rdb7.train].mean()            # mean error of the DFT barriers, from the training half
predictors = {"constant (training mean)": np.full(rdb7.n, y[rdb7.train].mean()),
              rdb7.models["structure"].label: rdb7.models["structure"].mu,
              "ωB97X-D3 barrier, uncorrected": barrier_dft,
              "ωB97X-D3 barrier, mean error removed": barrier_dft - dft_offset,
              rdb7.models["delta"].label: rdb7.models["delta"].mu}
rows = []
for label, prediction in predictors.items():
    a = np.abs(y - prediction)
    qs, covs = [], []
    for cal_b, test_b in P.resplits(rdb7.hold, 1000, 500, seed=2):
        q = conformal_quantile(a[cal_b], alpha)
        qs.append(q)
        covs.append(np.mean(a[test_b] <= q))
    rows.append({"predictor": label, "MAE": a[rdb7.hold].mean(), "half-width (kcal/mol)": np.mean(qs), "coverage": np.mean(covs),
                 "factor in k at 1000 K": float(P.rate_factor(np.mean(qs), 1000.0))})
sharpness = pd.DataFrame(rows).set_index("predictor")
shown = sharpness.round({"MAE": 2, "half-width (kcal/mol)": 1, "coverage": 3})
shown["factor in k at 1000 K"] = [f"{value:.3g}" for value in sharpness["factor in k at 1000 K"]]
display(shown)
print(f"ωB97X-D3 minus CCSD(T)-F12a barrier on the training half: mean {dft_offset:+.2f} kcal/mol, "
      f"positive for {np.mean((barrier_dft - y)[rdb7.train] > 0):.0%} of the reactions")

All five rows have 90% coverage. Even "predict 80 kcal/mol for every reaction" comes with a perfectly valid interval;
it is just uselessly wide. The third row needs no machine learning: for reactions like those in RDB7, an ωB97X-D3
barrier lies within the stated half-width of the CCSD(T)-F12a barrier in 90% of the cases. The fourth row shows that a
symmetric interval around a biased predictor is wasteful: the ωB97X-D3 barrier lies above the CCSD(T)-F12a barrier for
almost nine reactions in ten, and removing the mean difference, the simplest possible correction, already shrinks the
interval by a third. **Conformal prediction does not make a predictor better. It reports how large the errors of the
given predictor are**, in the unit you care about: here, how many kcal/mol (or how many orders of magnitude in the rate
constant) you should allow for.

## 8. When reference data are scarce: CV+

Split conformal prediction spends part of the reference data on calibration. With thousands of reactions that is
painless. With a hundred it hurts twice: the model is fitted on fewer reactions, and the small calibration set gives
an unstable $\hat q$ (Section 5). Calibrating on the training residuals instead is not a way out (Pitfall 1).

**CV+** (Barber, Candès, Ramdas, and Tibshirani, 2021) uses every labeled reaction for both purposes, by
cross-validation. Let $N$ be the number of labeled reactions.

1. Split the $N$ reactions into $K$ folds and fit $K$ models, each *without* one fold.
2. For every labeled reaction $i$, record the residual $R_i$ of the model that did not see it (an *out-of-fold*
   residual).
3. For a new reaction $x$, every labeled reaction "votes" for an interval: with $\hat\mu_{-i}$ the model fitted
   without the fold of reaction $i$, the votes are $\hat\mu_{-i}(x) - R_i$ for the lower end and
   $\hat\mu_{-i}(x) + R_i$ for the upper end.
4. The interval runs from the $\lfloor (N+1)\alpha\rfloor$-th smallest lower vote to the
   $\lceil (N+1)(1-\alpha)\rceil$-th smallest upper vote.

With $K = N$ (leave-one-out) this is the **jackknife+**. The price of using the data twice is a weaker worst-case
guarantee. For the jackknife+ the coverage is at least $1-2\alpha$ instead of $1-\alpha$; for CV+ with $K$ folds of
equal size it is at least

$$1 - 2\alpha - \min\left\{\frac{2\,(1-1/K)}{N/K+1},\ \frac{1-K/N}{K+1}\right\} ,$$

a little lower still (Barber et al., 2021, Theorems 1 and 4; the fitting procedure must not depend on the order of the
data). These bounds are worst cases, approached only when the fitted model changes violently from one fold to the
next. In practice the coverage is close to $1-\alpha$, as the experiment below shows.

In [ ]:
def cv_plus(y, oof_pred, fold, test_pred, alpha=0.10):
    """CV+ interval. `test_pred[k]` holds the test predictions of the model fitted without fold k."""
    n = len(y)
    residual = np.abs(y - oof_pred)                     # R_i: out-of-fold residuals
    k_hi = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))
    k_lo = int(np.floor((n + 1) * alpha + 1e-9))
    if k_hi > n or k_lo < 1:                            # too few labeled points for this level
        return np.full(test_pred.shape[1], -np.inf), np.full(test_pred.shape[1], np.inf)
    low = test_pred[fold] - residual[:, None]           # one row of "votes" per labeled reaction
    high = test_pred[fold] + residual[:, None]
    return np.partition(low, k_lo - 1, axis=0)[k_lo - 1], np.partition(high, k_hi - 1, axis=0)[k_hi - 1]


# the same function is stored in primer.py; check on arbitrary numbers that the two agree
_rng = np.random.default_rng(0)
_args = (_rng.normal(size=40), _rng.normal(size=40), _rng.integers(0, 4, size=40), _rng.normal(size=(4, 7)))
assert all(np.array_equal(a, b) for a, b in zip(cv_plus(*_args), P.cv_plus(*_args)))

For the experiment we pretend that only $N$ reference barriers exist. In every repetition, $N$ labeled reactions and
2000 test reactions are drawn at random, without overlap, from the held-out half of the data, so the labeled and the
test reactions are exchangeable by construction. Boosted trees are a poor choice for a few dozen data points, so this
section uses a model that suits the situation: ridge regression for the Δ-learning correction. (Conformal prediction
does not care; that is the point of "any model".) Three ways of using the $N$ reactions are compared:

* **split 50/50**: fit on one half, calibrate on the other (Section 1);
* **training residuals**: fit on all $N$ and calibrate on the same $N$ (Pitfall 1);
* **CV+** with $K = 10$ (and the jackknife+ for the smallest $N$).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, target, baseline = model.X.astype(float), model.target, model.baseline      # features and target of the Δ-learning model


def small_model():
    """Ridge regression on standardized features. (The argument `alpha` of Ridge is the strength of its penalty and
    has nothing to do with our miscoverage level; it would normally be chosen by cross-validation.)"""
    return make_pipeline(StandardScaler(), Ridge(alpha=30.0))


def predict_barrier(fitted, rows):
    return fitted.predict(X[rows]) + baseline[rows]


def cv_plus_intervals(labeled, test_rows, n_folds=10, seed=0):
    """Fit the fold models and return the CV+ interval for every test reaction."""
    fold, oof = np.empty(len(labeled), dtype=int), np.empty(len(labeled))
    test_pred = np.empty((n_folds, len(test_rows)))
    for k_fold, (fit_idx, out_idx) in enumerate(KFold(n_folds, shuffle=True, random_state=seed).split(labeled)):
        fold_model = small_model().fit(X[labeled[fit_idx]], target[labeled[fit_idx]])
        oof[out_idx] = predict_barrier(fold_model, labeled[out_idx])
        fold[out_idx] = k_fold
        test_pred[k_fold] = predict_barrier(fold_model, test_rows)
    return cv_plus(y[labeled], oof, fold, test_pred, alpha)


rows = []
for N in (50, 100, 300, 1000):
    for rep in range(100 if N <= 300 else 40):
        shuffled_pool = np.random.default_rng(1000 * N + rep).permutation(rdb7.hold)
        labeled, test_small = shuffled_pool[:N], shuffled_pool[N:N + 2000]      # N labeled reactions, 2000 others to predict
        intervals = {}
        # split 50/50
        fit_half, cal_half = labeled[: N // 2], labeled[N // 2:]
        half_model = small_model().fit(X[fit_half], target[fit_half])
        lo, hi, _ = split_conformal(y[cal_half], predict_barrier(half_model, cal_half), predict_barrier(half_model, test_small), alpha)
        intervals["split 50/50"] = (lo, hi)
        # training residuals (wrong)
        full_model = small_model().fit(X[labeled], target[labeled])
        lo, hi, _ = split_conformal(y[labeled], predict_barrier(full_model, labeled), predict_barrier(full_model, test_small), alpha)
        intervals["training residuals (wrong)"] = (lo, hi)
        # CV+ and, for the smallest N, the jackknife+
        intervals["CV+ (K = 10)"] = cv_plus_intervals(labeled, test_small, 10, seed=rep)
        if N == 50 and rep < 40:
            intervals["jackknife+ (K = N)"] = cv_plus_intervals(labeled, test_small, N, seed=rep)
        for method, (lo, hi) in intervals.items():
            rows.append({"N": N, "method": method, "coverage": np.mean((y[test_small] >= lo) & (y[test_small] <= hi)),
                         "half_width": np.mean(hi - lo) / 2})
small_data = pd.DataFrame(rows).groupby(["N", "method"], sort=False).agg(
    coverage=("coverage", "mean"), coverage_sd=("coverage", "std"), half_width=("half_width", "mean"), half_width_sd=("half_width", "std"))
print("Mean and standard deviation over the repetitions (different labeled sets); half-widths in kcal/mol")
display(small_data.rename(columns={"coverage": "mean coverage", "coverage_sd": "sd of coverage",
                                   "half_width": "mean half-width", "half_width_sd": "sd of half-width"}).round(3))

Three things to read off. Calibrating on the training residuals under-covers, and the more so the fewer reactions
there are. Split conformal prediction and CV+ both keep the promise on average. (The split intervals lie above 90% for
the smallest $N$ because of the rounding in $k$: with 25 calibration reactions, $k = 24$ and the expected coverage is
$24/26 \approx 92\%$.) But with 50 or 100 reactions CV+ gives **narrower intervals that vary much less from one data
set to the next**: its model is fitted on 90% of the data instead of 50%, and all $N$ residuals are used instead of
$N/2$. From about a thousand reactions on, the two are practically equivalent and the simpler split recipe is the
natural choice. The cost of CV+ is $K$ model fits, and all $K$ models have to be kept for prediction.

Note also what the half-widths say about the value of data: with 100 reference barriers the ridge model gives 90%
intervals of about ±6 kcal/mol, with 1000 about ±5, and the boosted trees fitted to 6000 reactions in Section 1
reached ±3.8.

## Take-aways

* Split conformal prediction = held-out residuals + one order statistic (the $k$-th smallest score). No distributional
  assumption, no retraining, any predictor.
* The guarantee is marginal, holds on average over calibration sets, requires exchangeability, and refers to the
  reference level.
* The coverage obtained from one calibration set follows a known Beta law. A few hundred calibration reactions pin it
  down to a few percentage points; fewer than $(1-\alpha)/\alpha$ cannot support the level at all.
* Never calibrate on data the model has seen, and use the $(n+1)$ rule.
* The width of the interval is the honest price of the predictor's errors.
* With little data, use CV+ or the jackknife+ instead of setting a calibration set aside.

## Try it yourself

1. Change `alpha` to 0.01. How large is $\hat q$ for the Δ-learning model, and how does the Beta law for $n = 1000$
   change (compare `P.coverage_law(1000, 0.01)` with `P.coverage_law(1000, 0.10)`)?
2. Repeat Section 5 with the structure-only model. Does the coverage distribution change? Why not?
3. Call `conformal_quantile` with only 9 calibration scores at 90%: what is $k$, and what does the interval look like?
   What happens with 8?
4. In Section 8, replace the ridge model by `P.make_model(max_iter=100, min_samples_leaf=5)` for $N = 100$. What happens
   to the coverage of the "training residuals" intervals, and why?

## References

* Vovk, V.; Gammerman, A.; Shafer, G. *Algorithmic Learning in a Random World*, 2nd ed.; Springer: Cham, 2022.
  https://doi.org/10.1007/978-3-031-06649-8
* Papadopoulos, H.; Proedrou, K.; Vovk, V.; Gammerman, A. Inductive Confidence Machines for Regression. In *Machine
  Learning: ECML 2002*; Springer: Berlin, Heidelberg, 2002; pp 345–356. https://doi.org/10.1007/3-540-36755-1_29
* Lei, J.; G'Sell, M.; Rinaldo, A.; Tibshirani, R. J.; Wasserman, L. Distribution-Free Predictive Inference for
  Regression. *J. Am. Stat. Assoc.* **2018**, *113*, 1094–1111. https://doi.org/10.1080/01621459.2017.1307116
* Vovk, V. Conditional Validity of Inductive Conformal Predictors. *Mach. Learn.* **2013**, *92*, 349–376.
  https://doi.org/10.1007/s10994-013-5355-6 (the distribution of the coverage obtained from one calibration set)
* Angelopoulos, A. N.; Bates, S. Conformal Prediction: A Gentle Introduction. *Found. Trends Mach. Learn.* **2023**,
  *16*, 494–591. https://doi.org/10.1561/2200000101
* Barber, R. F.; Candès, E. J.; Ramdas, A.; Tibshirani, R. J. Predictive Inference with the Jackknife+. *Ann. Stat.*
  **2021**, *49*, 486–507. https://doi.org/10.1214/20-AOS1965
* Norinder, U.; Carlsson, L.; Boyer, S.; Eklund, M. Introducing Conformal Prediction in Predictive Modeling. A
  Transparent and Flexible Alternative to Applicability Domain Determination. *J. Chem. Inf. Model.* **2014**, *54*,
  1596–1603. https://doi.org/10.1021/ci5001168
* Svensson, F.; Aniceto, N.; Norinder, U.; Cortés-Ciriano, I.; Spjuth, O.; Carlsson, L.; Bender, A. Conformal Regression
  for Quantitative Structure–Activity Relationship Modeling—Quantifying Prediction Uncertainty. *J. Chem. Inf. Model.*
  **2018**, *58*, 1132–1140. https://doi.org/10.1021/acs.jcim.8b00054
* Hu, Y.; Musielewicz, J.; Ulissi, Z. W.; Medford, A. J. Robust and Scalable Uncertainty Estimation with Conformal
  Prediction for Machine-Learned Interatomic Potentials. *Mach. Learn.: Sci. Technol.* **2022**, *3*, 045028.
  https://doi.org/10.1088/2632-2153/aca7b1

In [ ]:
results = {
    "single_split": {"n_cal": int(n), "k": int(k), "q_hat": q_hat, "coverage": float(covered.mean()), "n_test": len(test)},
    "levels": {f"{level:.2f}": {"conformal": c, "conformal_se": se, "gaussian": g}
               for level, c, se, g in zip(LEVELS, conformal_cov, conformal_se, gauss_cov)},
    "coverage_vs_n": {str(n_cal): {"mean": r["mean"], "lo": r["2.5%"], "hi": r["97.5%"], "theory_mean": r["theory mean"],
                                   "theory_lo": r["theory 2.5%"], "theory_hi": r["theory 97.5%"]}
                      for n_cal, r in coverage_table.iterrows()},
    "n_needed": {k_.replace("±", "pm"): int(v) for k_, v in needed.iloc[:, 0].items()},
    "pitfall_training": {"q": q_from_training, "coverage": float(pitfall_training)},
    "pitfall_quantile": {str(n_cal): {"plain": r.iloc[0], "conformal": r.iloc[1]} for n_cal, r in pitfall_quantile.iterrows()},
    "sharpness": {name: {"mae": r["MAE"], "q": r["half-width (kcal/mol)"], "coverage": r["coverage"]}
                  for name, r in zip(("constant", "structure", "dft", "dft_shifted", "delta"), sharpness.to_dict("records"))},
    "dft_offset": {"mean": float(dft_offset), "share_positive": float(np.mean((barrier_dft - y)[rdb7.train] > 0))},
    "small_data": {f"N={N} | {method}": dict(r) for (N, method), r in small_data.iterrows()},
}
P.save_results("nb2", results, rdb7)
P.check("nb2", results);

# Notebook 3 · Adaptive intervals: one width does not fit all

**Conformal Prediction for Physical Chemists: hands-on notebooks (3 of 6)**

The interval of Notebook 2 has the same half-width for every reaction. Its coverage is right *on average*, but
Notebook 1 showed that the errors of our models depend on the chemistry. A constant width is then too generous for easy
reactions and too tight for hard ones. This notebook builds intervals whose width follows the difficulty of the
individual prediction, and shows how to check whether the adaptation worked.

**You will**

1. see the problem: the right coverage overall, the wrong coverage for subgroups;
2. build a difficulty estimate $\hat\sigma(x)$ and *normalized* conformal intervals $\hat\mu(x) \pm \hat q\,\hat\sigma(x)$;
3. build *conformalized quantile regression* (CQR) intervals;
4. compare the three kinds of interval with diagnostics for conditional coverage;
5. learn what adaptive intervals can and cannot promise.

Runs on a CPU in under a minute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

P.set_style()
rdb7 = P.load_problem()
y = rdb7.y
alpha = 0.10

model = rdb7.models["delta"]        # the Δ-learning model, as in Notebook 2
mu = model.mu

# the same calibration/test split as in Notebook 2
shuffled = np.random.default_rng(P.SEED).permutation(model.hold)
cal, test = shuffled[:1000], shuffled[1000:]

conformal_quantile = P.conformal_quantile     # derived in Notebook 2

## 1. The problem: right on average, wrong for subgroups

We start from the split conformal interval of Notebook 2 and ask a sharper question than how many test barriers are
covered, namely, how many are covered *in each reaction class*.

In [ ]:
lower, upper, q_split = P.split_conformal(y[cal], mu[cal], mu[test], alpha)
covered = (y[test] >= lower) & (y[test] <= upper)

by_class = pd.DataFrame({
    "test reactions": pd.Series(rdb7.rxn_class[test]).value_counts(),
    "coverage": pd.Series(covered).groupby(rdb7.rxn_class[test]).mean(),
    "MAE (kcal/mol)": pd.Series(np.abs(y - mu)[test]).groupby(rdb7.rxn_class[test]).mean(),
}).reindex(P.CLASS_ORDER)
print(f"split conformal: every interval is mu ± {q_split:.2f} kcal/mol; overall coverage {covered.mean():.1%}")
display(by_class.round(3))

Overall the promise is kept, but the classes are treated very differently: nearly all reactions with a carbene product
are covered (the interval is wider than they need), while roughly one skeletal rearrangement in five falls outside its
"90%" interval. This is not a failure of the guarantee, which only ever spoke about the average over all reactions.
It is a reminder that the average can hide a great deal.

The remedy is to let the width depend on the reaction. Two standard constructions follow. Both keep the logic of
Notebook 2 (score, conformal quantile, interval) and change only the *score*.

## 2. A difficulty estimate $\hat\sigma(x)$

The first construction needs a number $\hat\sigma(x) > 0$ that is large when the model is likely to be wrong for
reaction $x$. Anything that can be computed without knowing the answer is allowed: the spread of an ensemble, the
distance to the training set, a predicted variance. Here we train a second model to predict the size of the first
model's error. Three steps, all on the **training set only**:

1. **Honest residuals.** The residuals of the training reactions themselves are useless for this purpose: the model
   has partly memorized them. We therefore use 5-fold cross-validation and record, for every training reaction, the
   error of a model that was fitted *without* it (an out-of-fold residual). The folds keep all reactions of one
   reactant together, in line with the split of Notebook 1.
2. **Learn the error size.** A second, smaller gradient-boosting model is fitted to predict $|\text{out-of-fold residual}|$
   from the same features.
3. **Keep it positive.** The prediction is bounded from below (here by half the median out-of-fold residual), because
   we are going to divide by it.

In [ ]:
# 1. out-of-fold residuals of the training reactions
train = model.train
fold = P.grouped_folds(rdb7.reactant[train], 5, P.SEED)          # fold number of every training reaction
oof = np.empty(len(train))
for k in range(5):
    fold_model = P.make_model().fit(model.X[train[fold != k]], model.target[train[fold != k]])
    oof[fold == k] = fold_model.predict(model.X[train[fold == k]])
abs_res_oof = np.abs(model.target[train] - oof)
abs_res_in = np.abs(model.target[train] - model.estimator.predict(model.X[train]))
print(f"mean |residual| of the training reactions: in-sample {abs_res_in.mean():.2f}, out-of-fold {abs_res_oof.mean():.2f} kcal/mol"
      f"   (held-out MAE: {np.abs(y - mu)[model.hold].mean():.2f})")

# 2. a second model predicts the size of the error;  3. bounded from below
sigma_model = P.make_model(max_iter=P.DIFFICULTY_TREES).fit(model.X[train], abs_res_oof)
floor = P.DIFFICULTY_FLOOR * np.median(abs_res_oof)
sigma = np.maximum(sigma_model.predict(model.X), floor)

assert np.allclose(sigma, P.fit_difficulty(rdb7, model.name, verbose=False))     # the same recipe is stored in primer.py
hold = model.hold
print(f"sigma on the held-out reactions: median {np.median(sigma[hold]):.2f}, "
      f"range {sigma[hold].min():.2f} to {sigma[hold].max():.2f} kcal/mol; {np.mean(sigma[hold] == floor):.0%} sit at the lower bound")

The out-of-fold residuals have the same size as the errors on the held-out reactions; the in-sample residuals are
about half as large. Before using $\hat\sigma$ we should check that it has anything to say about the actual errors of
reactions that neither model has seen:

In [ ]:
abs_err = np.abs(y - mu)
rank = np.empty(len(hold), dtype=int)
rank[np.argsort(sigma[hold], kind="stable")] = np.arange(len(hold))
tenth = rank * 10 // len(hold)                                    # 0 = the tenth with the smallest sigma, ..., 9 = the largest
reliability = pd.DataFrame({"mean sigma": pd.Series(sigma[hold]).groupby(tenth).mean(),
                            "mean |error|": pd.Series(abs_err[hold]).groupby(tenth).mean()})
reliability.index = [f"tenth {i + 1}" for i in reliability.index]
rho = stats.spearmanr(sigma[hold], abs_err[hold])[0]
display(reliability.T.round(2))
print(f"Spearman rank correlation between sigma and |error| on the held-out reactions: {rho:.2f}")

From the easiest to the hardest tenth of the held-out reactions the mean error grows about ninefold, and
$\hat\sigma$ follows it closely. The rank correlation for individual reactions is moderate (an error is a random
quantity; $\hat\sigma$ can only predict its typical size), but that is all the next step needs.

## 3. Normalized conformal intervals

Replace the score $|y - \hat\mu(x)|$ by the **normalized** score

$$ s(x, y) = \frac{|y - \hat\mu(x)|}{\hat\sigma(x)} , $$

take the conformal quantile $\hat q$ of the calibration scores exactly as before, and solve $s(x, y) \le \hat q$ for $y$:

$$ C(x) = \big[\hat\mu(x) - \hat q\,\hat\sigma(x),\ \hat\mu(x) + \hat q\,\hat\sigma(x)\big] . $$

$\hat q$ is now a dimensionless multiplier, and the half-width $\hat q\,\hat\sigma(x)$ differs from reaction to
reaction. The guarantee is untouched: the rank argument of Notebook 2 holds for *any* score, as long as the score
function (here $\hat\mu$ and $\hat\sigma$) was fixed before the calibration data were looked at. (Normalized scores go
back to Papadopoulos et al., 2002 and 2008; Lei et al., 2018, call the result "locally weighted" conformal prediction.)

In [ ]:
def normalized_conformal(y_cal, mu_cal, sigma_cal, mu_test, sigma_test, alpha=0.10):
    """Interval mu ± q * sigma(x) from the score |y - mu| / sigma. Returns (lower, upper, q)."""
    q = conformal_quantile(np.abs(y_cal - mu_cal) / sigma_cal, alpha)
    return mu_test - q * sigma_test, mu_test + q * sigma_test, q


lo_norm, hi_norm, q_norm = normalized_conformal(y[cal], mu[cal], sigma[cal], mu[test], sigma[test], alpha)
half_width = (hi_norm - lo_norm) / 2
print(f"q_hat = {q_norm:.2f} (dimensionless); coverage {np.mean((y[test] >= lo_norm) & (y[test] <= hi_norm)):.1%}")
print(f"half-widths: median {np.median(half_width):.2f}, 5th to 95th percentile {np.quantile(half_width, 0.05):.2f} "
      f"to {np.quantile(half_width, 0.95):.2f} kcal/mol   (split conformal: {q_split:.2f} for every reaction)")
print(f"{np.mean(half_width < q_split):.0%} of the test reactions now have a narrower interval than with split conformal")

## 4. Conformalized quantile regression (CQR)

The second construction (Romano, Patterson, and Candès, 2019) does not go through a point prediction at all. Gradient
boosting can be trained to predict a *quantile* of the barrier instead of its mean, by minimizing the "pinball" loss
$\rho_\tau(u) = \max\{\tau u,\ (\tau - 1)\,u\}$ of the residual $u$ for the quantile level $\tau$. We fit one model for
the 5% quantile and one for the 95% quantile; the band between them is a first, uncalibrated guess of a 90% interval.

In [ ]:
q_lo_model = P.make_model(loss="quantile", quantile=alpha / 2).fit(model.X[train], model.target[train])
q_hi_model = P.make_model(loss="quantile", quantile=1 - alpha / 2).fit(model.X[train], model.target[train])
band_lo = q_lo_model.predict(model.X) + model.baseline
band_hi = q_hi_model.predict(model.X) + model.baseline

raw_coverage = np.mean((y[hold] >= band_lo[hold]) & (y[hold] <= band_hi[hold]))
print(f"raw 5%–95% quantile band on the held-out reactions: coverage {raw_coverage:.1%} (nominal 90%)")

The raw band is too narrow: quantile models overfit like any other model. CQR repairs this with one more conformal
step. The score measures how far $y$ lies *outside* the band (it is negative when $y$ is inside),

$$ s(x, y) = \max\{\hat\ell(x) - y,\ \ y - \hat u(x)\} , $$

where $\hat\ell(x)$ and $\hat u(x)$ are the predictions of the lower and the upper quantile model. The interval is the
band, widened (or, if $\hat q < 0$, shrunk) by the conformal quantile $\hat q$ of the calibration scores:
$C(x) = [\hat\ell(x) - \hat q,\ \hat u(x) + \hat q]$.

In [ ]:
def cqr(y_cal, lo_cal, hi_cal, lo_test, hi_test, alpha=0.10):
    """Conformalized quantile regression. Returns (lower, upper, q)."""
    q = conformal_quantile(np.maximum(lo_cal - y_cal, y_cal - hi_cal), alpha)
    return lo_test - q, hi_test + q, q


lo_cqr, hi_cqr, q_cqr = cqr(y[cal], band_lo[cal], band_hi[cal], band_lo[test], band_hi[test], alpha)
print(f"q_hat = {q_cqr:.2f} kcal/mol is added on both sides; coverage {np.mean((y[test] >= lo_cqr) & (y[test] <= hi_cqr)):.1%}")

for name, mine, stored in (("normalized", (lo_norm, hi_norm), P.normalized_conformal(y[cal], mu[cal], sigma[cal], mu[test], sigma[test], alpha)),
                           ("CQR", (lo_cqr, hi_cqr), P.cqr(y[cal], band_lo[cal], band_hi[cal], band_lo[test], band_hi[test], alpha))):
    assert np.array_equal(mine[0], stored[0]) and np.array_equal(mine[1], stored[1])     # same functions as in primer.py

## 5. Did the intervals adapt? Diagnostics for conditional coverage

All three methods have 90% coverage overall. To compare them we need two more things.

**Efficiency.** Narrower is better, *given* the same coverage. We report the mean and the median half-width, and the
*interval score* (Gneiting and Raftery, 2007), which is the full width of the interval plus a penalty of $2/\alpha$
times the distance by which the interval misses the reference value; it rewards narrow intervals but charges for
misses in proportion to their size.

**Conditional coverage.** We would like 90% coverage not only overall but in every recognizable subgroup. Three
checks:

* *by a feature*: coverage within each reaction class;
* *by difficulty*: coverage within each fifth of the test reactions ordered by $\hat\sigma(x)$, from the easiest fifth
  to the hardest. This check uses the very quantity the normalized interval is built from, so it favors that method;
* *by the method's own interval width*: are the wide intervals covered as often as the narrow ones? This check can
  be made for any adaptive method and favors none.

Everything is averaged over 200 random calibration/test splits of the held-out reactions.

In [ ]:
def fifths(values):
    """0 for the fifth of the cases with the smallest values, ..., 4 for the fifth with the largest."""
    return np.digitize(values, np.quantile(values, [0.2, 0.4, 0.6, 0.8]))


def evaluate(lower, upper, test_rows, difficulty):
    """Coverage, half-width, and conditional-coverage diagnostics of one set of intervals."""
    inside = (y[test_rows] >= lower) & (y[test_rows] <= upper)
    half = (upper - lower) / 2
    out = {"coverage": inside.mean(), "mean half-width": half.mean(), "median half-width": np.median(half),
           "interval score": P.interval_score(y[test_rows], lower, upper, alpha)}
    for c in P.CLASS_ORDER:
        out[f"class: {P.CLASS_SHORT[c]}"] = inside[rdb7.rxn_class[test_rows] == c].mean()
    fifth = fifths(difficulty)
    for j in range(5):
        out[f"sigma fifth {j + 1}"] = inside[fifth == j].mean()
    if np.ptp(half) > 1e-9:                                         # only meaningful if the widths differ
        fifth = fifths(half)
        for j in range(5):
            out[f"own-width fifth {j + 1}"] = inside[fifth == j].mean()
    return out


METHODS = ["split", "normalized", "CQR"]
rows = []
for cal_b, test_b in P.resplits(model.hold, 1000, 200, seed=11):
    intervals = {
        "split": P.split_conformal(y[cal_b], mu[cal_b], mu[test_b], alpha),
        "normalized": normalized_conformal(y[cal_b], mu[cal_b], sigma[cal_b], mu[test_b], sigma[test_b], alpha),
        "CQR": cqr(y[cal_b], band_lo[cal_b], band_hi[cal_b], band_lo[test_b], band_hi[test_b], alpha),
    }
    for method, (lo, hi, _) in intervals.items():
        rows.append({"method": method, **evaluate(lo, hi, test_b, sigma[test_b])})
summary = pd.DataFrame(rows).groupby("method").mean().reindex(METHODS)

print("Efficiency (half-widths and interval score in kcal/mol)")
display(summary[["coverage", "mean half-width", "median half-width", "interval score"]].round(3))
print("Coverage by reaction class")
display(summary[[c for c in summary.columns if c.startswith("class")]].round(3))
print("Coverage by difficulty: fifths of the test reactions ordered by sigma (1 = easiest)")
display(summary[[c for c in summary.columns if c.startswith("sigma fifth")]].round(3))
print("Coverage by the method's own interval width (1 = narrowest fifth)")
display(summary.loc[["normalized", "CQR"], [c for c in summary.columns if c.startswith("own-width")]].round(3))

How to read the four tables:

* **Efficiency.** Both adaptive methods have a smaller mean half-width than the constant interval, a much smaller
  *median* half-width (most reactions are easy and now get a tighter interval), and a clearly better interval score.
* **Classes.** The constant interval covers only about 80% of the skeletal rearrangements and about 97% of the reactions
  with a carbene product. The normalized interval brings every class to within about one and a half percentage points
  of 90%, although it was not given the class label (the descriptors from which the classes are derived are among its
  inputs).
* **Difficulty.** The constant interval covers practically all of the easiest fifth and only about two thirds of the
  hardest fifth: its misses are concentrated exactly where a user most needs a warning. The normalized interval stays
  between about 88% and 93%. CQR improves on the constant interval but still over-covers easy and under-covers hard
  reactions here.
* **Own width.** For CQR the wide intervals are covered less often than the narrow ones: its widest intervals are still
  too narrow. The reason is visible in the recipe: CQR adds the *same* $\hat q$ to a narrow band and to a wide band.

Which adaptive method wins depends on the data and on the auxiliary models. On other problems CQR can be the better
choice, for example when the error distribution is skewed, because its two ends are fitted separately (Romano et al.,
2019; Sesia and Candès, 2020). The point of this section is the habit: **compute the conditional diagnostics before
trusting an "adaptive" interval.**

In [ ]:
fig, axes = P.panels(3)

# a: does sigma track the error?
ax = axes[0]
lim = float(np.ceil(1.1 * reliability.to_numpy().max()))            # 6 kcal/mol for the Δ-learning model
ax.plot([0, lim], [0, lim], color=P.GRAY, lw=0.8, ls="--")
ax.plot(reliability["mean sigma"], reliability["mean |error|"], marker="o", color=P.BLUE, lw=1.0)
ax.set(xlabel=r"predicted error size $\hat{\sigma}$ (kcal/mol)", ylabel="mean |error| (kcal/mol)", xlim=(0, lim), ylim=(0, lim))
if lim <= 6:
    ax.set_xticks(np.arange(lim + 1))
    ax.set_yticks(np.arange(lim + 1))
ax.text(0.96, 0.06, "tenths of the held-out\nreactions, ordered by " + r"$\hat{\sigma}$", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5)
P.panel_label(ax, "a", r"$\hat{\sigma}$ tracks the error")

# b: coverage by difficulty
ax = axes[1]
styles = {"split": (P.VERMILLION, "s", "-", "constant width"), "normalized": (P.BLUE, "o", "-", "normalized"), "CQR": (P.GREEN, "^", "--", "CQR")}
positions = np.arange(1, 6)
for method, (color, marker, line, label) in styles.items():
    ax.plot(positions, summary.loc[method, [f"sigma fifth {j}" for j in positions]].values, marker=marker, ls=line, color=color, label=label)
ax.axhline(1 - alpha, color=P.GRAY, lw=0.8, ls="--")
lowest = summary[[f"sigma fifth {j}" for j in positions]].to_numpy().min()
ax.set(xlabel=r"fifths of the test reactions by $\hat{\sigma}$", ylabel="coverage", ylim=(min(0.6, lowest - 0.05), 1.02), xlim=(0.7, 5.3))
ax.set_xticks(positions, ["easiest", "2", "3", "4", "hardest"])
ax.legend(loc="lower left", borderaxespad=0.3)
P.panel_label(ax, "b", "Coverage by difficulty")

# c: distribution of half-widths
ax = axes[2]
w_max = 12.0 if q_split < 5 else 4.0 * np.ceil(2.4 * q_split / 4)   # 12 kcal/mol for the Δ-learning model
grid = np.linspace(0, w_max, 400)
for method, half in (("normalized", (hi_norm - lo_norm) / 2), ("CQR", (hi_cqr - lo_cqr) / 2)):
    color, marker, line, label = styles[method]
    ax.plot(grid, [np.mean(half <= g) for g in grid], color=color, ls=line, label=label)
ax.axvline(q_split, color=P.VERMILLION, lw=1.2, ls=":")
ax.text(q_split + 0.02 * w_max, 0.42, f"constant width:\n±{q_split:.1f} for all", fontsize=6.5, va="center")
ax.set(xlabel="interval half-width (kcal/mol)", ylabel="cumulative fraction of test reactions", xlim=(0, w_max), ylim=(0, 1.02))
ax.set_xticks(np.linspace(0, w_max, 5))
ax.legend(loc="lower right", borderaxespad=0.3)
P.panel_label(ax, "c", "Most intervals shrink")
for ax in axes:
    ax.set_box_aspect(1)
P.savefig(fig, "nb3_adaptive_intervals")
plt.show()

Panel **a**: the predicted error size follows the observed error over a factor of about nine. Panel **b**: coverage in
each fifth of the test reactions ordered by difficulty (the vertical axis starts at 0.6); a perfectly adaptive method
would be a horizontal line at 0.90. Panel **c**: cumulative distribution of the half-widths. With the normalized score
about two thirds of the reactions get a narrower interval than before, and the remaining third a wider one. The
vertical step at the left end of the normalized curve is the lower bound of $\hat\sigma$: the easiest reactions all
receive the same, smallest half-width.

## 6. A wide interval is information

An adaptive interval ranks the predictions by how much they should be trusted. The widest ones are worth a look:

In [ ]:
order = np.argsort(half_width, kind="stable")
listing = pd.DataFrame({
    "reaction": [f"{P.readable(rdb7.df.rsmi[i])} >> {P.readable(rdb7.df.psmi[i])}" for i in test[order]],
    "class": rdb7.rxn_class[test[order]],
    "predicted": mu[test[order]].round(1),
    "± half-width": half_width[order].round(1),
    "reference": y[test[order]].round(1),
})
print("five of the narrowest and the five widest normalized intervals in the test set (kcal/mol)")
with pd.option_context("display.max_colwidth", None):              # show the SMILES strings in full
    display(pd.concat([listing.head(5), listing.tail(5)]))
widest = order[-len(order) // 10:]
print(f"widest 10% of the intervals: mean |error| {abs_err[test[widest]].mean():.2f} kcal/mol;   "
      f"all others: {abs_err[test[order[:-len(order) // 10]]].mean():.2f} kcal/mol")

In a screening campaign the reactions with the widest intervals are the natural candidates for a higher-level
calculation, and the narrow ones can be taken as they are.

## 7. A pitfall, and a limit

**Pitfall: a difficulty model trained on in-sample residuals.** Step 1 of Section 2 looks like a technicality. Skip
it (fit the difficulty model to the residuals of the training reactions themselves) and the result is still a valid
90% interval, because validity never depends on $\hat\sigma$. But the adaptation is damaged:

In [ ]:
sigma_in_model = P.make_model(max_iter=P.DIFFICULTY_TREES).fit(model.X[train], abs_res_in)
sigma_in = np.maximum(sigma_in_model.predict(model.X), P.DIFFICULTY_FLOOR * np.median(abs_res_in))

rows = []
for cal_b, test_b in P.resplits(model.hold, 1000, 200, seed=11):
    for label, s in (("out-of-fold residuals (Section 2)", sigma), ("in-sample residuals", sigma_in)):
        lo, hi, _ = normalized_conformal(y[cal_b], mu[cal_b], s[cal_b], mu[test_b], s[test_b], alpha)
        inside = (y[test_b] >= lo) & (y[test_b] <= hi)
        fifth = fifths(sigma[test_b])                                  # difficulty as estimated in Section 2, for both rows
        rows.append({"difficulty model trained on": label, "coverage": inside.mean(), "easiest fifth": inside[fifth == 0].mean(),
                     "hardest fifth": inside[fifth == 4].mean(),
                     "skeletal rearrangements": inside[rdb7.rxn_class[test_b] == "skeletal rearrangement"].mean()})
pitfall = pd.DataFrame(rows).groupby("difficulty model trained on", sort=False).mean()
display(pitfall.round(3))

With in-sample residuals the easy reactions are over-covered and the hard ones under-covered again (about 95% and
85% for the easiest and the hardest fifth; the fifths are those of the out-of-fold $\hat\sigma$ in both rows), and the
skeletal rearrangements fall back to about 87%: the difficulty estimate has lost part of its meaning.

**Limit: conditional coverage cannot be guaranteed.** Normalized scores and CQR make the coverage *more* uniform, and
the diagnostics above let us see how much. They do not turn the marginal guarantee into a conditional one. It is a
theorem (Vovk, 2013; Lei and Wasserman, 2014; Foygel Barber et al., 2021) that no method can guarantee
$\Pr\{Y \in C(X) \mid X = x\} \ge 1-\alpha$ for every individual $x$ and every distribution, unless its intervals are
infinitely wide (precisely: at every $x$ that does not recur exactly in the population, the expected width must be
infinite). What *can* be guaranteed is coverage within a finite number of groups chosen in advance. That is the subject
of Notebook 4.

## Take-aways

* A constant-width interval can have the right overall coverage and still fail badly for the hardest cases.
* Changing the score changes the shape of the interval, not the guarantee: divide the residual by a difficulty
  estimate (normalized score), or start from a quantile band (CQR).
* A difficulty estimate can be any quantity computed from $x$ alone. If it is learned from residuals, use
  out-of-fold residuals.
* Always check conditional coverage: by chemical class, by difficulty, and by the method's own interval width.
* Adaptive intervals usually give better conditional coverage in practice, never a guarantee for individual reactions.

## Try it yourself

1. Repeat the comparison for the structure-only model (set `model = rdb7.models["structure"]` in the cell that loads
   the data, where `model` is defined). Is the gain from normalization larger or smaller, and why?
2. Replace $\hat\sigma(x)$ by something much simpler, for example the absolute size of the correction that the
   Δ-learning model predicts, `np.abs(mu - model.baseline)` plus a small constant (`model.baseline` is the ωB97X-D3
   barrier). How far does that get?
3. Set `P.DIFFICULTY_FLOOR = 0.05` in the cell where `model` is defined and run the notebook again. How well is the
   narrowest fifth of the normalized intervals covered now, and what does the lower bound protect against?

## References

* Papadopoulos, H.; Proedrou, K.; Vovk, V.; Gammerman, A. Inductive Confidence Machines for Regression. In *Machine
  Learning: ECML 2002*; Springer: Berlin, Heidelberg, 2002; pp 345–356. https://doi.org/10.1007/3-540-36755-1_29
* Papadopoulos, H.; Gammerman, A.; Vovk, V. Normalized Nonconformity Measures for Regression Conformal Prediction. In
  *Proceedings of the IASTED International Conference on Artificial Intelligence and Applications (AIA 2008)*;
  ACTA Press, 2008; pp 64–69.
* Lei, J.; G'Sell, M.; Rinaldo, A.; Tibshirani, R. J.; Wasserman, L. Distribution-Free Predictive Inference for
  Regression. *J. Am. Stat. Assoc.* **2018**, *113*, 1094–1111. https://doi.org/10.1080/01621459.2017.1307116
* Romano, Y.; Patterson, E.; Candès, E. J. Conformalized Quantile Regression. In *Advances in Neural Information
  Processing Systems 32*; Curran Associates, Inc., 2019; pp 3543–3553.
  https://proceedings.neurips.cc/paper/2019/hash/5103c3584b063c431bd1268e9b5e76fb-Abstract.html
* Sesia, M.; Candès, E. J. A Comparison of Some Conformal Quantile Regression Methods. *Stat* **2020**, *9*, e261.
  https://doi.org/10.1002/sta4.261
* Gneiting, T.; Raftery, A. E. Strictly Proper Scoring Rules, Prediction, and Estimation. *J. Am. Stat. Assoc.* **2007**,
  *102*, 359–378. https://doi.org/10.1198/016214506000001437
* Vovk, V. Conditional Validity of Inductive Conformal Predictors. *Mach. Learn.* **2013**, *92*, 349–376.
  https://doi.org/10.1007/s10994-013-5355-6 (conference version 2012)
* Lei, J.; Wasserman, L. Distribution-Free Prediction Bands for Non-Parametric Regression. *J. R. Stat. Soc. Ser. B:
  Stat. Methodol.* **2014**, *76*, 71–96. https://doi.org/10.1111/rssb.12021
* Foygel Barber, R.; Candès, E. J.; Ramdas, A.; Tibshirani, R. J. The Limits of Distribution-Free Conditional
  Predictive Inference. *Inf. Inference* **2021**, *10*, 455–482. https://doi.org/10.1093/imaiai/iaaa017

In [ ]:
results = {
    "q": {"split": q_split, "normalized": q_norm, "cqr": q_cqr},
    "sigma": {"oof_mae": abs_res_oof.mean(), "in_sample_mae": abs_res_in.mean(), "spearman": rho, "floor": floor,
              "share_at_floor": float(np.mean(sigma[hold] == floor)),
              "tenth_sigma": reliability["mean sigma"].values, "tenth_error": reliability["mean |error|"].values},
    "raw_quantile_band_coverage": raw_coverage,
    "class_coverage_single_split": {P.CLASS_SHORT[c]: by_class.loc[c, "coverage"] for c in P.CLASS_ORDER},
    "summary": {method: {k: v for k, v in row.items() if np.isfinite(v)} for method, row in summary.to_dict("index").items()},
    "pitfall_in_sample": {k: dict(v) for k, v in pitfall.to_dict("index").items()},
    "fraction_narrower_than_split": float(np.mean(half_width < q_split)),
    "widest_tenth_error": {"widest": float(abs_err[test[widest]].mean()), "others": float(abs_err[test[order[:-len(order) // 10]]].mean())},
}
P.save_results("nb3", results, rdb7)
P.check("nb3", results);

# Notebook 4 · Mondrian conformal prediction: a guarantee for every reaction class

**Conformal Prediction for Physical Chemists: hands-on notebooks (4 of 6)**

Notebook 3 ended with a limit: coverage cannot be guaranteed for every individual reaction. It *can* be guaranteed for
every member of a finite list of groups that is fixed in advance. The recipe is very simple: run split conformal
prediction separately within each group. Because the groups tile the data like the rectangles of a Mondrian painting,
this is called Mondrian (or group-conditional) conformal prediction (Vovk, Gammerman, and Shafer, 2005; Vovk, 2013). In
cheminformatics it is in routine use, for example to treat active and inactive compounds separately (Norinder et al.,
2014).

**You will**

1. calibrate one conformal quantile per reaction class and obtain 90% coverage in *each* class;
2. understand the price: every class now has its own, smaller calibration set;
3. see a taxonomy that is too fine for the calibration budget, and how to repair it;
4. combine Mondrian calibration with the normalized score of Notebook 3;
5. learn how to choose the categories.

Runs on a CPU in under a minute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P.set_style()
rdb7 = P.load_problem()
y = rdb7.y
alpha = 0.10

model = rdb7.models["delta"]
mu = model.mu
abs_err = np.abs(y - mu)
classes = rdb7.rxn_class
sigma = P.fit_difficulty(rdb7, "delta")       # the difficulty estimate of Notebook 3

shuffled = np.random.default_rng(P.SEED).permutation(model.hold)
cal, test = shuffled[:1000], shuffled[1000:]  # the same split as in Notebooks 2 and 3

conformal_quantile = P.conformal_quantile

## 1. One quantile per class

Recall the five reaction classes of Notebook 1 (assigned automatically to each reaction) and how unevenly the
constant-width interval treats them. In Mondrian conformal prediction the calibration scores are sorted into the
classes, and each class gets its own conformal quantile.

In [ ]:
def mondrian_quantiles(scores, groups, alpha=0.10):
    """One conformal quantile per group: {group label: q}."""
    return {g: conformal_quantile(scores[groups == g], alpha) for g in np.unique(groups)}


def mondrian_conformal(y_cal, mu_cal, groups_cal, mu_test, groups_test, alpha=0.10):
    """Absolute-residual intervals with a separate half-width for every group. Returns (lower, upper, {group: q})."""
    q = mondrian_quantiles(np.abs(y_cal - mu_cal), groups_cal, alpha)
    q_test = np.array([q.get(g, np.inf) for g in groups_test])      # a group without calibration data gets no finite interval
    return mu_test - q_test, mu_test + q_test, q


q_global = conformal_quantile(abs_err[cal], alpha)
lower, upper, q_class = mondrian_conformal(y[cal], mu[cal], classes[cal], mu[test], classes[test], alpha)
assert q_class == P.mondrian_quantiles(abs_err[cal], classes[cal], alpha)

inside_global = abs_err[test] <= q_global
inside_mondrian = (y[test] >= lower) & (y[test] <= upper)
table = pd.DataFrame({
    "calibration reactions": pd.Series(classes[cal]).value_counts(),
    "half-width, one q for all": q_global,
    "half-width, Mondrian": pd.Series(q_class),
    "coverage, one q for all": pd.Series(inside_global).groupby(classes[test]).mean(),
    "coverage, Mondrian": pd.Series(inside_mondrian).groupby(classes[test]).mean(),
}).reindex(P.CLASS_ORDER)
display(table.round(3))
print(f"overall coverage: one q for all {inside_global.mean():.1%}, Mondrian {inside_mondrian.mean():.1%}")

The class-wise half-widths differ by a factor of two to three: reactions with a carbene product need roughly
±2 kcal/mol, skeletal rearrangements roughly ±6 to ±7 kcal/mol. With these half-widths every class is covered at close to
90%.

**The guarantee.** Within each class the calibration scores and the score of a new reaction *of that class* are
exchangeable, so the rank argument of Notebook 2 applies class by class:

$$\Pr\{\,Y \in C(X) \mid \text{class of } X = g\,\} \;\ge\; 1-\alpha \qquad \text{for every class } g .$$

This is a strictly stronger statement than the marginal guarantee, and it implies it.

The table above is for one calibration set. Averaged over 200 random calibration/test splits:

In [ ]:
def class_coverage(inside, groups):
    return {g: inside[groups == g].mean() for g in P.CLASS_ORDER}


rows, per_split = [], []
for cal_b, test_b in P.resplits(model.hold, 1000, 200, seed=21):
    q_b = conformal_quantile(abs_err[cal_b], alpha)
    lo, hi, qc = mondrian_conformal(y[cal_b], mu[cal_b], classes[cal_b], mu[test_b], classes[test_b], alpha)
    inside = {"one q for all": abs_err[test_b] <= q_b, "Mondrian": (y[test_b] >= lo) & (y[test_b] <= hi)}
    for method, ins in inside.items():
        rows.append({"method": method, "all": ins.mean(), "mean half-width": q_b if method == "one q for all" else np.mean(hi - lo) / 2,
                     **class_coverage(ins, classes[test_b])})
    per_split.append({"n_cal": pd.Series(classes[cal_b]).value_counts(), "coverage": pd.Series(class_coverage(inside["Mondrian"], classes[test_b])),
                      "q": pd.Series(qc)})
average = pd.DataFrame(rows).groupby("method", sort=False).mean()
print("Coverage (mean over 200 splits)")
display(average[["all"] + P.CLASS_ORDER].rename(columns=P.CLASS_SHORT).round(3))
print("mean half-width over all test reactions (kcal/mol):", average["mean half-width"].round(2).to_dict())

## 2. The price: smaller calibration sets

Nothing is free. A class with $n_g$ calibration reactions has the coverage distribution of a calibration set of size
$n_g$ (the Beta law of Notebook 2), not of size $n$. With 1000 calibration reactions in total, the smallest class has
only about 140. Its coverage is right on average but varies noticeably from one calibration set to the next:

In [ ]:
n_cal_class = pd.DataFrame([s["n_cal"] for s in per_split]).mean().reindex(P.CLASS_ORDER)
cov_class = pd.DataFrame([s["coverage"] for s in per_split]).reindex(columns=P.CLASS_ORDER)
spread = pd.DataFrame({
    "mean n_g": n_cal_class.round(0).astype(int),
    "mean coverage": cov_class.mean(),
    "2.5%": cov_class.quantile(0.025),
    "97.5%": cov_class.quantile(0.975),
    "Beta law 2.5%": [P.coverage_law(int(round(n)), alpha).ppf(0.025) for n in n_cal_class],
    "Beta law 97.5%": [P.coverage_law(int(round(n)), alpha).ppf(0.975) for n in n_cal_class],
})
print("Class-wise coverage of Mondrian intervals over 200 calibration sets of 1000 reactions")
display(spread.round(3))

(For some classes the observed range is a little wider than the Beta law predicts, because each class also has a finite
*test* set here and the number of its calibration reactions varies from split to split. With 200 splits, the 2.5% and
97.5% points are themselves uncertain by a few thousandths.)

The effect becomes dramatic when the total calibration budget is small. Below, the same experiment for calibration
sets of 50 to 2000 reactions: how wide are the class-wise intervals on average, and how often does at least one class
end up with fewer than the 9 calibration reactions that a 90% interval needs (Notebook 2) and therefore with an
infinite interval?

In [ ]:
SMALLEST = "skeletal rearrangement"                       # the class with the fewest reactions
rows = []
for n_cal in (50, 100, 200, 500, 1000, 2000):
    half_widths, infinite, class_cov = [], [], []
    for cal_b, test_b in P.resplits(model.hold, n_cal, 300, seed=22):
        qc = mondrian_quantiles(abs_err[cal_b], classes[cal_b], alpha)
        qs = np.array([qc.get(g, np.inf) for g in P.CLASS_ORDER])
        infinite.append(np.isinf(qs).any())
        half_widths.extend(qs[np.isfinite(qs)])
        for g, q in zip(P.CLASS_ORDER, qs):
            if np.isfinite(q):
                class_cov.append(np.mean(abs_err[test_b][classes[test_b] == g] <= q))
    rows.append({"calibration reactions": n_cal, "smallest class (mean n_g)": n_cal * np.mean(classes[model.hold] == SMALLEST),
                 "splits with an infinite interval": np.mean(infinite), "mean finite half-width": np.mean(half_widths),
                 "class coverage 2.5%": np.quantile(class_cov, 0.025), "class coverage 97.5%": np.quantile(class_cov, 0.975)})
budget = pd.DataFrame(rows).set_index("calibration reactions")
display(budget.round(3))

With 50 calibration reactions, one class or another gets an infinite interval in almost every split, and with 100 this
still happens now and then. From 200 on the intervals are finite, but the coverage of an individual class (last two
columns: the range that contains 95% of all class-wise coverages with a finite interval) can be far from 90%, and the
half-widths are inflated. As a rule of thumb, each category should hold a few hundred calibration points if its
coverage is to be within a few percentage points of the target: the planning table of Notebook 2 applies to every
category separately.

## 3. A taxonomy that is too fine

The authors of RDB7 labeled about 12% of the reactions with a reaction family of the Reaction Mechanism Generator (RMG;
Gao et al., 2016). Chemically this is the more natural classification. (The largest family, `ketoenol`, is RMG's
template for the 1,3-shift of an atom or group between an oxygen atom and a carbon or nitrogen atom, as in a keto–enol
tautomerization; in RDB7 about half of its 687 reactions shift a hydrogen atom and the other half a heavy-atom group.)
Statistically it is a problem:

In [ ]:
family = rdb7.df["rmg_family"].fillna("(no family)").to_numpy()
q_family = mondrian_quantiles(abs_err[cal], family[cal], alpha)
fam_table = pd.DataFrame({
    "calibration reactions": pd.Series(family[cal]).value_counts(),
    "test reactions": pd.Series(family[test]).value_counts(),
    "half-width (kcal/mol)": pd.Series(q_family),
})
fam_table["calibration reactions"] = fam_table["calibration reactions"].fillna(0).astype(int)
fam_table["test reactions"] = fam_table["test reactions"].fillna(0).astype(int)
fam_table["half-width (kcal/mol)"] = fam_table["half-width (kcal/mol)"].fillna(np.inf)      # no calibration data: no finite interval
fam_table = fam_table.sort_index().sort_values("calibration reactions", ascending=False, kind="stable")
display(fam_table.round(2))
unseen = sorted(set(family[test]) - set(family[cal]))
print(f"families in the test set that do not occur in the calibration set at all: {unseen}")

Only a few families have enough calibration reactions for a finite 90% interval, and even those rest on a few dozen
scores at most; some families of the test set never occur in the calibration set. The honest answer for these
reactions is "no guarantee at this level with these data", which the infinite half-width expresses.

The repair is to **merge small categories before calibrating**. The merging rule must not look at the calibration
*answers*; the cleanest way is to decide it on the training set, as below: a family keeps its own category only if
the training set contains at least 150 reactions of it.

In [ ]:
train_counts = pd.Series(family[model.train]).value_counts()
keep = set(train_counts[train_counts >= 150].index)
merged = np.array([f if f in keep else "(other families)" for f in family])
categories = sorted(set(merged))

rows = []
for cal_b, test_b in P.resplits(model.hold, 1000, 200, seed=23):
    qm = mondrian_quantiles(abs_err[cal_b], merged[cal_b], alpha)
    for g in categories:
        rows.append({"category": g, "calibration reactions": np.sum(merged[cal_b] == g), "half-width (kcal/mol)": qm[g],
                     "coverage": np.mean(abs_err[test_b][merged[test_b] == g] <= qm[g])})
per_category = pd.DataFrame(rows).groupby("category")
merged_table = pd.DataFrame({
    "training reactions": pd.Series(merged[model.train]).value_counts(),
    "calibration reactions (mean)": per_category["calibration reactions"].mean(),
    "half-width (mean)": per_category["half-width (kcal/mol)"].mean(),
    "coverage (mean)": per_category["coverage"].mean(),
    "coverage 2.5%": per_category["coverage"].quantile(0.025),
    "coverage 97.5%": per_category["coverage"].quantile(0.975),
}).sort_values("training reactions", ascending=False, kind="stable")
print("Merged taxonomy, 200 calibration sets of 1000 reactions")
display(merged_table.round(3))

Three categories survive. Each is covered at 90% on average, and the two small ones show the scatter that
Section 2 predicts for a few dozen calibration reactions. The merged taxonomy also tells us something about the
chemistry: reactions that match a known template get clearly narrower intervals (about ±2 to ±3 kcal/mol) than the
large remainder of unusual reactions that no template describes (about ±4 kcal/mol).

## 4. Mondrian and normalized scores together

Mondrian calibration equalizes coverage *between* classes; inside a class every reaction still gets the same width.
The normalized score of Notebook 3 adapts the width to the individual reaction but guarantees nothing per class.
The two ideas are independent and can be combined: compute the normalized score $|y-\hat\mu|/\hat\sigma$ and take its
conformal quantile class by class.

In [ ]:
def fifths(values):
    return np.digitize(values, np.quantile(values, [0.2, 0.4, 0.6, 0.8]))


rows = []
for cal_b, test_b in P.resplits(model.hold, 1000, 200, seed=21):
    candidates = {}
    q = conformal_quantile(abs_err[cal_b], alpha)
    candidates["split (one q)"] = np.full(len(test_b), q)
    qc = mondrian_quantiles(abs_err[cal_b], classes[cal_b], alpha)
    candidates["Mondrian"] = np.array([qc[g] for g in classes[test_b]])
    q = conformal_quantile(abs_err[cal_b] / sigma[cal_b], alpha)
    candidates["normalized"] = q * sigma[test_b]
    qc = mondrian_quantiles(abs_err[cal_b] / sigma[cal_b], classes[cal_b], alpha)
    candidates["Mondrian + normalized"] = np.array([qc[g] for g in classes[test_b]]) * sigma[test_b]
    difficulty = fifths(sigma[test_b])
    for method, half in candidates.items():
        inside = abs_err[test_b] <= half
        rows.append({"method": method, "coverage": inside.mean(), "median half-width": np.median(half), "mean half-width": half.mean(),
                     **{g: inside[classes[test_b] == g].mean() for g in P.CLASS_ORDER},
                     "easiest fifth": inside[difficulty == 0].mean(), "hardest fifth": inside[difficulty == 4].mean()})
means = pd.DataFrame(rows).groupby("method", sort=False).mean()
combined = means[["coverage", "median half-width", "mean half-width"]].copy()
combined["worst-covered class"] = means[P.CLASS_ORDER].min(axis=1)
combined["best-covered class"] = means[P.CLASS_ORDER].max(axis=1)
combined[["easiest fifth", "hardest fifth"]] = means[["easiest fifth", "hardest fifth"]]
display(combined.round(3))

Read the columns as three questions. *Is the overall coverage right?* For all four methods. *Are the classes treated
equally?* Only with Mondrian calibration is this guaranteed (columns "worst-covered class" and "best-covered class");
the normalized score comes close without a guarantee. *Are easy and hard reactions treated equally?* Much more nearly
with the normalized score than without it. The combination does well on all three counts, and it keeps the small
median half-width of the normalized score.

In [ ]:
fig, axes = P.panels(3)
short = [P.CLASS_SHORT[c] for c in P.CLASS_ORDER]
ypos = np.arange(len(P.CLASS_ORDER))[::-1]

# a: class-wise coverage, one quantile versus Mondrian
ax = axes[0]
ax.axvline(1 - alpha, color=P.GRAY, lw=0.8, ls="--")
ax.plot(average.loc["one q for all", P.CLASS_ORDER], ypos, ls="none", marker="s", color=P.VERMILLION, label=r"one $\hat{q}$ for all")
ax.plot(average.loc["Mondrian", P.CLASS_ORDER], ypos, ls="none", marker="o", color=P.BLUE, label="Mondrian")
ax.set_yticks(ypos, short)
ax.set(xlabel="coverage", xlim=(0.75, 1.0), ylim=(-0.6, 4.8))
ax.legend(loc="lower left", handletextpad=0.2, borderaxespad=0.3)
P.panel_label(ax, "a", "Coverage by class")

# b: class-wise half-widths
ax = axes[1]
q_mean = pd.DataFrame([s["q"] for s in per_split]).mean().reindex(P.CLASS_ORDER)
q_all = average.loc["one q for all", "mean half-width"]
ax.axvline(q_all, color=P.VERMILLION, lw=1.2, ls="--")
ax.hlines(ypos, 0, q_mean, color=P.BLUE, lw=5)
for yy, value in zip(ypos, q_mean):
    ax.text(8.6, yy, f"{value:.1f}", va="center", ha="right", fontsize=7)
ax.text(q_all + 0.15, 4.5, r"one $\hat{q}$ for all", fontsize=6.5, va="center")
ax.set_yticks(ypos, short)
ax.set(xlabel="half-width of 90% interval (kcal/mol)", xlim=(0, 8.7), ylim=(-0.6, 4.8))
ax.set_xticks([0, 2, 4, 6, 8])
P.panel_label(ax, "b", "Mondrian half-widths")

# c: how much does the coverage of one class scatter?
ax = axes[2]
grid_n = np.unique(np.round(np.logspace(1, np.log10(3000), 150)).astype(int))
laws = [P.coverage_law(int(n), alpha) for n in grid_n]
ax.fill_between(grid_n, [l.ppf(0.025) for l in laws], [l.ppf(0.975) for l in laws], color=P.SKY, alpha=0.35, lw=0)
ax.plot(spread["mean n_g"], spread["2.5%"], ls="none", marker="v", color=P.VERMILLION)
ax.plot(spread["mean n_g"], spread["97.5%"], ls="none", marker="^", color=P.VERMILLION)
ax.axhline(1 - alpha, color=P.GRAY, lw=0.8, ls="--")
ax.text(0.96, 0.17, "band: 95% range (Beta law)", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5)
ax.text(0.96, 0.05, "triangles: the five classes,\n200 calibration sets", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5)
ax.set_xscale("log")
ax.set(xlabel="calibration reactions per class, $n_g$", ylabel="coverage of the class", xlim=(10, 3000), ylim=(0.6, 1.0))
ax.set_xticks([10, 100, 1000], ["10", "100", "1000"])
P.panel_label(ax, "c", "The price: smaller $n$")
for ax in axes:
    ax.set_box_aspect(1)
P.savefig(fig, "nb4_mondrian")
plt.show()

Panel **a**: class-wise coverage with a single quantile (squares) and with Mondrian calibration (circles), averaged over
200 splits. Panel **b**: the class-wise half-widths behind it (bars) and the single half-width (dashed line). Panel
**c**: each class pays with the coverage variability of its own, smaller calibration set; the triangles are the observed
2.5% and 97.5% points of the five class-wise coverages. The coverage axes start at 0.75 (a) and 0.6 (c).

## 5. Choosing the categories

Any rule may be used that assigns a category to a reaction **from $x$ alone and was fixed before calibration**:

* chemistry: reaction class, functional group, element content, charge, or spin state;
* provenance: which data source, which level of theory, which laboratory;
* size: number of heavy atoms, in bins;
* the model itself: bins of the predicted value, or bins of a difficulty estimate (with bin edges fixed beforehand,
  for example on the training set).

A category may **not** be defined through the unknown answer $y$ in regression ("reactions with a barrier below
40 kcal/mol" is not a valid category; "reactions with a *predicted* barrier below 40 kcal/mol" is). In classification
there is a second, very useful variant in which the categories are the true labels (label-conditional conformal
prediction); it guarantees coverage for every class of the outcome and is the standard remedy for imbalanced data.
Guarantees for groups that overlap need more than this recipe (Gibbs, Cherian, and Candès, 2025).

Three practical rules: keep the number of categories small; give each one a calibration set large enough for the
precision you need; and when the natural taxonomy is too fine, merge on the training set. A further benefit comes
for free: because every category is calibrated on its own, the guarantee survives when the *proportions* of the
categories change between calibration and use. Notebook 5 uses exactly that.

## Take-aways

* Mondrian conformal prediction = split conformal prediction within each category. It guarantees the coverage of
  every category, which the marginal guarantee does not.
* Each category pays with the Beta law of its own calibration set; categories with fewer than $(1-\alpha)/\alpha$
  calibration points get infinite intervals.
* Merge small categories using the training set, never the calibration answers.
* Mondrian calibration and normalized scores address different problems and combine without difficulty.

## Try it yourself

1. Use the number of heavy atoms of the reactant (`rdb7.n_heavy`, merged into "up to 5", "6", and "7") as the taxonomy.
   Are the size classes treated unequally by the single quantile?
2. Use fifths of the difficulty estimate `sigma` as categories, with the four cut points computed once from the
   training reactions (`np.quantile(sigma[model.train], [0.2, 0.4, 0.6, 0.8])`). How does this "binned" adaptivity
   compare with the normalized score?
3. Lower the merging threshold of Section 3 from 150 to 30 training reactions and look at the coverage of the small
   families over many splits.

## References

* Vovk, V.; Gammerman, A.; Shafer, G. *Algorithmic Learning in a Random World*; Springer: New York, 2005
  (https://doi.org/10.1007/b106715); 2nd ed., Springer: Cham, 2022 (https://doi.org/10.1007/978-3-031-06649-8).
  Mondrian conformal predictors are introduced in this book.
* Vovk, V. Conditional Validity of Inductive Conformal Predictors. *Mach. Learn.* **2013**, *92*, 349–376.
  https://doi.org/10.1007/s10994-013-5355-6
* Norinder, U.; Carlsson, L.; Boyer, S.; Eklund, M. Introducing Conformal Prediction in Predictive Modeling. A
  Transparent and Flexible Alternative to Applicability Domain Determination. *J. Chem. Inf. Model.* **2014**, *54*,
  1596–1603. https://doi.org/10.1021/ci5001168
* Gibbs, I.; Cherian, J. J.; Candès, E. J. Conformal Prediction with Conditional Guarantees. *J. R. Stat. Soc. Ser. B:
  Stat. Methodol.* **2025**, *87*, 1100–1126. https://doi.org/10.1093/jrsssb/qkaf008 (guarantees for overlapping groups
  and beyond)
* Gao, C. W.; Allen, J. W.; Green, W. H.; West, R. H. Reaction Mechanism Generator: Automatic Construction of Chemical
  Kinetic Mechanisms. *Comput. Phys. Commun.* **2016**, *203*, 212–225. https://doi.org/10.1016/j.cpc.2016.02.013

In [ ]:
results = {
    "single_split": {"q_global": q_global, "q_class": {P.CLASS_SHORT[g]: q_class[g] for g in P.CLASS_ORDER},
                     "coverage_global": {P.CLASS_SHORT[g]: table.loc[g, "coverage, one q for all"] for g in P.CLASS_ORDER},
                     "coverage_mondrian": {P.CLASS_SHORT[g]: table.loc[g, "coverage, Mondrian"] for g in P.CLASS_ORDER}},
    "average": {method: {P.CLASS_SHORT.get(k, k): v for k, v in row.items()} for method, row in average.to_dict("index").items()},
    "class_spread": {P.CLASS_SHORT[g]: dict(row) for g, row in spread.to_dict("index").items()},
    "budget": {str(n): dict(row) for n, row in budget.to_dict("index").items()},
    "families": {"n_finite": int(np.isfinite(fam_table["half-width (kcal/mol)"]).sum()), "n_families_cal": int((fam_table["calibration reactions"] > 0).sum()),
                 "unseen_in_cal": len(unseen), "kept_after_merging": sorted(keep)},
    "merged": {g: dict(row) for g, row in merged_table.to_dict("index").items()},
    "combined": {method: dict(row) for method, row in combined.to_dict("index").items()},
}
P.save_results("nb4", results, rdb7)
P.check("nb4", results);

# Notebook 5 · When the guarantee breaks: exchangeability in practice

**Conformal Prediction for Physical Chemists: hands-on notebooks (5 of 6)**

Every guarantee in Notebooks 2 to 4 rests on one assumption: the calibration reactions and the reactions we want to
predict are *exchangeable*. So far we have ensured this by drawing both at random from the same pool. In real work the
calibration set is whatever has been computed already, and tomorrow's reactions are chosen by a chemist with a
different question. This notebook breaks the assumption in controlled ways, measures the damage, and shows what can be
repaired and what cannot.

**You will**

1. see how the coverage of "90%" intervals changes when the deployment set differs from the calibration set:
   in its mix of reaction classes, in molecular size, and by a reaction class the model has never seen;
2. repair the first kind of shift with *weighted* conformal prediction and with Mondrian calibration;
3. see why no reweighting can repair the other two, and recalibrate with a few reference calculations from the new
   domain instead;
4. measure whether hidden dependence in the data (several reactions of the same reactant) matters;
5. learn two warning signs: a classifier that can tell calibration from deployment inputs, and a coverage alarm.

Runs on a CPU in four to five minutes (it refits the models many times).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

P.set_style()
rdb7 = P.load_problem()
y = rdb7.y
alpha = 0.10
classes = rdb7.rxn_class
conformal_quantile = P.conformal_quantile

model = rdb7.models["delta"]                       # the Δ-learning model is again the running example
abs_err = {name: np.abs(y - mod.mu) for name, mod in rdb7.models.items()}
scenario_coverage = {}                             # collected for the summary figure: {scenario: {model: coverage}}

## 1. What exchangeability asks for

Exchangeability is a statement about **how the calibration set relates to the cases you will predict**, not about the
model. In practical terms it asks one question:

> *Could my calibration reactions and my future reactions have been drawn from the same hat?*

Three things can go wrong, and they are worth keeping apart.

* **The inputs change** (covariate shift): tomorrow's reactions are larger, belong to other classes, or come in other
  proportions, while the relation between a reaction and its barrier stays the same.
* **The relation changes**: the reference level of theory, the solvent model, or the experimental protocol is
  different. The same reaction now has a different reference value.
* **The data are not independent draws**: conformers of one molecule, forward and reverse reactions, or many reactions
  of the same reactant sit on both sides of a split.

We start with the mildest case.

## 2. A different mix of reaction classes

Suppose the calibration set is a random sample of RDB7-like chemistry, but the next project is about skeletal
rearrangements: 60% of the reactions to be predicted are of this class, against roughly one in seven in the
calibration set. Nothing about any individual reaction has changed; only the mix has.

In [ ]:
HARD = "skeletal rearrangement"


def deployment_set(pool, rng, n_deploy=600, share=0.60):
    """n_deploy reactions from `pool`, a fraction `share` of them skeletal rearrangements (these come first)."""
    hard, rest = pool[classes[pool] == HARD], pool[classes[pool] != HARD]
    n_hard = int(round(share * n_deploy))
    return np.concatenate([rng.choice(hard, n_hard, replace=False), rng.choice(rest, n_deploy - n_hard, replace=False)])


rng = np.random.default_rng(P.SEED)
shuffled = rng.permutation(model.hold)
cal, pool = shuffled[:1500], shuffled[1500:]
deploy = deployment_set(pool, rng)

err = abs_err["delta"]
q_plain = conformal_quantile(err[cal], alpha)
print(f"share of skeletal rearrangements: calibration {np.mean(classes[cal] == HARD):.0%}, deployment {np.mean(classes[deploy] == HARD):.0%}")
print(f"split conformal, q = {q_plain:.2f} kcal/mol: coverage on the deployment set {np.mean(err[deploy] <= q_plain):.1%}")

The "90%" interval covers clearly less than 90% of the reactions in the new project, because the class with the
largest errors is now over-represented. Two repairs are available.

### Repair 1: weighted conformal prediction

If we knew how much more (or less) likely each kind of reaction is in the deployment set than in the calibration set,
the **likelihood ratio**

$$ w(x) = \frac{p_\text{deployment}(x)}{p_\text{calibration}(x)} , $$

we could let every calibration score count with weight $w(x_i)$: scores of reactions that are common in the new
project count more. The conformal quantile becomes a *weighted* quantile, and the test point itself contributes its
own weight at $+\infty$ (the weighted counterpart of the "$n+1$" in Notebook 2). Tibshirani, Barber, Candès, and Ramdas
(2019) showed that with the true $w$ this restores the guarantee under covariate shift.

In [ ]:
def weighted_conformal_quantile(scores, w_cal, w_test, alpha=0.10):
    """Weighted conformal quantile, one value per test point."""
    order = np.argsort(scores)
    cum = np.cumsum(w_cal[order])                       # weight at or below each sorted score
    need = (1 - alpha) * (cum[-1] + w_test)             # (1 - alpha) of the total weight, test point included
    idx = np.searchsorted(cum, need * (1 - 1e-9), side="left")      # the factor guards against floating-point round-up
    return np.append(scores[order], np.inf)[idx]        # infinite if the calibration weight does not suffice


def class_ratio_weights(classes_cal, classes_deploy):
    """Weights when only the class proportions differ: share in the deployment sample / share in the calibration sample."""
    share_cal = pd.Series(classes_cal).value_counts(normalize=True)
    share_dep = pd.Series(classes_deploy).value_counts(normalize=True).reindex(share_cal.index).fillna(0.0)
    ratio = (share_dep / share_cal).to_dict()            # 0 for a class that does not occur in the deployment sample
    return np.array([ratio[g] for g in classes_cal]), np.array([ratio[g] for g in classes_deploy]), ratio


# here the shift is entirely in the class proportions, so w(x) is a ratio of class frequencies
w_cal, w_deploy, ratio = class_ratio_weights(classes[cal], classes[deploy])
print("weights by class:", {P.CLASS_SHORT[g]: round(ratio[g], 2) for g in P.CLASS_ORDER})

q_weighted = weighted_conformal_quantile(err[cal], w_cal, w_deploy, alpha)
assert np.array_equal(q_weighted, P.weighted_conformal_quantile(err[cal], w_cal, w_deploy, alpha))
print(f"weighted conformal: coverage {np.mean(err[deploy] <= q_weighted):.1%}, mean half-width {q_weighted.mean():.2f} kcal/mol")

In real life nobody hands us $w(x)$, and the shift is rarely a clean change of class frequencies. The standard trick is
to **learn $w$ from the inputs alone** (Bickel, Brückner, and Scheffer, 2009): train a classifier to tell calibration
reactions from deployment reactions using their features (no barriers needed), and convert its probability $\pi(x)$ of
"deployment" into

$$ \hat w(x) = \frac{\pi(x)}{1-\pi(x)}\cdot\frac{n_\text{calibration}}{n_\text{deployment}} . $$

With an estimated $\hat w$ the guarantee holds only approximately, to the extent that the estimate is accurate. The
same classifier gives two diagnostics. Its **AUC** (the area under its receiver operating characteristic curve) says
whether the two sets can be told apart at all (0.5: not at all; 1: perfectly). The **effective sample size**
$(\sum w_i)^2/\sum w_i^2$ is a rule of thumb for how many equally weighted calibration reactions the weighted set is
worth.

In [ ]:
def estimate_weights(X_cal, X_deploy, seed=P.SEED, clip=0.01):
    """Likelihood-ratio weights from a classifier 'calibration or deployment?'. Returns (w_cal, w_deploy, AUC)."""
    X = np.vstack([X_cal, X_deploy])
    is_deploy = np.r_[np.zeros(len(X_cal)), np.ones(len(X_deploy))]
    classifier = HistGradientBoostingClassifier(max_iter=100, max_depth=3, random_state=seed)
    folds = StratifiedKFold(5, shuffle=True, random_state=seed)        # shuffled: the rows may come in a systematic order
    prob = cross_val_predict(classifier, X, is_deploy, cv=folds, method="predict_proba")[:, 1]    # out-of-fold probabilities
    auc = roc_auc_score(is_deploy, prob)
    prob = np.clip(prob, clip, 1 - clip)                # keeps the weights finite; see Section 3
    w = prob / (1 - prob) * len(X_cal) / len(X_deploy)
    return w[: len(X_cal)], w[len(X_cal):], auc


w_cal_hat, w_deploy_hat, auc = estimate_weights(model.X[cal], model.X[deploy])
q_learned = weighted_conformal_quantile(err[cal], w_cal_hat, w_deploy_hat, alpha)
print(f"classifier AUC {auc:.2f}; effective calibration sample size {P.effective_sample_size(w_cal_hat):.0f} of {len(cal)}")
print(f"weighted conformal with learned weights: coverage {np.mean(err[deploy] <= q_learned):.1%}")
print(f"(one calibration set and one deployment set of {len(deploy)} reactions: such coverages scatter by several percentage points; "
      "see the standard deviations in the next table)")

### Repair 2: Mondrian calibration

Notebook 4 already contains the second repair. If every class is calibrated on its own, the class proportions drop
out of the argument altogether: Mondrian intervals stay valid under *any* change of the mix, with no weights to
estimate. The comparison below repeats the whole experiment 200 times with new calibration and deployment sets (the
learned weights, which need a classifier each time, in the first 40 repetitions). The standard deviation says how
much the coverage of one such pair of sets scatters around the mean.

In [ ]:
rows, diagnostics = [], []
rng = np.random.default_rng(31)
for rep in range(200):
    shuffled = rng.permutation(model.hold)
    cal_b, pool_b = shuffled[:1500], shuffled[1500:]
    dep_b = deployment_set(pool_b, rng)
    w_c, w_d, _ = class_ratio_weights(classes[cal_b], classes[dep_b])
    learned = estimate_weights(model.X[cal_b], model.X[dep_b], seed=rep) if rep < 40 else None
    if learned is not None:
        diagnostics.append({"AUC": learned[2], "effective sample size": P.effective_sample_size(learned[0])})
    for name in ("delta", "structure"):
        e = abs_err[name]
        half = {"split conformal (no repair)": np.full(len(dep_b), conformal_quantile(e[cal_b], alpha)),
                "weighted, class-frequency ratios": weighted_conformal_quantile(e[cal_b], w_c, w_d, alpha)}
        if name == "delta" and learned is not None:           # the learned weights use the features of the Δ-learning model
            half["weighted, learned weights"] = weighted_conformal_quantile(e[cal_b], learned[0], learned[1], alpha)
        qm = P.mondrian_quantiles(e[cal_b], classes[cal_b], alpha)
        half["Mondrian by class"] = np.array([qm[g] for g in classes[dep_b]])
        for method, h in half.items():
            rows.append({"model": rdb7.models[name].label, "method": method, "coverage": np.mean(e[dep_b] <= h), "mean half-width": h.mean()})
mix = pd.DataFrame(rows).groupby(["model", "method"], sort=False).agg(**{
    "coverage": ("coverage", "mean"), "s.d. of coverage": ("coverage", "std"), "mean half-width": ("mean half-width", "mean")})
display(mix.round(3))
diagnostics = pd.DataFrame(diagnostics).mean()
print(f"classifier AUC {diagnostics['AUC']:.2f}; effective sample size {diagnostics['effective sample size']:.0f} of 1500")
scenario_coverage["different mix of classes"] = {name: mix.loc[(rdb7.models[name].label, "split conformal (no repair)"), "coverage"] for name in rdb7.models}

The class-frequency weights and Mondrian calibration bring the coverage back to 90%. So do the learned weights, within
the noise of their 40 repetitions (the standard error of that mean is about 0.3 percentage points), although the
classifier was not given the class label (it can reconstruct the class from the descriptors). For learned weights this
is an observation and not a guarantee: how close the coverage comes depends on how well the classifier estimates $w$.
All repairs pay in the same way: the intervals are wider, because the new project really is harder. Note the effective
sample size: 1500 calibration reactions are worth only a few hundred once they are reweighted, so the coverage obtained
from one calibration set is less certain than the Beta law of Notebook 2 says for $n = 1500$. The table shows this in
the standard deviations: those of the repaired rows are larger than that of the unrepaired row. (All of them also
contain the sampling noise of the 600 deployment reactions, about 1.2 to 1.5 percentage points, which the Beta law
leaves out; the effective sample size is only a rough guide.)

## 3. Outside the calibration domain: larger molecules

Now a shift of a different kind. The models are fitted and calibrated on reactions of molecules with **at most six**
heavy atoms and then used for molecules with **seven** (in a real project: the data set was built on small molecules,
and the interesting chemistry is one size up). The models have to be refitted for this experiment; as always, no
reactant is shared between the fitting set and the calibration set.

In [ ]:
small, large = np.flatnonzero(rdb7.n_heavy <= 6), np.flatnonzero(rdb7.n_heavy == 7)
first, second = P.group_split(rdb7.reactant[small], 2 / 3, seed=P.SEED)
fit_small, cal_small = small[first], small[second]
print(f"fitting set {len(fit_small)}, calibration set {len(cal_small)} (both up to 6 heavy atoms); deployment set {len(large)} (7 heavy atoms)")

size_models, rows = {}, []
for name in rdb7.models:
    size_models[name] = mod = P.fit_model(rdb7, name, fit_small)
    e = np.abs(y - mod.mu)
    q = conformal_quantile(e[cal_small], alpha)
    rows.append({"model": mod.label, "MAE, calibration set": e[cal_small].mean(), "MAE, larger molecules": e[large].mean(),
                 "half-width used": q, "coverage on larger molecules": np.mean(e[large] <= q),
                 "half-width needed for 90%": np.quantile(e[large], 1 - alpha)})
size_table = pd.DataFrame(rows).set_index("model")
display(size_table.round(3))
scenario_coverage["larger molecules"] = {name: size_table.loc[rdb7.models[name].label, "coverage on larger molecules"] for name in rdb7.models}

The errors on the larger molecules are somewhat bigger, and the coverage falls short of the promise by two to three
percentage points for both models. A modest loss, but a systematic one: it does not go away with a larger calibration
set. Can weighting repair it?

In [ ]:
mod_small = size_models["delta"]
err_small = np.abs(y - mod_small.mu)
subset = np.random.default_rng(P.SEED).choice(large, 2000, replace=False)
w_c, w_d, auc_size = estimate_weights(mod_small.X[cal_small], mod_small.X[subset])
q_w = weighted_conformal_quantile(err_small[cal_small], w_c, w_d, alpha)
print(f"classifier AUC: {auc_size:.3f}")
print(f"weighted conformal intervals that are infinite: {np.mean(np.isinf(q_w)):.0%}")

No. The classifier separates the two sets perfectly (the number of heavy atoms is one of its features), which means
that **no calibration reaction looks like a deployment reaction**. The estimated likelihood ratio is then enormous
for every new reaction, the calibration scores carry almost no weight in comparison, and weighted conformal
prediction returns the only interval it can still vouch for: an infinite one. That is the correct answer. Reweighting
can redistribute the evidence in a calibration set; it cannot create evidence about a region the calibration set
never visited.

(A technical remark. `estimate_weights` clips the classifier's probabilities to [0.01, 0.99]; this limit decides how
extreme a weight can become. It is a tuning constant: with a much tighter clip the weights would be too tame to signal
the problem, and the intervals would come out finite without having earned it. When the AUC is close to 1, a finite
weighted interval should not be trusted.)

### The repair that needs no overlap: reference calculations in the deployment domain

What is needed is information about the deployment domain: a (small) number $m$ of reference calculations for
reactions drawn at random from the deployment set. Recalibrating on those alone restores exchangeability, and
Notebook 2 tells us what to expect from $m$ calibration points:

In [ ]:
def recalibration_table(errors_new, sizes=(10, 20, 50, 100, 200, 500), n_rep=500, seed=32):
    """Coverage and half-width when m randomly chosen new cases are labeled and used as the calibration set."""
    rng = np.random.default_rng(seed)
    rows = []
    for m in sizes:
        cov, half = [], []
        for _ in range(n_rep):
            perm = rng.permutation(errors_new)
            q = conformal_quantile(perm[:m], alpha)
            half.append(q)
            cov.append(np.mean(perm[m:] <= q))
        rows.append({"reference calculations m": m, "mean coverage": np.mean(cov), "5%": np.quantile(cov, 0.05),
                     "95%": np.quantile(cov, 0.95), "median half-width": np.median(half)})
    return pd.DataFrame(rows).set_index("reference calculations m")


recal = recalibration_table(err_small[large])
before = size_table.loc[model.label]
print(f"Δ-learning model on the larger molecules; before recalibration: half-width {before['half-width used']:.2f} kcal/mol, "
      f"coverage {before['coverage on larger molecules']:.1%}")
display(recal.round(3))

On average the coverage is back at 90% for any $m \ge 9$. What $m$ buys is *reliability*: with 20 calculations
the coverage of a particular recalibration lies between about 78% and 98% in nine cases out of ten, with 200 between
about 86% and 93%. The price in width is visible too: small calibration sets give inflated half-widths.

Old and new calibration data can also be combined, with the new ones weighted more strongly (Barber et al., 2023;
Exercise 2 tries this); and when enough reference values from the deployment domain accumulate, the model itself
should be refitted, which goes beyond this tutorial.

## 4. A reaction class the model has never seen

The most drastic shift: an entire reaction class is missing from the fitting set **and** from the calibration set and
then turns up at deployment. We leave out each of the five classes in turn.

In [ ]:
rows, class_models = [], {}
for name in rdb7.models:
    for g in P.CLASS_ORDER:
        fit_rows = rdb7.train[classes[rdb7.train] != g]
        cal_rows = rdb7.hold[classes[rdb7.hold] != g]
        new_rows = rdb7.hold[classes[rdb7.hold] == g]
        mod = P.fit_model(rdb7, name, fit_rows)
        e = np.abs(y - mod.mu)
        q = conformal_quantile(e[cal_rows], alpha)
        after = recalibration_table(e[new_rows], sizes=(50,), n_rep=300, seed=33).iloc[0]
        rows.append({"model": mod.label, "unseen class": g, "MAE, known classes": e[cal_rows].mean(), "MAE, unseen class": e[new_rows].mean(),
                     "half-width used": q, "coverage on the unseen class": np.mean(e[new_rows] <= q),
                     "coverage after 50 reference calculations": after["mean coverage"], "half-width after": after["median half-width"]})
        class_models[(name, g)] = (e, cal_rows, new_rows, q)
        scenario_coverage.setdefault(f"unseen class: {P.CLASS_SHORT[g]}", {})[name] = rows[-1]["coverage on the unseen class"]
unseen = pd.DataFrame(rows).set_index(["model", "unseen class"])
display(unseen.round(3))

Here the damage is of a different order. Depending on the class and the model, the "90%" interval covers between
about one half and 86% of the reactions of the unseen class. The errors on an unseen class are up to almost two and a
half times larger than on the classes the model knows, and the calibration set, which does not contain the class either,
has no way of knowing. Fifty reference calculations from the unseen class restore the coverage, at the price of
intervals that are honest about the model's poor grasp of that chemistry.

## 5. Hidden dependence: several reactions per reactant

RDB7 contains on average ten reactions for each reactant molecule. Throughout these notebooks the split into fitting
and held-out data was therefore made **by reactant**. Was that necessary? The question can be put to the data. In each
of 60 repetitions the reactants are divided at random into "known" and "new" ones. The models are fitted to 80% of the
reactions of the known reactants, and the interval is calibrated on the remaining 20%: reactions the model has not
seen, of reactants it has seen. The interval is then tested on all reactions of the new reactants. Had we calibrated on
reactions of new reactants instead, the coverage would be $k/(n+1)$ on average (Notebook 2); the shortfall from that
value is the price of calibrating on known reactants.

In [ ]:
rows = []
for rep in range(60):
    known, new = P.group_split(rdb7.reactant, 0.5, seed=100 + rep)             # a fresh division of the reactants
    known = np.random.default_rng(200 + rep).permutation(known)
    n_fit = int(0.8 * len(known))
    fit_rows, cal_known = known[:n_fit], known[n_fit:]                         # known reactants: 80% of the reactions to fit, 20% to calibrate
    k = int(np.ceil((len(cal_known) + 1) * (1 - alpha) - 1e-9))
    for name in rdb7.models:
        e = np.abs(y - P.fit_model(rdb7, name, fit_rows).mu)
        q_known = conformal_quantile(e[cal_known], alpha)
        rows.append({"model": rdb7.models[name].label, "coverage": np.mean(e[new] <= q_known), "expected": k / (len(cal_known) + 1),
                     "width_ratio": q_known / np.quantile(e[new], 1 - alpha)})
runs = pd.DataFrame(rows)
runs["loss"] = 100 * (runs["expected"] - runs["coverage"])                     # percentage points of coverage lost
dependence = runs.groupby("model", sort=False).agg(
    coverage=("coverage", "mean"), expected=("expected", "mean"), loss=("loss", "mean"), loss_se=("loss", "sem"),
    width_ratio=("width_ratio", "mean"))
display(dependence.rename(columns={"coverage": "coverage on new reactants", "expected": "expected without dependence",
                                   "loss": "loss (percentage points)", "loss_se": "standard error of the loss",
                                   "width_ratio": "half-width used / half-width needed"}).round(3))
for name in rdb7.models:
    scenario_coverage.setdefault("calibration reactants seen in fitting", {})[name] = dependence.loc[rdb7.models[name].label, "coverage"]

In RDB7 the loss is small but systematic: calibrating on other reactions of *known* reactants lowers the coverage on
new reactants by about half a percentage point for either model (0.5 to 0.6 points, more than three times the standard
error of the mean over the 60 repetitions), and the half-width is about 2% too small. Unlike the noise of a single
calibration set, such a loss does not average out and does not shrink with more calibration data. (The experiment needs
its 60 repetitions: a single repetition scatters by about one percentage point, twice the size of the effect.)

Do not generalize the small size of the effect. In data sets with near-duplicates (conformers of one molecule, the
same reaction in several solvents, forward and reverse reactions) the same experiment can reveal a large gap, and
Notebook 2 showed the limiting case: calibrating on the fitting set itself gave about three quarters instead of 90%.
The experiment costs some extra fits; run it instead of assuming.

Clusters have a second effect even when the split is done correctly (Dunn, Wasserman, and Ramdas, 2023, treat such
two-layer data in general). A calibration set of about 1200 reactions that come from only 113 reactants carries less
information than 1200 unrelated reactions. In this experiment the coverage is still 90% on average, but it varies more
from one calibration set to the next than for the same number of unrelated reactions:

In [ ]:
rows = []
for name, mod in rdb7.models.items():
    e, hold, groups = abs_err[name], rdb7.hold, rdb7.reactant[rdb7.hold]
    unique = np.unique(groups)
    rng = np.random.default_rng(36)
    by_reactant, by_reaction = [], []
    for _ in range(1000):
        chosen = np.isin(groups, rng.permutation(unique)[: len(unique) // 5])      # one fifth of the reactants ...
        by_reactant.append(np.mean(e[hold[~chosen]] <= conformal_quantile(e[hold[chosen]], alpha)))
        perm = rng.permutation(hold)                                               # ... versus as many single reactions
        by_reaction.append(np.mean(e[perm[chosen.sum():]] <= conformal_quantile(e[perm[: chosen.sum()]], alpha)))
    rows.append({"model": mod.label, "mean coverage, whole reactants": np.mean(by_reactant), "standard deviation, whole reactants": np.std(by_reactant),
                 "mean coverage, single reactions": np.mean(by_reaction), "standard deviation, single reactions": np.std(by_reaction)})
clusters = pd.DataFrame(rows).set_index("model")
display(clusters.round(4))
ratio_sd = (clusters["standard deviation, whole reactants"] / clusters["standard deviation, single reactions"]).mean()
print(f"calibrating on whole reactants: the coverage scatters {ratio_sd - 1:.0%} more than for the same number of unrelated reactions")

## 6. Two warning signs

In an experiment we know what we broke. In a project we do not, so we need symptoms.

**Before any reference value from the deployment set exists: can a classifier tell the two sets apart?** The AUC of the
calibration-versus-deployment classifier of Section 2 (a "domain classifier") needs only inputs. Collected for the
scenarios above:

In [ ]:
rng = np.random.default_rng(37)
shuffled = rng.permutation(model.hold)
same_population = estimate_weights(model.X[shuffled[:1500]], model.X[shuffled[1500:2100]])[2]
e_cls, cal_cls, new_cls, q_cls = class_models[("delta", HARD)]
auc_class = estimate_weights(model.X[rng.choice(cal_cls, 1500, replace=False)], model.X[rng.choice(new_cls, 600, replace=False)])[2]

# coverage without any shift, measured in the same way as for the other scenarios
scenario_coverage["same population"] = {
    name: float(np.mean([np.mean(abs_err[name][t] <= conformal_quantile(abs_err[name][c], alpha)) for c, t in P.resplits(rdb7.hold, 1500, 200, seed=40)]))
    for name in rdb7.models}

auc_table = pd.DataFrame({"AUC of the domain classifier": {
    "same population (no shift)": same_population, "different mix of classes": diagnostics["AUC"],
    "larger molecules": auc_size, "unseen class (skeletal)": auc_class}})
auc_table["coverage of the Δ-learning intervals"] = [scenario_coverage[s]["delta"] for s in
                                                    ("same population", "different mix of classes", "larger molecules", "unseen class: skeletal")]
display(auc_table.round(3))

As a rule of thumb: an AUC near 0.5, a little above or below it, means that the classifier found nothing and the inputs
give no reason for concern. An AUC between 0.5 and about 0.9 means that the sets differ but overlap: weighting has
something to work with. An AUC close to 1 means that the deployment reactions are recognizably *not* from the
calibration population, and the intervals should not be trusted until reference values from the deployment set are
available. Two cautions. The AUC measures whether the inputs differ, not whether the difference matters for the errors
(the two shifts with an AUC of 1 above cost very different amounts of coverage). And it is blind to a change in the
relation between inputs and outcome.

**With a few reference values from the deployment set: a coverage alarm.** Decide in advance on a number $m$, draw $m$
deployment reactions **at random**, compute them at the reference level, and count how many fall outside their
intervals. If nothing has changed, the number of misses among $m$ cases follows the Beta-binomial law of Notebook 2,
so we know how many misses are "too many":

In [ ]:
def alarm_threshold(n_cal, m, alpha=0.10, level=0.05):
    """Smallest number t such that t or more misses among m new cases occur with probability <= level if nothing has changed."""
    k = int(np.ceil((n_cal + 1) * (1 - alpha) - 1e-9))
    return int(stats.betabinom(m, n_cal + 1 - k, k).isf(level)) + 1


sizes = (10, 20, 50, 100, 200, 500)
print("misses that trigger the alarm (calibration set of 1000, 90% intervals):",
      {m: alarm_threshold(1000, m) for m in sizes})


def alarm_rate(errors, draw_new, m, n_cal=1000, n_rep=2000, seed=38):
    """How often does the alarm ring? Each repetition draws a calibration set of n_cal from `errors` and m new cases.
    `draw_new(rng, rest, m)` returns the errors of the new cases; `rest` are the positions not used for calibration."""
    rng = np.random.default_rng(seed)
    threshold, rings = alarm_threshold(n_cal, m), 0
    for _ in range(n_rep):
        perm = rng.permutation(len(errors))
        q = conformal_quantile(errors[perm[:n_cal]], alpha)
        rings += np.sum(draw_new(rng, perm[n_cal:], m) > q) >= threshold
    return rings / n_rep


hold_err, hold_is_hard = abs_err["delta"][rdb7.hold], classes[rdb7.hold] == HARD


def new_cases_no_shift(rng, rest, m):                 # m of the held-out reactions that are not in the calibration set
    return hold_err[rest[:m]]


def new_cases_other_mix(rng, rest, m):                # the same, but 60% of them skeletal rearrangements
    n_hard = int(round(0.6 * m))
    return np.r_[hold_err[rng.choice(rest[hold_is_hard[rest]], n_hard, replace=False)],
                 hold_err[rng.choice(rest[~hold_is_hard[rest]], m - n_hard, replace=False)]]


def new_cases_larger(rng, rest, m):                   # molecules with seven heavy atoms (model of Section 3)
    return rng.choice(err_small[large], m, replace=False)


def new_cases_unseen_class(rng, rest, m):             # the class that was left out (model of Section 4)
    return rng.choice(e_cls[new_cls], m, replace=False)


scenarios = {"same population (no shift)": (hold_err, new_cases_no_shift),
             "different mix of classes": (hold_err, new_cases_other_mix),
             "larger molecules": (err_small[cal_small], new_cases_larger),
             "unseen class (skeletal)": (e_cls[cal_cls], new_cases_unseen_class)}
alarm = pd.DataFrame({label: {m: alarm_rate(errors, draw, m) for m in sizes} for label, (errors, draw) in scenarios.items()}).T
alarm.columns.name = "reference calculations m"
# the exact probability of a false alarm, from the Beta-binomial law (k = 901 for n = 1000 and alpha = 0.10)
alarm_exact = {m: float(stats.betabinom(m, 1001 - 901, 901).sf(alarm_threshold(1000, m) - 1)) for m in sizes}
print("Probability that the alarm rings (first row: exact; the others: estimated from 2000 simulated cases each)")
display(pd.concat([pd.DataFrame({"no shift, exact probability": alarm_exact}).T, alarm]).round(3))

# the same alarm when the count is inspected after each of the six values of m instead of once
rng = np.random.default_rng(39)
thresholds = [alarm_threshold(1000, m) for m in sizes]
rings = 0
for _ in range(2000):
    perm = rng.permutation(hold_err)
    misses = perm[1000:1000 + max(sizes)] > conformal_quantile(perm[:1000], alpha)
    rings += any(misses[:m].sum() >= t for m, t in zip(sizes, thresholds))
repeated_looks = rings / 2000
print(f"no shift, but the count is inspected at all six values of m: the alarm rings in {repeated_looks:.0%} of the cases")

Without a shift the alarm rings in at most 5% of the cases, as designed (first row; the simulated values in the second
row scatter around it). A drastic shift (the unseen class) is caught with 20 to 50 reference calculations. A loss of a
few percentage points (the larger molecules) is still missed almost half of the time with 500: small violations are
hard to detect for the same reason that small calibration sets give noisy coverage. An alarm that stays silent after
20 calculations is therefore weak evidence that all is well.

The 5% level holds for **one look at a number $m$ fixed in advance**. If the count is inspected again and again while the
results arrive, false alarms become more frequent (last line of the output); in that case divide the 5% among the
planned looks, or use a sequential test.

In [ ]:
fig, axes = P.panels(3, height=2.9, gridspec_kw={"width_ratios": [1.25, 1, 1]})

# a: coverage of nominal 90% intervals in every scenario, both models
ax = axes[0]
order = ["same population", "calibration reactants seen in fitting", "different mix of classes", "larger molecules"] + \
        [f"unseen class: {P.CLASS_SHORT[g]}" for g in ("carbene product", "zwitterionic product", "fragmentation", "H-transfer isomerization", "skeletal rearrangement")]
labels = ["same population", "reactants seen in fitting", "different class mix", "larger molecules",
          "unseen: carbene", "unseen: zwitterion", "unseen: fragmentation", "unseen: H transfer", "unseen: skeletal"]
ypos = np.arange(len(order))[::-1]
ax.axvline(1 - alpha, color=P.GRAY, lw=0.8, ls="--")
for name, shift in (("structure", 0.14), ("delta", -0.14)):            # the two markers of a row are drawn slightly apart
    ax.plot([scenario_coverage[s][name] for s in order], ypos + shift, ls="none", marker=P.MODEL_MARKERS[name], color=P.MODEL_COLORS[name],
            label=rdb7.models[name].label.replace(" model", ""))
ax.set_yticks(ypos, labels)
ax.set(xlabel="coverage of 90% intervals", xlim=(0.4, 1.0), ylim=(-0.7, len(order) - 0.3))
ax.set_xticks([0.4, 0.6, 0.8, 1.0])
ax.legend(loc="upper left", handletextpad=0.2, borderaxespad=0.2)
P.panel_label(ax, "a", "What a shift costs")

# b: recalibration with m reference calculations from the deployment set (larger molecules)
ax = axes[1]
ax.fill_between(recal.index, recal["5%"], recal["95%"], color=P.SKY, alpha=0.35, lw=0)
ax.plot(recal.index, recal["mean coverage"], marker="o", color=P.BLUE)
ax.axhline(1 - alpha, color=P.GRAY, lw=0.8, ls="--")
ax.axhline(before["coverage on larger molecules"], color=P.VERMILLION, lw=1.2, ls=":")
ax.text(480, before["coverage on larger molecules"] - 0.01, "before recalibration", fontsize=6.5, va="top", ha="right")
ax.text(13, 0.955, "90% of all\nrecalibrations", fontsize=6.5, va="center")
ax.text(480, 0.912, "mean", fontsize=6.5, va="bottom", ha="right")
ax.set_xscale("log")
ax.set_xticks(list(recal.index), [str(m) for m in recal.index])
ax.minorticks_off()
ax.set(xlabel="reference calculations $m$", ylabel="coverage on larger molecules", ylim=(0.68, 1.0))
P.panel_label(ax, "b", "Recalibration")

# c: how fast does the coverage alarm ring?
ax = axes[2]
alarm_styles = {"unseen class (skeletal)": (P.VERMILLION, "o", "unseen class"), "different mix of classes": (P.ORANGE, "s", "class mix"),
                "larger molecules": (P.GREEN, "^", "larger"), "same population (no shift)": (P.GRAY, "D", "no shift")}
for label, (color, marker, short) in alarm_styles.items():
    ax.plot(sizes, alarm.loc[label], marker=marker, color=color, label=short)
ax.set_xscale("log")
ax.set_xticks(list(sizes), [str(m) for m in sizes])
ax.minorticks_off()
ax.set(xlabel="reference calculations $m$", ylabel="probability that the alarm rings", ylim=(-0.03, 1.38))
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.legend(loc="upper center", ncol=2, columnspacing=1.0, handletextpad=0.4, borderaxespad=0.2)
P.panel_label(ax, "c", "Coverage alarm")
P.savefig(fig, "nb5_exchangeability")
plt.show()

Panel **a**: coverage of nominal 90% split conformal intervals in every scenario of this notebook, for both models
(the axis starts at 0.4). Panel **b**: recalibrating on $m$ reference calculations from the deployment set puts the
coverage back at 90% on average; $m$ decides how far a single recalibration can be off (band: the range of 90% of all
recalibrations; the axis starts at 0.68). Panel **c**: the probability that the coverage alarm rings after $m$ reference
calculations.

## 7. What to do

| What changed between calibration and use | How it shows | What to do |
|---|---|---|
| nothing: the calibration set is a random sample of what will be predicted | classifier AUC ≈ 0.5 | use the intervals as they are |
| the proportions of known categories | category frequencies differ | Mondrian calibration, or weights from the frequency ratios |
| the distribution of the inputs, with overlap | AUC between 0.5 and about 0.9; effective sample size still large | weighted conformal prediction with learned weights |
| inputs outside the calibration domain (other sizes, other classes) | AUC ≈ 1; weighted intervals become infinite | do not trust the intervals; compute reference values for a random sample of the deployment cases and recalibrate; refit when possible |
| the relation between input and outcome (another level of theory, other conditions) | invisible in the inputs; only the coverage alarm shows it | recalibrate on data from the deployment setting |
| dependence within the data (clusters, near-duplicates) | known from how the data were generated | split by cluster; expect noisier coverage |

For data that arrive as a stream and drift slowly (a growing database, an active-learning loop), the calibration set
can be updated continuously; "adaptive conformal inference" (Gibbs and Candès, 2021) is the keyword.

## Take-aways

* The guarantee is about the pair "calibration set and deployment set". Ask whether they could have been drawn from
  the same hat.
* A changed mix of known categories is harmless with Mondrian calibration and repairable with weights.
* Reweighting cannot extrapolate. For other sizes and other classes of chemistry the honest interval is infinite until
  reference values from the deployment domain exist; a few dozen of them restore validity.
* Split by cluster when the data have clusters, and measure whether it matters.
* Monitor: a domain classifier before reference values exist, a coverage alarm once they arrive.

## Try it yourself

1. In Section 2, raise the share of skeletal rearrangements to 80%. What happens to the effective sample size and to
   the width of the weighted intervals?
2. In Section 3, calibrate on a *mixture* of the old calibration set and 50 reactions with seven heavy atoms, giving
   each of the 50 the weight of 20 old ones and the test point the same weight as one of the 50 (use
   `weighted_conformal_quantile`). Compare with recalibrating on the 50 alone.
3. Apply the coverage alarm to the "different mix" scenario after Mondrian calibration. Does it still ring?
4. In Section 5, raise the number of repetitions from 60 to 200 (about ten minutes). Does a difference between the two
   models emerge? Which model would you expect to depend more on having seen the reactant, and why?

## References

* Tibshirani, R. J.; Barber, R. F.; Candès, E. J.; Ramdas, A. Conformal Prediction Under Covariate Shift. In *Advances
  in Neural Information Processing Systems 32*; Curran Associates, Inc., 2019; pp 2530–2540.
  https://proceedings.neurips.cc/paper/2019/hash/8fb21ee7a2207526da55a679f0332de2-Abstract.html
* Bickel, S.; Brückner, M.; Scheffer, T. Discriminative Learning Under Covariate Shift. *J. Mach. Learn. Res.* **2009**,
  *10* (75), 2137–2155. https://jmlr.org/papers/v10/bickel09a.html (likelihood ratios from a classifier)
* Barber, R. F.; Candès, E. J.; Ramdas, A.; Tibshirani, R. J. Conformal Prediction Beyond Exchangeability. *Ann. Stat.*
  **2023**, *51*, 816–845. https://doi.org/10.1214/23-AOS2276
* Dunn, R.; Wasserman, L.; Ramdas, A. Distribution-Free Prediction Sets for Two-Layer Hierarchical Models. *J. Am.
  Stat. Assoc.* **2023**, *118*, 2491–2502. https://doi.org/10.1080/01621459.2022.2060112 (clustered data)
* Gibbs, I.; Candès, E. J. Adaptive Conformal Inference Under Distribution Shift. In *Advances in Neural Information
  Processing Systems 34*; Curran Associates, Inc., 2021; pp 1660–1672.
  https://proceedings.neurips.cc/paper/2021/hash/0d441de75945e5acbc865406fc9a2559-Abstract.html

In [ ]:
results = {
    "mix": {f"{model_label} | {method}": dict(row) for (model_label, method), row in mix.to_dict("index").items()},
    "mix_diagnostics": dict(diagnostics),
    "single_example": {"q_plain": q_plain, "coverage_plain": float(np.mean(err[deploy] <= q_plain)),
                       "coverage_weighted": float(np.mean(err[deploy] <= q_weighted)), "coverage_learned": float(np.mean(err[deploy] <= q_learned)),
                       "share_hard_cal": float(np.mean(classes[cal] == HARD))},
    "size": {label: dict(row) for label, row in size_table.to_dict("index").items()},
    "size_auc": auc_size, "size_weighted_infinite": float(np.mean(np.isinf(q_w))),
    "recalibration": {str(m): dict(row) for m, row in recal.to_dict("index").items()},
    "unseen_class": {f"{model_label} | {P.CLASS_SHORT[g]}": dict(row) for (model_label, g), row in unseen.to_dict("index").items()},
    "dependence": {label: dict(row) for label, row in dependence.to_dict("index").items()},
    "clusters": {label: dict(row) for label, row in clusters.to_dict("index").items()},
    "clusters_extra_scatter": float(ratio_sd - 1),
    "same_population": scenario_coverage["same population"],
    "auc": {label: dict(row) for label, row in auc_table.to_dict("index").items()},
    "alarm_thresholds": {str(m): alarm_threshold(1000, m) for m in sizes},
    "alarm": {label: {str(m): v for m, v in row.items()} for label, row in alarm.to_dict("index").items()},
    "alarm_exact": {str(m): v for m, v in alarm_exact.items()},
    "alarm_repeated_looks": repeated_looks,
}
P.save_results("nb5", results, rdb7)
P.check("nb5", results);

# Notebook 6 · From intervals to decisions: selecting reactions with a guaranteed error rate

**Conformal Prediction for Physical Chemists: hands-on notebooks (6 of 6)**

A prediction interval answers the question of where the barrier of *this* reaction lies. Screening asks a different
question: *which* of these several thousand reactions have a barrier below some threshold? The output is a
shortlist, and the natural quality measure of a shortlist is the fraction of its entries that should not be on it.
This notebook shows how the calibration set of Notebook 2 can be used to produce shortlists whose expected fraction
of false entries is at most a level $q$ chosen by the user.

**You will**

1. see three intuitive selection rules and why none of them controls the fraction of false discoveries;
2. turn a calibration set into *conformal p-values*, one for each candidate reaction;
3. apply the Benjamini–Hochberg procedure and obtain a shortlist with a false discovery rate of at most $q$;
4. learn what determines how many true hits are found, and how large the calibration set has to be.

A word on notation: $q$ is the usual symbol for the target false discovery rate. It has nothing to do with the
conformal quantile $\hat q$ of the earlier notebooks. The letter $k$ stands for a rate constant in Section 1; the rank of
a candidate in the Benjamini–Hochberg step is called $r$.

Runs on a CPU in under a minute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P.set_style()
rdb7 = P.load_problem()
y = rdb7.y
conformal_quantile = P.conformal_quantile
sigma_of = {name: P.fit_difficulty(rdb7, name) for name in rdb7.models}      # difficulty estimates of Notebook 3

## 1. The screening task

We call a reaction a **hit** if its reference barrier is below a threshold $c$, and take $c = 50$ kcal/mol as an
illustration. With the Eyring estimate $k \approx (k_\mathrm{B}T/h)\,e^{-E/RT}$ (with the barrier height $E$ in place
of the Gibbs energy of activation, an approximation that is good to within one or two orders of magnitude for the rate
constant), a unimolecular reaction with a 50 kcal/mol barrier has a half-life of a few milliseconds at 1000 K, a
typical shock-tube time scale. The threshold is a choice: at higher temperatures or longer residence times a higher one
is appropriate (Exercise 1). In RDB7, reactions below 50 kcal/mol are rare.

In [ ]:
c = 50.0                                   # kcal/mol: a "hit" has a reference barrier below c

rate = P.KB_OVER_H * 1000 * np.exp(-c / (P.R_KCAL * 1000))
print(f"Eyring estimate for a {c:.0f} kcal/mol barrier at 1000 K: k = {rate:.3g} s⁻¹, half-life {1000 * np.log(2) / rate:.1f} ms")
print(f"hits among the held-out reactions: {np.mean(y[rdb7.hold] < c):.1%}")

# one calibration/test split, as in the earlier notebooks; the test reactions play the role of the candidates
shuffled = np.random.default_rng(P.SEED).permutation(rdb7.hold)
cal, test = shuffled[:1000], shuffled[1000:]
is_hit = y[test] < c
print(f"candidates: {len(test)}, of which {is_hit.sum()} are hits (unknown to the person who screens)")

Two numbers describe a shortlist:

* the **false discovery proportion** (FDP): the fraction of selected reactions that are not hits;
* the **power**: the fraction of all hits that were selected.

## 2. Three tempting rules

**Rule A: trust the point prediction.** Select every reaction whose *predicted* barrier is below $c$.

**Rule B: use the interval.** Select a reaction if its whole 90% conformal interval lies below $c$.

**Rule C: test every reaction at the 10% level.** For each candidate, ask how surprising its prediction would be if it
were not a hit, and select it if the answer is "would happen in fewer than 10% of such cases". (The p-value used for
this is constructed in Section 3.)

In [ ]:
def report(selected, is_hit):
    """Size of a shortlist, its false discovery proportion and its power."""
    n_sel = int(selected.sum())
    false = int((selected & ~is_hit).sum())
    found = int((selected & is_hit).sum())
    return {"selected": n_sel, "false discoveries": false, "FDP": false / max(n_sel, 1),
            "hits found": found, "power": found / max(int(is_hit.sum()), 1)}


def in_words(r):
    return (f"{r['selected']} selected, {r['false discoveries']} of them false (FDP {r['FDP']:.1%}); "
            f"{r['hits found']} hits found (power {r['power']:.1%})")


rows = []
for name, mod in rdb7.models.items():
    mu = mod.mu
    q_hat = conformal_quantile(np.abs(y - mu)[cal], 0.10)                               # half-width of the 90% interval
    p_values = P.conformal_pvalues((mu - c)[cal], y[cal] >= c, (mu - c)[test])          # explained in Section 3
    rules = {"A: predicted barrier below c": mu[test] < c,
             "B: 90% interval entirely below c": mu[test] + q_hat < c,
             "C: p-value below 0.10 for each reaction": p_values <= 0.10}
    for rule, selected in rules.items():
        rows.append({"model": mod.label, "rule": rule, **report(selected, is_hit)})
tempting = pd.DataFrame(rows).set_index(["model", "rule"])
display(tempting.round(3))

* **Rule A** gives no control. With the accurate Δ-learning model only a few percent of the selected reactions are false;
  with the structure-only model more than one in five is. The user of rule A cannot know which situation applies
  without the reference values that the screening was meant to avoid.
* **Rule B** makes hardly any false selection here, but that is a property of these data, not of the rule. The interval
  guarantee is a statement about single reactions, averaged over all candidates: a candidate is a non-hit *and* has its
  whole interval below $c$ with a probability of at most 10% (its barrier then lies above the interval). The guarantee
  therefore limits the number of false selections only to about one tenth of all candidates, a bound that does not
  shrink with the length of the list: a short list may consist entirely of non-hits. When hits are rare and errors
  heavy-tailed, most of a rule-B list can be false. The rule also answers the wrong question: it loses about 30% of the
  hits with the Δ-learning model and practically all of them with the structure-only model, whose intervals are so wide
  that hardly any reaction passes.
* **Rule C** looks statistically respectable and performs worst. A test at the 10% level lets about one non-hit in ten
  pass, and more than 90% of the candidates are non-hits: several hundred false discoveries swamp the true ones. This
  is the multiple-testing problem, well known from genomics and from virtual screening.

What we need is a rule that controls the error *of the list*: the **false discovery rate**

$$ \mathrm{FDR} = \mathbb{E}\!\left[\frac{\text{number of selected non-hits}}{\max(\text{number selected},\,1)}\right] \le q . $$

## 3. Conformal p-values

First, a measure of evidence for each candidate. Take any **score** computed from the candidate alone, such that a
smaller score looks more promising; the obvious choice is $\hat\mu(x) - c$, the predicted barrier measured from the
threshold (negative when the prediction lies below it). On its own the score has no scale: is "predicted 4 kcal/mol
below the threshold" convincing? The calibration set provides the scale. Among the calibration reactions, look only at
the **non-hits** (reference barrier at or above $c$), and count how many of them look at least as promising as
candidate $j$:

$$ p_j = \frac{1 + \#\{\,i \in \text{calibration}:\ y_i \ge c \ \text{ and } \ s_i \le s_j \,\}}{n + 1} . $$

Here $n$ is the size of the whole calibration set, hits included. A small $p_j$ means: hardly any reaction that we
*know* to be a non-hit ever looked this good.

Why is this a p-value? By the rank argument of Notebook 2. Put candidate $j$ together with the $n$ calibration
reactions: $n+1$ exchangeable reactions, any of which could equally well have been the candidate. Line up the non-hits
of this group by score. Candidate $j$ is a non-hit *and* receives $p_j \le t$ only if it is among the
$\lfloor t\,(n+1) \rfloor$ most promising-looking non-hits of the group, and at most $\lfloor t\,(n+1) \rfloor$ of the
$n+1$ reactions are in that position. Hence

$$ \Pr\{\, p_j \le t \ \text{ and candidate } j \text{ is a non-hit} \,\} \;\le\; t \qquad \text{for every } t : $$

a false alarm at level $t$ happens with probability at most $t$. (The construction is the "conformal selection" of Jin
and Candès, 2023, with their "clipped" score, written here for a threshold that is the same for all candidates.
Conformal p-values are also used to test for outliers; see Bates et al., 2023.)

In [ ]:
def conformal_pvalues(score_cal, null_cal, score_test):
    """p_j = (1 + number of calibration non-hits that look at least as promising as j) / (n + 1)."""
    null_scores = np.sort(score_cal[null_cal])
    return (1 + np.searchsorted(null_scores, score_test, side="right")) / (len(score_cal) + 1)


model = rdb7.models["delta"]
mu = model.mu
score = mu - c                                   # smaller = more promising
non_hit_cal = y[cal] >= c
p = conformal_pvalues(score[cal], non_hit_cal, score[test])
assert np.array_equal(p, P.conformal_pvalues(score[cal], non_hit_cal, score[test]))

print(f"calibration set: {non_hit_cal.sum()} non-hits, {(~non_hit_cal).sum()} hits; smallest possible p-value 1/(n+1) = {1 / (len(cal) + 1):.4f}")
print(f"median p-value of the candidates that are hits: {np.median(p[is_hit]):.4f};  of the non-hits: {np.median(p[~is_hit]):.3f}")

## 4. The Benjamini–Hochberg step

Selecting every candidate with $p_j \le 0.10$ was rule C, and it failed. The Benjamini–Hochberg (BH) procedure
(Benjamini and Hochberg, 1995) replaces the fixed cutoff by one that adapts to the number of candidates $m$:

1. sort the p-values, $p_{(1)} \le p_{(2)} \le \dots \le p_{(m)}$;
2. find the largest rank $r$ with $p_{(r)} \le q\,r/m$;
3. select the $r$ candidates with the smallest p-values (none if no such $r$ exists).

BH was designed for independent p-values, and ours are not independent: they all use the same calibration set. Jin and
Candès (2023) proved that the procedure nevertheless gives $\mathrm{FDR} \le q$ with these conformal p-values when the
calibration and candidate reactions are independent draws from the same population. Nothing is assumed about the
model or about the distribution of the barriers. (Their theorem breaks ties between equal scores at random. We count a
tie against the candidate instead, which is the least favorable outcome of that random draw for the candidate. Ties
are rare here, and the tests of the toolbox check that the false discovery rate stays below $q$ for heavily tied
scores as well.)

In [ ]:
def benjamini_hochberg(pvalues, q):
    """Boolean mask of the candidates selected by the Benjamini–Hochberg procedure at level q."""
    m = len(pvalues)
    order = np.argsort(pvalues, kind="stable")
    line = q * np.arange(1, m + 1) / m * (1 + 1e-12)     # the tiny factor makes "p equal to the line" count as below it
    passed = np.flatnonzero(pvalues[order] <= line)
    selected = np.zeros(m, dtype=bool)
    if passed.size:
        selected[order[: passed[-1] + 1]] = True
    return selected


q = 0.10
selected = benjamini_hochberg(p, q)
assert np.array_equal(selected, P.benjamini_hochberg(p, q))
print(f"Δ-learning model, target FDR {q:.0%}:     {in_words(report(selected, is_hit))}")

model_s = rdb7.models["structure"]
p_s = conformal_pvalues((model_s.mu - c)[cal], non_hit_cal, (model_s.mu - c)[test])
selected_s = benjamini_hochberg(p_s, q)
print(f"Structure-only model, target FDR {q:.0%}: {in_words(report(selected_s, is_hit))}")

One split proves little, because the guarantee is about an expectation. For the structure-only model in particular
the outcome of a single split varies a great deal at this level: the list is short, its false discovery proportion can
lie above or below $q$, and for some calibration sets the list is empty (Section 5 explains why). The experiment below
repeats the selection for 300 random calibration/candidate splits, for both models, at several target levels $q$, and
adds the three tempting rules for comparison.

In [ ]:
LEVELS = (0.02, 0.05, 0.10, 0.20, 0.30)


def selection_experiment(name, c=c, n_cal=1000, n_rep=300, levels=LEVELS, seed=41, normalized=False):
    """FDP and power of conformal selection (and of rules A-C) over random calibration/candidate splits."""
    mu = rdb7.models[name].mu
    abs_err = np.abs(y - mu)
    score = (mu - c) / sigma_of[name] if normalized else mu - c
    rows = []
    for cal_b, test_b in P.resplits(rdb7.hold, n_cal, n_rep, seed=seed):
        hit_b = y[test_b] < c
        p_b = conformal_pvalues(score[cal_b], y[cal_b] >= c, score[test_b])
        rows.append({"rule": "A: predicted barrier below c", "q": np.nan, **report(mu[test_b] < c, hit_b)})
        rows.append({"rule": "B: 90% interval below c", "q": np.nan, **report(mu[test_b] + conformal_quantile(abs_err[cal_b], 0.10) < c, hit_b)})
        for level in levels:
            rows.append({"rule": "C: p-value below q, no correction", "q": level, **report(p_b <= level, hit_b)})
            rows.append({"rule": "conformal selection", "q": level, **report(benjamini_hochberg(p_b, level), hit_b)})
    return pd.DataFrame(rows)


def summarize(runs):
    """One row per rule and level: averages over the splits."""
    runs = runs.assign(fdp_if_not_empty=runs["FDP"].where(runs["selected"] > 0))      # undefined for an empty list
    g = runs.groupby(["rule", "q"], sort=False, dropna=False)
    return pd.DataFrame({"selected (mean)": g["selected"].mean(), "FDR (mean FDP)": g["FDP"].mean(),
                         "FDP, 90th percentile": g["FDP"].quantile(0.90), "power (mean)": g["power"].mean(),
                         "splits with an empty list": g["selected"].apply(lambda s: np.mean(s == 0)),
                         "mean FDP of the non-empty lists": g["fdp_if_not_empty"].mean()})


def show_summary(table):
    """Display a summary table; a dash stands for 'does not apply'."""
    shown = table.round(3).astype(object).where(table.notna(), "–")
    shown.index = pd.MultiIndex.from_tuples([(rule, "–" if np.isnan(level) else f"{level:.2f}") for rule, level in table.index],
                                            names=["rule", "q"])
    display(shown)


runs = {name: selection_experiment(name) for name in rdb7.models}
for name, mod in rdb7.models.items():
    print(mod.label)
    show_summary(summarize(runs[name]))

# rule A against conformal selection at the false discovery rate that rule A turned out to have
rows = []
for name, mod in rdb7.models.items():
    rule_a = summarize(runs[name]).loc["A: predicted barrier below c"].iloc[0]
    same_level = selection_experiment(name, levels=(float(rule_a["FDR (mean FDP)"]),))
    same_level = same_level[same_level["rule"] == "conformal selection"]
    rows.append({"model": mod.label, "FDR of rule A": rule_a["FDR (mean FDP)"], "power of rule A": rule_a["power (mean)"],
                 "FDR of conformal selection at q = FDR of rule A": same_level["FDP"].mean(), "its power": same_level["power"].mean()})
versus_rule_a = pd.DataFrame(rows).set_index("model")
display(versus_rule_a.round(3))

Read the rows "conformal selection" first. For both models and every level, the mean false discovery proportion is at
or below the target $q$: **the promise is kept by a model with a mean absolute error (MAE) of 1.6 kcal/mol and equally
by one with an MAE of 8.6 kcal/mol.** What the model quality decides is the *power*. The Δ-learning model finds more
than 90% of the hits already at $q = 5\%$; the structure-only model finds less than a quarter of the hits at $q = 10\%$
and needs $q = 30\%$ to find more than half of them. As with the intervals of Notebook 2: validity does not depend on
the model, power does.

Compare with the tempting rules. Rule A happens to be fine for the Δ-learning model and delivers more than 20% false
discoveries for the structure-only model. The last table puts the two approaches side by side at the same error rate:
there rule A finds slightly more hits (the stars in panel c below lie a little above the curves). That is the price of
the guarantee: rule A gives a slightly better list whose error rate is unknown until the reference values exist;
conformal selection gives up a few percent of the hits and states the error rate in advance. Rule C exceeds every
target by a wide margin.

Two further observations. The FDR is an *average*: the column "FDP, 90th percentile" shows that an individual list can
contain a larger fraction of false entries than $q$. And at strict levels the list is sometimes empty. An empty list
enters the average as a list without errors, so part of the promise can be kept by selecting nothing: when empty lists
are frequent, the lists that are *not* empty may contain more false entries than $q$ on average, as they do here for
the structure-only model at the strict levels (last column). Section 5 explains when the list is empty, and Section 7
comes back to what the promise means.

In [ ]:
fig, axes = P.panels(3)

# a: the Benjamini–Hochberg picture for one split (Δ-learning model)
ax = axes[0]
m_cand = len(p)
n_selected = int(selected.sum())
r_max = min(m_cand, max(800, 200 * int(np.ceil(1.5 * n_selected / 200))))   # 800 candidates unless the list is longer
ranks = np.arange(1, r_max + 1)
p_sorted = np.sort(p)[:r_max]
p_top = max(0.04, 1.6 * q * r_max / m_cand)
ax.axvspan(0, n_selected, color=P.SKY, alpha=0.30, lw=0)
ax.plot(ranks, q * ranks / m_cand, color=P.VERMILLION, lw=1.2, ls="--")
ax.plot(ranks, p_sorted, color=P.BLUE)
ax.text(0.55 * n_selected, 0.775 * p_top, f"selected:\n{n_selected} reactions", ha="center", va="center", fontsize=6.5)
r_label = int(0.81 * r_max)
ax.text(r_label, q * r_label / m_cand + 0.1 * p_top, "BH line $q\\,r/m$", ha="center", va="center", fontsize=6.5)
r_label = max(1, min(int(1.3 * n_selected), r_max - 1))
right = r_label > 0.8 * r_max                                         # near the right edge: label to the left of the curve
ax.annotate("sorted p-values", (r_label, min(p_sorted[r_label - 1], 0.95 * p_top)), xytext=(-4 if right else 4, 0),
            textcoords="offset points", ha="right" if right else "left", va="center", fontsize=6.5)
ax.set(xlabel="candidates ordered by p-value, $r$", ylabel="p-value", xlim=(0, r_max), ylim=(0, p_top))
if r_max < m_cand:                                                    # a multiple of 200: five round tick marks
    ax.set_xticks(np.linspace(0, r_max, 5))
P.panel_label(ax, "a", "Benjamini–Hochberg step")

# b: realized FDR against the target
ax = axes[1]
ax.plot([0, 0.32], [0, 0.32], color=P.GRAY, lw=0.8, ls="--", label="FDR = $q$")
for name in ("structure", "delta"):
    stats_q = summarize(runs[name]).loc["conformal selection"]
    ax.plot(stats_q.index, stats_q["FDR (mean FDP)"], marker=P.MODEL_MARKERS[name], color=P.MODEL_COLORS[name],
            label=rdb7.models[name].label.replace(" model", ""))
ax.set(xlabel="target level $q$", ylabel="false discovery rate", xlim=(0, 0.32), ylim=(0, 0.32))
ax.set_xticks([0, 0.1, 0.2, 0.3])
ax.set_yticks([0, 0.1, 0.2, 0.3])
ax.legend(loc="upper left", borderaxespad=0.3)
P.panel_label(ax, "b", "FDR at or below $q$")

# c: power against the target
ax = axes[2]
for name in ("structure", "delta"):
    stats_q = summarize(runs[name]).loc["conformal selection"]
    ax.plot(stats_q.index, stats_q["power (mean)"], marker=P.MODEL_MARKERS[name], color=P.MODEL_COLORS[name],
            label=rdb7.models[name].label.replace(" model", ""))
    rule_a = versus_rule_a.loc[rdb7.models[name].label]
    ax.plot([rule_a["FDR of rule A"]], [rule_a["power of rule A"]], ls="none", marker="*", markersize=9, color=P.MODEL_COLORS[name],
            markeredgecolor="white", markeredgewidth=0.6, label="rule A" if name == "delta" else None)
ax.set(xlabel="target level $q$", ylabel="fraction of hits found (power)", xlim=(0, 0.32), ylim=(0, 1.08))
ax.set_xticks([0, 0.1, 0.2, 0.3])
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.legend(loc="center left", bbox_to_anchor=(0.02, 0.57), borderaxespad=0)
P.panel_label(ax, "c", "Power is earned")
for ax in axes:
    ax.set_box_aspect(1)
P.savefig(fig, "nb6_selection")
plt.show()

Panel **a**: the BH step for one split and the Δ-learning model; only the first 800 of the 4935 candidates are shown.
BH selects all candidates up to the last rank $r$ at which the sorted p-values (solid) are at or below the line $q\,r/m$
(dashed). For the first 49 candidates the curve lies above the line, which is why a list cannot be shorter than 50
here (Section 5). Panel **b**: the realized false discovery rate, averaged over 300 splits, against the target. Panel
**c**: the power. The stars show rule A (select if the predicted barrier is below $c$) at the false discovery rate it
turned out to have.

## 5. How large must the calibration set be?

A conformal p-value cannot be smaller than $1/(n+1)$. BH selects $r$ candidates only if the $r$-th smallest p-value is
at most $q\,r/m$. Together:

$$ \frac{1}{n+1} \le \frac{q\,r}{m} \quad\Longrightarrow\quad r \;\ge\; \frac{m}{q\,(n+1)} . $$

A BH shortlist is therefore either empty or contains at least $m/(q(n+1))$ candidates. Put differently, a shortlist
can only be as short as a fraction $f$ of the candidates if **the calibration set has at least about $1/(q f)$
reactions.** If 5% of the candidates are hits, a list of that length at $q = 10\%$ needs $n \ge 200$; in a
needle-in-a-haystack screen with 0.1% hits it needs $n \ge 10{,}000$. The experiment confirms the threshold:

In [ ]:
rows = []
for name in rdb7.models:
    for n_cal in (100, 200, 500, 1000, 2000):
        r = selection_experiment(name, n_cal=n_cal, n_rep=200, levels=(0.10,), seed=42)
        r = r[r["rule"] == "conformal selection"]
        m_cand = len(rdb7.hold) - n_cal
        nonempty = r.loc[r["selected"] > 0, "selected"]
        rows.append({"model": rdb7.models[name].label, "calibration reactions n": n_cal,
                     "smallest possible list m/(q(n+1))": int(np.ceil(m_cand / (0.10 * (n_cal + 1)))),
                     "smallest non-empty list observed": int(nonempty.min()) if len(nonempty) else np.nan,
                     "splits with an empty list": np.mean(r["selected"] == 0), "FDR": r["FDP"].mean(), "power": r["power"].mean()})
resolution = pd.DataFrame(rows).set_index(["model", "calibration reactions n"])
display(resolution.round(3))

With 100 calibration reactions a list needs at least 578 entries, more than there are hits. However good the model,
the list is then empty in the great majority of the splits (and when it is not, it has to contain many non-hits). From
200 on the Δ-learning model finds most hits. For the structure-only model a larger calibration set keeps helping,
because many of its p-values lie close to the BH line.

## 6. A better score gives more power

The score $\hat\mu(x) - c$ treats all predictions as equally reliable. Dividing by the difficulty estimate of Notebook 3
gives $(\hat\mu(x) - c)/\hat\sigma(x)$: a candidate predicted 5 kcal/mol below the threshold counts for more when the
model is usually accurate for reactions of its kind. Any function of $x$ is a legitimate score, so the guarantee is
unchanged; only the power can differ.

In [ ]:
rows = []
for name in rdb7.models:
    for label, flag in (("mu - c", False), ("(mu - c) / sigma", True)):
        r = selection_experiment(name, normalized=flag, levels=(0.05, 0.10, 0.20))
        r = r[r["rule"] == "conformal selection"].groupby("q").mean(numeric_only=True)
        for level, row in r.iterrows():
            rows.append({"model": rdb7.models[name].label, "score": label, "q": level, "FDR": row["FDP"], "power": row["power"]})
scores = pd.DataFrame(rows).pivot_table(index=["model", "q"], columns="score", values=["FDR", "power"])
display(scores.round(3))

For the structure-only model the normalized score finds clearly more hits at the strict levels; for the Δ-learning
model, which already finds nearly everything, there is little left to gain.

## 7. Reading the result, and its limits

* **What the list means.** "FDR at most 10%" says: averaged over random calibration and candidate sets, at most one in
  ten of the selected reactions is a non-hit, *where an empty list counts as a list without errors*. It is a statement
  about the list as a whole; it is not a 90% probability for each entry. When empty lists are frequent, the lists that
  are not empty can contain more false entries than $q$ on average (for them alone, the bound is $q$ divided by the
  probability that the list is not empty): for the structure-only model at $q = 10\%$ the list is empty in about a
  third of the splits and the other lists contain 12% non-hits on average (at $q = 5\%$: empty in 85% of the splits,
  13% non-hits in the others). For the Δ-learning model, whose list is practically never empty here, there is no such
  difference.
* **The threshold must be fixed in advance.** Tuning $c$ or $q$ after looking at the list invalidates the guarantee.
  (The threshold may differ from candidate to candidate if it is a known function of $x$ that is defined in the same
  way for the calibration reactions.)
* **Exchangeability, again.** The calibration set *as a whole* must be exchangeable with the candidates: hits and
  non-hits in the proportions of the candidate population, and its non-hits a fair sample of the candidates' non-hits.
  A calibration set enriched in hits (reference calculations that were run because the reactions looked promising)
  breaks the guarantee. Everything in Notebook 5 applies; weighted conformal p-values exist for covariate shift (Jin
  and Candès, 2026).
* **Do not attach ordinary intervals to the selected reactions.** Selection changes the population: the reactions on
  the list are those whose predictions were low, and predictions that are too low are over-represented among them.
  Intervals that remain valid after selection need a separate construction.

## Take-aways

* Screening needs error control for a *list*, not for single predictions. The false discovery rate is the natural
  quantity.
* Conformal p-values compare each candidate with the calibration reactions that are known non-hits; the
  Benjamini–Hochberg step turns them into a shortlist with FDR at most $q$, for any model.
* A poor model does not break the guarantee. It shortens the list.
* A shortlist that is a fraction $f$ of the candidates requires a calibration set of about $1/(q f)$ reactions.

## Try it yourself

1. Change the threshold to `c = 60.0` (a half-life of about 0.4 s at 1000 K). How do the fraction of hits among the
   candidates and the power of the structure-only model change?
2. Select reactions with a *high* barrier instead (say, above 110 kcal/mol). Three definitions have to change in
   Sections 1 to 4: the score, which calibration reactions count as non-hits, and which candidates count as hits.
3. Run rule C at the level `q / m` (the Bonferroni correction). It controls an even stricter error rate. What happens
   to the power?

**This is the last notebook.** Together the six notebooks cover the path from a point prediction to an interval with a
guarantee (2), to intervals that adapt (3) and are valid class by class (4), to the limits of the guarantee (5), and
finally to decisions (6).

## References

* Jin, Y.; Candès, E. J. Selection by Prediction with Conformal p-Values. *J. Mach. Learn. Res.* **2023**, *24* (244),
  1–41. https://jmlr.org/papers/v24/22-1176.html
* Benjamini, Y.; Hochberg, Y. Controlling the False Discovery Rate: A Practical and Powerful Approach to Multiple
  Testing. *J. R. Stat. Soc. Ser. B: Methodol.* **1995**, *57*, 289–300. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x
* Bates, S.; Candès, E.; Lei, L.; Romano, Y.; Sesia, M. Testing for Outliers with Conformal p-Values. *Ann. Stat.*
  **2023**, *51*, 149–178. https://doi.org/10.1214/22-AOS2244
* Jin, Y.; Candès, E. J. Model-Free Selective Inference Under Covariate Shift via Weighted Conformal p-Values.
  *Biometrika* **2026**, *113*, asaf066. https://doi.org/10.1093/biomet/asaf066

In [ ]:
def table_dict(frame):
    out = {}
    for (rule, level), row in frame.iterrows():
        key = rule if not np.isfinite(level) else f"{rule} | q={level:.2f}"
        out[key] = {name: value for name, value in row.items() if np.isfinite(value)}     # undefined entries are left out
    return out


results = {
    "threshold": c, "hit_rate_hold": float(np.mean(y[rdb7.hold] < c)), "eyring_half_life_ms": float(1000 * np.log(2) / rate),
    "n_candidates": len(test),
    "single_split": {f"{model_label} | {rule}": dict(row) for (model_label, rule), row in tempting.to_dict("index").items()},
    "single_split_bh": {"delta": report(selected, is_hit), "structure": report(selected_s, is_hit)},
    "experiment": {name: table_dict(summarize(runs[name])) for name in rdb7.models},
    "versus_rule_a": {label: dict(row) for label, row in versus_rule_a.to_dict("index").items()},
    "resolution": {f"{model_label} | n={n}": dict(row) for (model_label, n), row in resolution.to_dict("index").items()},
    "scores": {f"{model_label} | q={level:.2f} | {stat} | {score}": value
               for (model_label, level), row in scores.iterrows() for (stat, score), value in row.items()},
}
P.save_results("nb6", results, rdb7)
P.check("nb6", results);

# Results archive and comparison with the reference run

In [ ]:
# Pack the results and compare them with the reference run
import glob, json, zipfile

print("Comparison with the reference run")
summary = P.check_all()
all_agree = summary["different"] == 0 and summary["missing"] == 0

archive = "ConformalPrimer_outputs.zip"
run_info = {"environment": P.environment(), "minutes": round((time.time() - T_START) / 60, 2),
            "comparison_with_reference": summary, "reference_environment": P.REFERENCE_RESULTS.get("_environment", {}),
            "all_numbers_agree_with_reference": bool(all_agree)}
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(glob.glob("outputs/*")):
        z.write(path)
    z.writestr("run_info.json", json.dumps(run_info, indent=1))
print(f"\nwrote {archive} ({os.path.getsize(archive) / 1e6:.1f} MB) after {(time.time() - T_START) / 60:.1f} minutes")
if "google.colab" in sys.modules and "KAGGLE_KERNEL_RUN_TYPE" not in os.environ:      # on Colab: offer the file for download
    try:
        from google.colab import files
        files.download(archive)
    except Exception:
        print("Download the file from the file browser on the left.")